In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_val.bin
/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/latest.pt
/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt


In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

# Retrieve token securely from Kaggle Secrets
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

# Authenticate with Hugging Face
login(token=HF_TOKEN, add_to_git_credential=False)

# Verify authentication
api = HfApi()
user = api.whoami(token=HF_TOKEN)

print("Hugging Face authentication successful!")
print("Authenticated account:", user.get("name", "Verified"))

Hugging Face authentication successful!
Authenticated account: Anuj-chaudhary-210167


In [3]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# Read token securely from Kaggle Secrets
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

REPO_ID = "Exploration-Lab/CS-781-Capstone"
LOCAL_DIR = "/kaggle/working/Phase_3_RR"

# Download only Phase 3 files
path = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    token=hf_token,
    allow_patterns=[
        "Phase_3_RR/**"
    ],
    local_dir=LOCAL_DIR,
)

print("Download completed!")
print("Dataset location:", path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Download completed!
Dataset location: /kaggle/working/Phase_3_RR


In [4]:
import os

for root, dirs, files in os.walk("/kaggle/working/Phase_3_RR"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/working/Phase_3_RR/Phase_3_RR/Test/IT_test.json
/kaggle/working/Phase_3_RR/Phase_3_RR/Test/CL_test.json
/kaggle/working/Phase_3_RR/Phase_3_RR/Train/CL_train.json
/kaggle/working/Phase_3_RR/Phase_3_RR/Train/CL_dev.json
/kaggle/working/Phase_3_RR/Phase_3_RR/Train/label_vocab.json
/kaggle/working/Phase_3_RR/Phase_3_RR/Train/IT_train.json
/kaggle/working/Phase_3_RR/Phase_3_RR/Train/IT_dev.json
/kaggle/working/Phase_3_RR/.cache/huggingface/.gitignore
/kaggle/working/Phase_3_RR/.cache/huggingface/CACHEDIR.TAG
/kaggle/working/Phase_3_RR/.cache/huggingface/download/Phase_3_RR/Test/CL_test.json.lock
/kaggle/working/Phase_3_RR/.cache/huggingface/download/Phase_3_RR/Test/IT_test.json.metadata
/kaggle/working/Phase_3_RR/.cache/huggingface/download/Phase_3_RR/Test/IT_test.json.lock
/kaggle/working/Phase_3_RR/.cache/huggingface/download/Phase_3_RR/Test/CL_test.json.metadata
/kaggle/working/Phase_3_RR/.cache/huggingface/download/Phase_3_RR/Train/CL_dev.json.lock
/kaggle/working/Phase_3_RR/.ca

In [5]:
import json
from pathlib import Path
from collections import Counter

# Dataset paths
DATA_DIR = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")
TRAIN_DIR = DATA_DIR / "Train"
TEST_DIR = DATA_DIR / "Test"

FILES = {
    "CL_train": TRAIN_DIR / "CL_train.json",
    "CL_dev": TRAIN_DIR / "CL_dev.json",
    "IT_train": TRAIN_DIR / "IT_train.json",
    "IT_dev": TRAIN_DIR / "IT_dev.json",
    "CL_test": TEST_DIR / "CL_test.json",
    "IT_test": TEST_DIR / "IT_test.json",
    "label_vocab": TRAIN_DIR / "label_vocab.json",
}

# Check all files
print("FILE VERIFICATION")
for name, path in FILES.items():
    print(f"{name:12s} | exists={path.exists()} | {path}")

missing = [name for name, path in FILES.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

# Load JSON files
data = {}
for name, path in FILES.items():
    with open(path, "r", encoding="utf-8") as f:
        data[name] = json.load(f)

print("\nDATA LOADED SUCCESSFULLY")

# Inspect structure without printing entire documents
for name in [
    "CL_train", "CL_dev", "IT_train",
    "IT_dev", "CL_test", "IT_test"
]:
    obj = data[name]

    print(f"\n{'=' * 60}")
    print(f"{name}: type={type(obj).__name__}")

    if isinstance(obj, dict):
        print("Number of top-level entries:", len(obj))
        first_key = next(iter(obj), None)
        if first_key is not None:
            print("First key:", first_key)
            print("First value type:", type(obj[first_key]).__name__)
            print("First value preview:", repr(obj[first_key])[:700])

    elif isinstance(obj, list):
        print("Number of documents:", len(obj))
        if obj:
            print("First document keys:",
                  list(obj[0].keys()) if isinstance(obj[0], dict) else None)
            print("First document preview:", repr(obj[0])[:1000])

print("\nLABEL VOCABULARY")
print(json.dumps(data["label_vocab"], ensure_ascii=False, indent=2)[:4000])

FILE VERIFICATION
CL_train     | exists=True | /kaggle/working/Phase_3_RR/Phase_3_RR/Train/CL_train.json
CL_dev       | exists=True | /kaggle/working/Phase_3_RR/Phase_3_RR/Train/CL_dev.json
IT_train     | exists=True | /kaggle/working/Phase_3_RR/Phase_3_RR/Train/IT_train.json
IT_dev       | exists=True | /kaggle/working/Phase_3_RR/Phase_3_RR/Train/IT_dev.json
CL_test      | exists=True | /kaggle/working/Phase_3_RR/Phase_3_RR/Test/CL_test.json
IT_test      | exists=True | /kaggle/working/Phase_3_RR/Phase_3_RR/Test/IT_test.json
label_vocab  | exists=True | /kaggle/working/Phase_3_RR/Phase_3_RR/Train/label_vocab.json

DATA LOADED SUCCESSFULLY

CL_train: type=list
Number of documents: 40
First document keys: ['id', 'text', 'labels']
First document preview: {'id': 'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733', 'text': ['Consequent upon the repeal of the MRTP Act this complaint has been received by transfer on 04.03.2010 from the office of DGIR, MRTP Commission un

In [6]:
from collections import Counter

LABEL_VOCAB = data["label_vocab"]

# Reverse mapping: label ID -> label name
ID_TO_LABEL = {v: k for k, v in LABEL_VOCAB.items()}
VALID_LABELS = set(LABEL_VOCAB.keys())

print("Number of labels:", len(LABEL_VOCAB))
print("Labels:", list(LABEL_VOCAB.keys()))

# Check every train/dev document
label_counts = Counter()
total_sentences = 0
total_documents = 0
errors = []

for split in ["CL_train", "CL_dev", "IT_train", "IT_dev"]:
    documents = data[split]
    split_sentences = 0

    for doc in documents:
        doc_id = doc.get("id", "<missing_id>")
        texts = doc.get("text", [])
        labels = doc.get("labels", [])

        if len(texts) != len(labels):
            errors.append(
                f"{split} | {doc_id}: "
                f"{len(texts)} sentences vs {len(labels)} labels"
            )
            continue

        for label in labels:
            if label not in VALID_LABELS:
                errors.append(
                    f"{split} | {doc_id}: unknown label {label!r}"
                )
            else:
                label_counts[label] += 1

        split_sentences += len(texts)

    total_sentences += split_sentences
    total_documents += len(documents)

    print(
        f"{split:10s}: "
        f"{len(documents):2d} documents | "
        f"{split_sentences:5d} sentences"
    )

print("\nTOTAL DOCUMENTS:", total_documents)
print("TOTAL SENTENCES:", total_sentences)

print("\nLABEL DISTRIBUTION")
for label, count in label_counts.most_common():
    print(f"{label:28s} {count:6d} ({count / max(total_sentences, 1):.2%})")

print("\nVALIDATION RESULT")
if errors:
    print(f"FAILED: {len(errors)} issue(s)")
    for error in errors[:20]:
        print(" -", error)
else:
    print("PASS: All train/dev documents have aligned sentences and labels.")

Number of labels: 13
Labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
CL_train  : 40 documents | 10593 sentences
CL_dev    :  5 documents |  1396 sentences
IT_train  : 40 documents |  6252 sentences
IT_dev    :  5 documents |   746 sentences

TOTAL DOCUMENTS: 90
TOTAL SENTENCES: 18987

LABEL DISTRIBUTION
RatioOfTheDecision             4704 (24.77%)
Fact                           4542 (23.92%)
PrecedentReliedUpon            2199 (11.58%)
ArgumentPetitioner             2013 (10.60%)
ArgumentRespondent             1620 (8.53%)
Statute                        1219 (6.42%)
RulingByLowerCourt              976 (5.14%)
PrecedentNotReliedUpon          619 (3.26%)
RulingByPresentCourt            356 (1.87%)
None                            330 (1.74%)
Dissent                         284 (1.50%)
Issue           

In [7]:
for split in ["CL_test", "IT_test"]:
    test_docs = data[split]

    print(f"\n{split}")
    print("-" * 50)

    total = 0

    for doc_id, sentences in test_docs.items():
        if not isinstance(sentences, list):
            raise TypeError(
                f"{split}: expected list of sentences for {doc_id}"
            )

        print(f"{doc_id}: {len(sentences)} sentences")
        total += len(sentences)

    print("Documents:", len(test_docs))
    print("Total sentences:", total)

print("\nTest data verification complete.")


CL_test
--------------------------------------------------
HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489: 18 sentences
COM_TPM_Consultants_Pvt_Ltd_vs_Competition_Commission_TA2016110716153210127COM903224: 230 sentences
SC_Ultra Tech Cement Ltd vs Competition Commission of India and Ors 12062013 SC: 35 sentences
CCI_Ms_Mineral_Enterprises_Limited_vs_Ministry_of_RailCO120089COM803307: 114 sentences
HC_Vodafone_India_Limited_and_Ors_vs_The_Competition_MH2017051017164746274COM906883: 942 sentences
Documents: 5
Total sentences: 1339

IT_test
--------------------------------------------------
SC_2009_710: 420 sentences
SC_2017_195: 186 sentences
BHC_BomHC_2014_498: 53 sentences
BHC_BomHC_1981_263: 40 sentences
BHC_BomHC_1997_583: 159 sentences
Documents: 5
Total sentences: 858

Test data verification complete.


In [8]:
from pathlib import Path

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

matches = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        name = path.name.lower()
        full_path = str(path).lower()

        if (
            "arch4" in name
            or "arch4" in full_path
        ) and path.suffix in {".pt", ".pth", ".ckpt", ".bin"}:
            matches.append(path)

print("Arch 4 checkpoint candidates:", len(matches))

for path in matches:
    print(f"{path}  ({path.stat().st_size / (1024**2):.1f} MB)")

Arch 4 checkpoint candidates: 2
/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/latest.pt  (2305.5 MB)
/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt  (768.5 MB)


In [9]:
import torch
from pathlib import Path

CHECKPOINT_DIR = Path(
    "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/"
    "transformers/default/1/arch4_hybrid_gqa_201m"
)

CHECKPOINTS = {
    "best": CHECKPOINT_DIR / "best.pt",
    "latest": CHECKPOINT_DIR / "latest.pt",
}

for name, path in CHECKPOINTS.items():
    print("\n" + "=" * 70)
    print(f"CHECKPOINT: {name}")
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    print(f"File size: {path.stat().st_size / (1024**2):.1f} MB")

    try:
        # Checkpoint metadata only; do not load model weights onto the GPU.
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        print("Loaded object type:", type(checkpoint).__name__)

        if isinstance(checkpoint, dict):
            print("Top-level keys:", list(checkpoint.keys())[:50])

            for key, value in checkpoint.items():
                if isinstance(value, dict):
                    print(f"\n{key}: dictionary with {len(value)} entries")
                    print("Sample keys:", list(value.keys())[:12])

                elif isinstance(value, (str, int, float, bool, type(None))):
                    print(f"{key}: {value}")

                elif torch.is_tensor(value):
                    print(f"{key}: Tensor {tuple(value.shape)}")

        del checkpoint

    except Exception as e:
        print("Safe inspection failed:", type(e).__name__, str(e))


CHECKPOINT: best
Path: /kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt
Exists: True
File size: 768.5 MB
Loaded object type: dict
Top-level keys: ['step', 'model', 'best_val', 'cfg']
step: 10000

model: dictionary with 100 entries
Sample keys: ['token_embedding.weight', 'layers.0.norm1.weight', 'layers.0.norm1.bias', 'layers.0.attn.qkv_proj.weight', 'layers.0.attn.out_proj.weight', 'layers.0.norm2.weight', 'layers.0.norm2.bias', 'layers.0.ffn.in_proj.weight', 'layers.0.ffn.out_proj.weight', 'layers.1.norm1.weight', 'layers.1.norm1.bias', 'layers.1.attn.qkv_proj.weight']
best_val: 2.475979009747505

cfg: dictionary with 34 entries
Sample keys: ['DATA_DIR', 'VOCAB_SIZE', 'D_MODEL', 'N_LAYERS', 'N_Q_HEADS', 'N_KV_HEADS', 'D_FF', 'MAX_SEQ_LEN', 'SEQ_LEN', 'WINDOW', 'GLOBAL_EVERY', 'ROPE_BASE']

CHECKPOINT: latest
Path: /kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/latest.pt
Exists:

In [10]:
import torch
from pathlib import Path

CHECKPOINT_PATH = Path(
    "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/"
    "transformers/default/1/arch4_hybrid_gqa_201m/best.pt"
)

ckpt = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

cfg = ckpt["cfg"]
state = ckpt["model"]

print("=" * 65)
print("ARCH 4 CONFIGURATION")
print("=" * 65)

for key, value in cfg.items():
    print(f"{key:22s}: {value}")

print("\n" + "=" * 65)
print("IMPORTANT TENSOR SHAPES")
print("=" * 65)

for key in [
    "token_embedding.weight",
    "position_embedding.weight",
    "final_norm.weight",
    "layers.0.norm1.weight",
    "layers.0.attn.qkv_proj.weight",
    "layers.0.attn.out_proj.weight",
    "layers.0.ffn.in_proj.weight",
    "layers.0.ffn.out_proj.weight",
]:
    if key in state:
        print(f"{key:38s}: {tuple(state[key].shape)}")

print("\nTotal state-dict tensors:", len(state))
print("Training step:", ckpt["step"])
print("Best validation loss:", ckpt["best_val"])

del ckpt, state

ARCH 4 CONFIGURATION
DATA_DIR              : /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1
VOCAB_SIZE            : 40000
D_MODEL               : 1024
N_LAYERS              : 12
N_Q_HEADS             : 16
N_KV_HEADS            : 8
D_FF                  : 3328
MAX_SEQ_LEN           : 1024
SEQ_LEN               : 1024
WINDOW                : 256
GLOBAL_EVERY          : 3
ROPE_BASE             : 10000.0
DROPOUT               : 0.0
INIT_STD              : 0.02
TOKENS_PER_STEP       : 16384
CANDIDATES            : [(8, False), (4, False), (8, True), (4, True), (2, True)]
MAX_STEPS             : 10000
LR                    : 0.0003
MIN_LR_RATIO          : 0.1
WARMUP_STEPS          : 500
WEIGHT_DECAY          : 0.1
GRAD_CLIP             : 1.0
LOSS_CHUNK            : 2048
USE_FUSED_ADAMW       : True
USE_COMPILE           : True
COMPILE_MODE          : default
EVAL_EVERY            : 500
EVAL_BATCHES_PER_SOURCE: 10
SAVE_EVERY            : 500
LOG_EVERY             : 50
TIME_LIMIT_H

In [11]:
from tokenizers import Tokenizer
import json

TOKENIZER_PATH = (
    "/kaggle/input/datasets/achiver07/"
    "phase1-tokens-v40k-clean1/tokenizer.json"
)

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

with open(TOKENIZER_PATH, "r", encoding="utf-8") as f:
    tokenizer_json = json.load(f)

tokenizer_vocab_size = tokenizer.get_vocab_size()

ckpt = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

embedding_vocab_size = ckpt["model"]["token_embedding.weight"].shape[0]

print("Tokenizer vocabulary:", tokenizer_vocab_size)
print("Checkpoint vocabulary:", embedding_vocab_size)
print("Vocabulary matches:", tokenizer_vocab_size == embedding_vocab_size)

print("\nSpecial tokens:")
for token in ["<pad>", "<bos>", "<eos>"]:
    print(f"{token}: {tokenizer.token_to_id(token)}")

assert tokenizer_vocab_size == embedding_vocab_size, (
    "Vocabulary mismatch! Do not load this tokenizer with this checkpoint."
)

del ckpt

Tokenizer vocabulary: 40000
Checkpoint vocabulary: 40000
Vocabulary matches: True

Special tokens:
<pad>: 0
<bos>: 1
<eos>: 2


In [12]:
import torch
from pathlib import Path

CHECKPOINT_PATH = Path(
    "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/"
    "transformers/default/1/arch4_hybrid_gqa_201m/best.pt"
)

ckpt = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

cfg = ckpt["cfg"]
state = ckpt["model"]

print("ARCH 4 STRUCTURE INSPECTION")
print("=" * 70)

# Print every parameter name and shape.
for name, tensor in state.items():
    print(f"{name:55s} {tuple(tensor.shape)}")

print("\n" + "=" * 70)
print("Attention projection shapes by layer")

for i in range(cfg["N_LAYERS"]):
    prefix = f"layers.{i}.attn."

    qkv = state.get(prefix + "qkv_proj.weight")
    out = state.get(prefix + "out_proj.weight")

    if qkv is not None:
        print(
            f"Layer {i:02d}: "
            f"qkv={tuple(qkv.shape)}, "
            f"out={tuple(out.shape)}"
        )

print("\n" + "=" * 70)
print("Checkpoint and configuration checks")

print("Layers:", cfg["N_LAYERS"])
print("Hidden size:", cfg["D_MODEL"])
print("Query heads:", cfg["N_Q_HEADS"])
print("KV heads:", cfg["N_KV_HEADS"])
print("FFN dimension:", cfg["D_FF"])
print("Window:", cfg["WINDOW"])
print("Global attention every:", cfg["GLOBAL_EVERY"])
print("Checkpoint step:", ckpt["step"])

del ckpt, state

ARCH 4 STRUCTURE INSPECTION
token_embedding.weight                                  (40000, 1024)
layers.0.norm1.weight                                   (1024,)
layers.0.norm1.bias                                     (1024,)
layers.0.attn.qkv_proj.weight                           (2048, 1024)
layers.0.attn.out_proj.weight                           (1024, 1024)
layers.0.norm2.weight                                   (1024,)
layers.0.norm2.bias                                     (1024,)
layers.0.ffn.in_proj.weight                             (6656, 1024)
layers.0.ffn.out_proj.weight                            (1024, 3328)
layers.1.norm1.weight                                   (1024,)
layers.1.norm1.bias                                     (1024,)
layers.1.attn.qkv_proj.weight                           (2048, 1024)
layers.1.attn.out_proj.weight                           (1024, 1024)
layers.1.norm2.weight                                   (1024,)
layers.1.norm2.bias                     

In [13]:
import torch
from pathlib import Path

CHECKPOINT_PATH = Path(
    "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/"
    "transformers/default/1/arch4_hybrid_gqa_201m/best.pt"
)

ckpt = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

cfg = ckpt["cfg"]
state = ckpt["model"]

d_model = cfg["D_MODEL"]
n_q_heads = cfg["N_Q_HEADS"]
n_kv_heads = cfg["N_KV_HEADS"]

head_dim = d_model // n_q_heads
q_dim = n_q_heads * head_dim
kv_dim = n_kv_heads * head_dim
expected_qkv_dim = q_dim + 2 * kv_dim

actual_qkv_dim = state[
    "layers.0.attn.qkv_proj.weight"
].shape[0]

print("=" * 60)
print("ARCH 4 ATTENTION DIMENSION CHECK")
print("=" * 60)

print("Hidden dimension       :", d_model)
print("Query heads             :", n_q_heads)
print("KV heads                :", n_kv_heads)
print("Head dimension          :", head_dim)
print("Expected Q dimension    :", q_dim)
print("Expected K dimension    :", kv_dim)
print("Expected V dimension    :", kv_dim)
print("Expected total QKV      :", expected_qkv_dim)
print("Actual QKV projection   :", actual_qkv_dim)

print("\nAdditional architecture keys:")
for key in [
    "ROPE_BASE",
    "WINDOW",
    "GLOBAL_EVERY",
    "DROPOUT",
    "MAX_SEQ_LEN",
    "SEQ_LEN",
]:
    print(f"{key:20s}: {cfg.get(key)}")

print("\nAttention-related config:")
for key, value in cfg.items():
    if any(term in key.upper() for term in [
        "ATTN", "HEAD", "WINDOW", "GLOBAL", "ROPE", "CAUSAL"
    ]):
        print(f"{key:25s}: {value}")

del ckpt, state

ARCH 4 ATTENTION DIMENSION CHECK
Hidden dimension       : 1024
Query heads             : 16
KV heads                : 8
Head dimension          : 64
Expected Q dimension    : 1024
Expected K dimension    : 512
Expected V dimension    : 512
Expected total QKV      : 2048
Actual QKV projection   : 2048

Additional architecture keys:
ROPE_BASE           : 10000.0
WINDOW              : 256
GLOBAL_EVERY        : 3
DROPOUT             : 0.0
MAX_SEQ_LEN         : 1024
SEQ_LEN             : 1024

Attention-related config:
N_Q_HEADS                : 16
N_KV_HEADS               : 8
WINDOW                   : 256
GLOBAL_EVERY             : 3
ROPE_BASE                : 10000.0


In [14]:
from pathlib import Path

SEARCH_ROOTS = [
    Path("/kaggle/working"),
    Path("/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m"),
]

KEYWORDS = [
    "class Arch4",
    "class Hybrid",
    "class GQA",
    "qkv_proj",
    "GLOBAL_EVERY",
]

print("=" * 70)
print("SEARCHING FOR EXISTING ARCH 4 IMPLEMENTATION")
print("=" * 70)

found = set()

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*.py"):
        try:
            # Skip excessively large source files.
            if path.stat().st_size > 2_000_000:
                continue

            content = path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

            matched = [
                keyword
                for keyword in KEYWORDS
                if keyword in content
            ]

            if matched:
                found.add(path)
                print(f"\nFile: {path}")
                print("Matched:", ", ".join(matched))

        except (OSError, PermissionError):
            continue

if not found:
    print(
        "\nNo matching Python implementation found "
        "in the searched directories."
    )

SEARCHING FOR EXISTING ARCH 4 IMPLEMENTATION

No matching Python implementation found in the searched directories.


In [15]:
import torch
from pathlib import Path

CHECKPOINT_PATH = Path(
    "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/"
    "transformers/default/1/arch4_hybrid_gqa_201m/best.pt"
)

ckpt = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

cfg = ckpt["cfg"]
state = ckpt["model"]

print("=" * 70)
print("FULL CONFIGURATION")
print("=" * 70)

for key, value in cfg.items():
    print(f"{key}: {value}")

print("\n" + "=" * 70)
print("ALL NON-LAYER PARAMETERS")
print("=" * 70)

for key, value in state.items():
    if not key.startswith("layers."):
        print(f"{key:40s} {tuple(value.shape)}")

print("\n" + "=" * 70)
print("FIRST LAYER COMPLETE PARAMETER SHAPES")
print("=" * 70)

for key, value in state.items():
    if key.startswith("layers.0."):
        print(f"{key:55s} {tuple(value.shape)}")

print("\n" + "=" * 70)
print("LAST LAYER COMPLETE PARAMETER SHAPES")
print("=" * 70)

for key, value in state.items():
    if key.startswith("layers.11."):
        print(f"{key:55s} {tuple(value.shape)}")

del ckpt, state

FULL CONFIGURATION
DATA_DIR: /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1
VOCAB_SIZE: 40000
D_MODEL: 1024
N_LAYERS: 12
N_Q_HEADS: 16
N_KV_HEADS: 8
D_FF: 3328
MAX_SEQ_LEN: 1024
SEQ_LEN: 1024
WINDOW: 256
GLOBAL_EVERY: 3
ROPE_BASE: 10000.0
DROPOUT: 0.0
INIT_STD: 0.02
TOKENS_PER_STEP: 16384
CANDIDATES: [(8, False), (4, False), (8, True), (4, True), (2, True)]
MAX_STEPS: 10000
LR: 0.0003
MIN_LR_RATIO: 0.1
WARMUP_STEPS: 500
WEIGHT_DECAY: 0.1
GRAD_CLIP: 1.0
LOSS_CHUNK: 2048
USE_FUSED_ADAMW: True
USE_COMPILE: True
COMPILE_MODE: default
EVAL_EVERY: 500
EVAL_BATCHES_PER_SOURCE: 10
SAVE_EVERY: 500
LOG_EVERY: 50
TIME_LIMIT_HOURS: 11.0
OUTPUT_DIR: /kaggle/working/arch4_hybrid_gqa_201m
RESUME_FROM: None
SEED: 42

ALL NON-LAYER PARAMETERS
token_embedding.weight                   (40000, 1024)
final_norm.weight                        (1024,)
final_norm.bias                          (1024,)
lm_head.weight                           (40000, 1024)

FIRST LAYER COMPLETE PARAMETER SHAPES
layer

In [16]:
from pathlib import Path

SEARCH_ROOTS = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

KEYWORDS = [
    "class Arch4",
    "class HybridGQA",
    "class GQA",
    "GLOBAL_EVERY",
    "qkv_proj",
    "ROPE_BASE",
    "arch4_hybrid_gqa_201m",
]

extensions = {".py", ".ipynb", ".txt", ".md"}

matches = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in extensions:
            continue

        try:
            if path.stat().st_size > 5_000_000:
                continue

            content = path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

            found = [
                keyword
                for keyword in KEYWORDS
                if keyword in content
            ]

            if found:
                matches.append((path, found))

        except (OSError, PermissionError):
            continue

print("Files containing relevant architecture clues:", len(matches))

for path, found in matches[:100]:
    print(f"\n{path}")
    print("  Matches:", ", ".join(found))

Files containing relevant architecture clues: 1

/kaggle/working/__notebook__.ipynb
  Matches: class Arch4, class HybridGQA, class GQA, GLOBAL_EVERY, qkv_proj, ROPE_BASE, arch4_hybrid_gqa_201m


In [17]:
from pathlib import Path

NOTEBOOK_PATH = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

print("File exists:", NOTEBOOK_PATH.exists())

if NOTEBOOK_PATH.exists():
    print("File size:", NOTEBOOK_PATH.stat().st_size, "bytes")

    with open(NOTEBOOK_PATH, "rb") as f:
        raw = f.read(1000)

    print("\nFirst 500 bytes:")
    print(repr(raw[:500]))

    print("\nFirst 500 bytes as text:")
    print(raw.decode("utf-8", errors="replace")[:500])

File exists: False


In [18]:
from pathlib import Path

SOURCE_PATH = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

source = SOURCE_PATH.read_text(
    encoding="utf-8",
    errors="replace"
)

lines = source.splitlines()

keywords = [
    "class Arch4",
    "class HybridGQA",
    "class GQA",
    "def build_arch4",
    "def apply_rope",
    "def rotary",
]

print("=" * 80)
print("ARCH 4 SOURCE LOCATIONS")
print("=" * 80)

found = []

for i, line in enumerate(lines):
    if any(keyword in line for keyword in keywords):
        found.append(i)
        print(f"Line {i + 1}: {line.strip()}")

print("\nTotal matching lines:", len(found))

print("\n" + "=" * 80)
print("MATCHING CODE WITH CONTEXT")
print("=" * 80)

shown_ranges = []

for idx in found:
    start = max(0, idx - 5)
    end = min(len(lines), idx + 35)

    # Avoid printing overlapping regions repeatedly.
    if any(start <= old_end and end >= old_start
           for old_start, old_end in shown_ranges):
        continue

    shown_ranges.append((start, end))

    print(f"\n--- Lines {start + 1}-{end} ---")
    for j in range(start, end):
        print(f"{j + 1:5d}: {lines[j]}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/.virtual_documents/__notebook_source__.ipynb'

In [ ]:
from pathlib import Path
import re

SOURCE_PATH = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

source = SOURCE_PATH.read_text(
    encoding="utf-8",
    errors="replace"
)

lines = source.splitlines()

patterns = [
    r"^\s*class\s+Arch4\b",
    r"^\s*class\s+HybridGQA\b",
    r"^\s*class\s+GQA\b",
    r"^\s*def\s+build_arch4\b",
    r"^\s*def\s+apply_rope\b",
    r"^\s*def\s+rotary\b",
]

print("=" * 75)
print("ACTUAL PYTHON DEFINITIONS")
print("=" * 75)

matches = []

for i, line in enumerate(lines):
    for pattern in patterns:
        if re.search(pattern, line):
            matches.append(i)
            print(f"Line {i + 1}: {line.strip()}")

if not matches:
    print("No actual class/function definitions found.")

print("\n" + "=" * 75)
print("OTHER CLASS AND FUNCTION DEFINITIONS")
print("=" * 75)

for i, line in enumerate(lines):
    if re.match(r"^\s*(class|def)\s+\w+", line):
        print(f"Line {i + 1}: {line.strip()}")

print("\nTotal source lines:", len(lines))

In [ ]:
import torch
import inspect

print("=" * 70)
print("CHECKING LIVE KAGGLE SESSION")
print("=" * 70)

names = [
    "Arch4",
    "HybridGQA",
    "GQA",
    "build_arch4",
    "apply_rope",
    "rotary",
]

for name in names:
    obj = globals().get(name)

    print(f"\n{name}:")

    if obj is None:
        print("  Not defined in current session")
        continue

    print("  Type:", type(obj))

    try:
        print("  Source file:", inspect.getsourcefile(obj))
    except (TypeError, OSError):
        print("  Source file: unavailable")

    try:
        source = inspect.getsource(obj)
        print("  Source preview:")
        print(source[:2500])
    except (TypeError, OSError):
        print("  Source code unavailable")

print("\n" + "=" * 70)
print("CHECKING OTHER ARCHITECTURE OBJECTS")
print("=" * 70)

for name, obj in list(globals().items()):
    if any(term in name.lower() for term in ["arch4", "hybridgqa"]):
        if not name.startswith("_"):
            print(name, "->", type(obj))

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RotaryEmbedding:
    def __init__(self, head_dim, base=10000.0):
        self.head_dim = head_dim
        self.base = base

    def apply(self, x, positions):
        # x: [B, heads, T, head_dim]
        half = self.head_dim // 2

        inv_freq = 1.0 / (
            self.base
            ** (
                torch.arange(
                    0, self.head_dim, 2,
                    device=x.device,
                    dtype=torch.float32
                ) / self.head_dim
            )
        )

        angles = positions.float().unsqueeze(-1) * inv_freq
        cos = angles.cos().to(x.dtype)[:, None, :, :]
        sin = angles.sin().to(x.dtype)[:, None, :, :]

        x1 = x[..., :half]
        x2 = x[..., half:]

        # This implementation assumes split-half RoPE.
        return torch.cat(
            [
                x1 * cos - x2 * sin,
                x2 * cos + x1 * sin
            ],
            dim=-1
        )


class Arch4Attention(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()

        self.d_model = cfg["D_MODEL"]
        self.n_q_heads = cfg["N_Q_HEADS"]
        self.n_kv_heads = cfg["N_KV_HEADS"]
        self.head_dim = self.d_model // self.n_q_heads

        self.window = cfg["WINDOW"]
        self.global_every = cfg["GLOBAL_EVERY"]

        self.is_global = (
            (layer_idx + 1) % self.global_every == 0
        )

        self.rope = RotaryEmbedding(
            self.head_dim,
            cfg["ROPE_BASE"]
        )

        self.qkv_proj = nn.Linear(
            self.d_model,
            self.d_model + 2 * self.n_kv_heads * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            self.d_model,
            self.d_model,
            bias=False
        )

    def forward(self, x):
        B, T, D = x.shape

        qkv = self.qkv_proj(x)

        q_size = self.n_q_heads * self.head_dim
        kv_size = self.n_kv_heads * self.head_dim

        q, k, v = torch.split(
            qkv,
            [q_size, kv_size, kv_size],
            dim=-1
        )

        q = q.view(
            B, T, self.n_q_heads, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T, self.n_kv_heads, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T, self.n_kv_heads, self.head_dim
        ).transpose(1, 2)

        positions = torch.arange(T, device=x.device)

        q = self.rope.apply(q, positions)
        k = self.rope.apply(k, positions)

        # Expand grouped K/V heads to match query heads.
        repeat_factor = self.n_q_heads // self.n_kv_heads

        k = k.repeat_interleave(repeat_factor, dim=1)
        v = v.repeat_interleave(repeat_factor, dim=1)

        # Causal mask: local layers attend to the preceding window;
        # global layers attend to all preceding tokens.
        query_pos = torch.arange(T, device=x.device)[:, None]
        key_pos = torch.arange(T, device=x.device)[None, :]

        allowed = key_pos <= query_pos

        if not self.is_global:
            allowed = allowed & (
                key_pos > query_pos - self.window
            )

        attn_mask = allowed[None, None, :, :]

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=0.0,
            is_causal=False
        )

        out = out.transpose(1, 2).contiguous().view(B, T, D)

        return self.out_proj(out)


class Arch4FFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.in_proj = nn.Linear(
            cfg["D_MODEL"],
            2 * cfg["D_FF"],
            bias=False
        )

        self.out_proj = nn.Linear(
            cfg["D_FF"],
            cfg["D_MODEL"],
            bias=False
        )

    def forward(self, x):
        gate, up = self.in_proj(x).chunk(2, dim=-1)

        # Assumed gated-SiLU FFN; original activation is unverified.
        return self.out_proj(F.silu(gate) * up)


class Arch4Block(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()

        self.norm1 = nn.LayerNorm(cfg["D_MODEL"])
        self.attn = Arch4Attention(cfg, layer_idx)

        self.norm2 = nn.LayerNorm(cfg["D_MODEL"])
        self.ffn = Arch4FFN(cfg)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class Arch4(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.cfg = cfg

        self.token_embedding = nn.Embedding(
            cfg["VOCAB_SIZE"],
            cfg["D_MODEL"]
        )

        self.layers = nn.ModuleList([
            Arch4Block(cfg, i)
            for i in range(cfg["N_LAYERS"])
        ])

        self.final_norm = nn.LayerNorm(cfg["D_MODEL"])

        self.lm_head = nn.Linear(
            cfg["D_MODEL"],
            cfg["VOCAB_SIZE"],
            bias=False
        )

    def forward(self, input_ids, return_hidden=False):
        x = self.token_embedding(input_ids)

        for layer in self.layers:
            x = layer(x)

        hidden = self.final_norm(x)

        if return_hidden:
            return hidden

        return self.lm_head(hidden)

In [ ]:
import inspect

print(inspect.getsource(RotaryEmbedding.apply))

In [ ]:
class RotaryEmbedding:
    def __init__(self, head_dim, base=10000.0):
        self.head_dim = head_dim
        self.base = base

    def apply(self, x, positions):
        half = self.head_dim // 2

        inv_freq = 1.0 / (
            self.base ** (
                torch.arange(
                    0, self.head_dim, 2,
                    device=x.device,
                    dtype=torch.float32
                ) / self.head_dim
            )
        )

        angles = positions.float().unsqueeze(-1) * inv_freq

        cos = angles.cos().to(dtype=x.dtype)[None, None, :, :]
        sin = angles.sin().to(dtype=x.dtype)[None, None, :, :]

        x1 = x[..., :half]
        x2 = x[..., half:]

        return torch.cat(
            (
                x1 * cos - x2 * sin,
                x2 * cos + x1 * sin
            ),
            dim=-1
        )

print(inspect.getsource(RotaryEmbedding.apply))

In [ ]:
import os
import torch
from pathlib import Path

# Phase 3 dataset
DATA_DIR = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")

# Pretrained Arch4 checkpoint
CHECKPOINT_PATH = Path(
    "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/"
    "transformers/default/1/arch4_hybrid_gqa_201m/best.pt"
)

# Tokenizer
TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/achiver07/"
    "phase1-tokens-v40k-clean1/tokenizer.json"
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n--- Dataset files ---")

for filename in [
    "CL_train.json",
    "CL_dev.json",
    "CL_test.json",
    "IT_train.json",
    "IT_dev.json",
    "IT_test.json",
    "label_vocab.json",
]:
    path = DATA_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

print("\n--- Model files ---")
print("Arch4 checkpoint:", CHECKPOINT_PATH.exists())
print("Tokenizer:", TOKENIZER_PATH.exists())

In [ ]:
from pathlib import Path

SEARCH_DIRS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

TARGET_FILES = {
    "CL_train.json",
    "CL_dev.json",
    "CL_test.json",
    "IT_train.json",
    "IT_dev.json",
    "IT_test.json",
    "label_vocab.json",
}

found_files = {}

for root in SEARCH_DIRS:
    if not root.exists():
        continue

    print(f"Searching: {root}")

    for path in root.rglob("*"):
        if path.is_file() and path.name in TARGET_FILES:
            found_files[path.name] = path
            print(f"  FOUND: {path}")

print("\n--- SEARCH SUMMARY ---")

for name in sorted(TARGET_FILES):
    if name in found_files:
        print(f"OK      {name}: {found_files[name]}")
    else:
        print(f"MISSING {name}")

if len(found_files) == len(TARGET_FILES):
    print("\nAll dataset files found!")
else:
    print(
        f"\nFound {len(found_files)}/{len(TARGET_FILES)} files."
    )
    print(
        "If files are missing, open Kaggle's Add Input panel "
        "and attach the Phase 3 dataset."
    )
    

In [ ]:
import json
from pathlib import Path
from collections import Counter

BASE_DIR = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")
TRAIN_DIR = BASE_DIR / "Train"
TEST_DIR = BASE_DIR / "Test"

# Load JSON files
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

CL_train = load_json(TRAIN_DIR / "CL_train.json")
CL_dev   = load_json(TRAIN_DIR / "CL_dev.json")
IT_train = load_json(TRAIN_DIR / "IT_train.json")
IT_dev   = load_json(TRAIN_DIR / "IT_dev.json")

CL_test = load_json(TEST_DIR / "CL_test.json")
IT_test = load_json(TEST_DIR / "IT_test.json")

label_vocab = load_json(TRAIN_DIR / "label_vocab.json")

print("Dataset loaded successfully!")
print("CL train documents:", len(CL_train))
print("CL dev documents:", len(CL_dev))
print("IT train documents:", len(IT_train))
print("IT dev documents:", len(IT_dev))
print("CL test documents:", len(CL_test))
print("IT test documents:", len(IT_test))

print("\nLabel vocabulary:")
print(label_vocab)

# Inspect one training document
print("\n--- Sample document structure ---")
sample = CL_train[0]
print("Document keys:", list(sample.keys()))
print("Document ID:", sample["id"])
print("Number of sentences:", len(sample["text"]))
print("Number of labels:", len(sample["labels"]))
print("\nFirst 3 sentences:")
for sentence, label in zip(sample["text"][:3], sample["labels"][:3]):
    print("TEXT :", str(sentence)[:200])
    print("LABEL:", label)
    print()

In [ ]:
from pathlib import Path
from collections import Counter
from tokenizers import Tokenizer
import torch

# Paths
TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
)

ARCH4_CHECKPOINT = Path(
    "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/"
    "transformers/default/1/arch4_hybrid_gqa_201m/best.pt"
)

OUTPUT_DIR = Path("/kaggle/working/Phase3_Submissions/04_arch4_lora")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load tokenizer
assert TOKENIZER_PATH.exists(), f"Tokenizer missing: {TOKENIZER_PATH}"
assert ARCH4_CHECKPOINT.exists(), f"Arch4 checkpoint missing: {ARCH4_CHECKPOINT}"

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print("Tokenizer loaded")
print("Tokenizer vocabulary:", tokenizer.get_vocab_size())
print("Arch4 checkpoint exists:", ARCH4_CHECKPOINT.exists())
print("Output directory:", OUTPUT_DIR)

# Confirm special tokens
for token in ["<pad>", "<bos>", "<eos>"]:
    print(f"{token} ->", tokenizer.token_to_id(token))

# Check labels
num_labels = len(label_vocab)
print("\nNumber of labels:", num_labels)
print("Label IDs:", sorted(label_vocab.values()))

# Validate document structures and labels
all_train_docs = CL_train + IT_train
all_dev_docs = CL_dev + IT_dev

label_counts = Counter()
sentence_count = 0
unknown_labels = set()
empty_sentences = 0

for doc in all_train_docs + all_dev_docs:
    assert len(doc["text"]) == len(doc["labels"]), (
        f"Text/label length mismatch in document {doc['id']}"
    )

    for sentence, label in zip(doc["text"], doc["labels"]):
        sentence_count += 1
        label_counts[label] += 1

        if label not in label_vocab:
            unknown_labels.add(label)

        if not str(sentence).strip():
            empty_sentences += 1

print("\nDataset validation")
print("Total train + dev sentences:", sentence_count)
print("Empty sentences:", empty_sentences)
print("Unknown labels:", unknown_labels)

print("\nLabel distribution:")
for label, count in label_counts.most_common():
    print(f"{label:28s} {count:5d}")

# Token length sample
sample_sentences = [
    sentence
    for doc in all_train_docs[:5]
    for sentence in doc["text"][:20]
]

token_lengths = [
    len(tokenizer.encode(str(sentence)).ids)
    for sentence in sample_sentences
]

if token_lengths:
    token_lengths_sorted = sorted(token_lengths)
    print("\nToken length sample:")
    print("Count:", len(token_lengths))
    print("Min:", min(token_lengths))
    print("Median:", token_lengths_sorted[len(token_lengths_sorted) // 2])
    print("Max:", max(token_lengths))

In [ ]:
import torch
from pathlib import Path

ARCH4_CHECKPOINT = Path(
    "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/"
    "transformers/default/1/arch4_hybrid_gqa_201m/best.pt"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

# Load checkpoint on CPU to avoid unnecessary GPU memory usage
checkpoint = torch.load(
    ARCH4_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys()))

    cfg = checkpoint.get("cfg", {})
    print("\nModel configuration:")
    if isinstance(cfg, dict):
        for key in [
            "VOCAB_SIZE", "D_MODEL", "N_LAYERS",
            "N_Q_HEADS", "N_KV_HEADS", "D_FF",
            "MAX_SEQ_LEN", "SEQ_LEN", "WINDOW",
            "GLOBAL_EVERY", "ROPE_BASE", "DROPOUT"
        ]:
            if key in cfg:
                print(f"{key}: {cfg[key]}")
    else:
        print("Config type:", type(cfg))

    # Locate model weights
    state_dict = None
    for key in ["model", "model_state_dict", "state_dict", "model_state"]:
        if key in checkpoint and isinstance(checkpoint[key], dict):
            state_dict = checkpoint[key]
            print("\nWeights found under:", key)
            break

    if state_dict is None:
        if any(
            isinstance(k, str) and k.startswith("layers.")
            for k in checkpoint.keys()
        ):
            state_dict = checkpoint
            print("\nCheckpoint itself appears to contain the state dict.")

    if state_dict is not None:
        print("Number of parameter tensors:", len(state_dict))

        print("\nFirst 25 state-dict entries:")
        for key, value in list(state_dict.items())[:25]:
            if torch.is_tensor(value):
                print(f"{key:55s} {tuple(value.shape)}")
            else:
                print(f"{key:55s} {type(value)}")
    else:
        print("\nCould not automatically identify the model state dict.")

In [ ]:
import torch

state_dict = checkpoint["model"]
cfg = checkpoint["cfg"]

print("=== Attention and FFN structure ===")

for i in range(cfg["N_LAYERS"]):
    prefix = f"layers.{i}."

    qkv_shape = tuple(state_dict[prefix + "attn.qkv_proj.weight"].shape)
    out_shape = tuple(state_dict[prefix + "attn.out_proj.weight"].shape)
    ffn_in_shape = tuple(state_dict[prefix + "ffn.in_proj.weight"].shape)
    ffn_out_shape = tuple(state_dict[prefix + "ffn.out_proj.weight"].shape)

    print(
        f"Layer {i:02d} | "
        f"QKV={qkv_shape} | "
        f"AttnOut={out_shape} | "
        f"FFN-In={ffn_in_shape} | "
        f"FFN-Out={ffn_out_shape}"
    )

print("\n=== Remaining checkpoint parameters ===")

for key, value in state_dict.items():
    if key.startswith("layers.0."):
        continue

    if not key.startswith("layers."):
        if torch.is_tensor(value):
            print(f"{key:35s} {tuple(value.shape)}")
        else:
            print(f"{key:35s} {type(value)}")

print("\n=== Configuration checks ===")

d_model = cfg["D_MODEL"]
n_q_heads = cfg["N_Q_HEADS"]
n_kv_heads = cfg["N_KV_HEADS"]

print("Head dimension:", d_model // n_q_heads)
print("Query heads:", n_q_heads)
print("KV heads:", n_kv_heads)
print("Queries per KV head:", n_q_heads // n_kv_heads)
print("Layers:", cfg["N_LAYERS"])
print("Vocabulary:", cfg["VOCAB_SIZE"])

In [ ]:
import torch

required_names = [
    "RotaryEmbedding",
    "Arch4Attention",
    "Arch4FFN",
    "Arch4Block",
    "Arch4",
]

print("=== Arch4 implementation check ===")

for name in required_names:
    print(f"{name}: {'Available' if name in globals() else 'MISSING'}")

print("\n=== Checkpoint checks ===")
print("Checkpoint loaded:", "checkpoint" in globals())
print("Configuration loaded:", "cfg" in globals())
print("State dict loaded:", "state_dict" in globals())

print("\n=== GPU status ===")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Allocated GPU memory:",
        round(torch.cuda.memory_allocated(0) / 1024**3, 2),
        "GB"
    )

In [ ]:
import torch

# Reuse the checkpoint and configuration already loaded.
# Load model weights strictly to catch missing/unexpected parameters.

model = Arch4(cfg)

load_result = model.load_state_dict(state_dict, strict=True)
print("Checkpoint loaded strictly:", load_result)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

# Short test sequence; special tokens are BOS + EOS.
test_ids = tokenizer.encode(
    "The court considered the arguments presented by both parties."
).ids

input_ids = torch.tensor(
    [[1] + test_ids + [2]],
    dtype=torch.long,
    device=device
)

print("\nInput shape:", tuple(input_ids.shape))

with torch.no_grad():
    output = model(input_ids)

# Support models that return either logits or (hidden states, logits).
if isinstance(output, tuple):
    print("Output is a tuple; component shapes:",
          [tuple(x.shape) for x in output if torch.is_tensor(x)])
    logits = next(
        x for x in reversed(output)
        if torch.is_tensor(x) and x.ndim == 3
    )
elif isinstance(output, dict):
    print("Output keys:", list(output.keys()))
    logits = output.get("logits")
    if logits is None:
        raise ValueError("Model output dictionary has no 'logits' entry.")
else:
    logits = output

assert logits.ndim == 3, f"Expected [batch, seq, vocab], got {logits.shape}"
assert logits.shape[0] == 1
assert logits.shape[1] == input_ids.shape[1]
assert torch.isfinite(logits).all(), "Non-finite logits detected."

print("\nLogits shape:", tuple(logits.shape))
print("All logits finite:", bool(torch.isfinite(logits).all()))
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("\nForward-pass test passed.")

In [ ]:
import inspect

print("Arch4 forward signature:")
print(inspect.signature(model.forward))

print("\nArch4 forward implementation:")
print(inspect.getsource(model.forward))

In [ ]:
import numpy as np
import torch
from tqdm.auto import tqdm

# Model and tokenizer should already be loaded.
model.eval()
device = next(model.parameters()).device

MAX_TOKENS = 256
BATCH_SIZE = 8

label_to_id = label_vocab
num_labels = len(label_to_id)

@torch.no_grad()
def encode_sentences(sentences, batch_size=BATCH_SIZE):
    """
    Encode each sentence independently.
    Returns the final-token hidden representation for each sentence.
    """
    all_embeddings = []

    for start in tqdm(range(0, len(sentences), batch_size)):
        batch_sentences = sentences[start:start + batch_size]
        batch_ids = []

        for sentence in batch_sentences:
            ids = tokenizer.encode(str(sentence)).ids

            # Reserve space for BOS and EOS.
            ids = ids[:MAX_TOKENS - 2]
            ids = [1] + ids + [2]
            batch_ids.append(ids)

        max_len = max(len(ids) for ids in batch_ids)

        input_ids = torch.full(
            (len(batch_ids), max_len),
            fill_value=0,
            dtype=torch.long,
            device=device
        )

        attention_mask = torch.zeros(
            (len(batch_ids), max_len),
            dtype=torch.bool,
            device=device
        )

        for i, ids in enumerate(batch_ids):
            input_ids[i, :len(ids)] = torch.tensor(ids, device=device)
            attention_mask[i, :len(ids)] = True

        # The reconstructed Arch4 forward method accepts input_ids only.
        # Process each sequence at its own length by padding to the batch max.
        hidden = model(input_ids, return_hidden=True)

        # Select each sentence's final non-padding token (EOS).
        lengths = attention_mask.sum(dim=1)
        rows = torch.arange(len(batch_ids), device=device)
        sentence_embeddings = hidden[rows, lengths - 1]

        all_embeddings.append(
            sentence_embeddings.float().cpu().numpy()
        )

    return np.concatenate(all_embeddings, axis=0)


def flatten_documents(documents):
    texts, labels, doc_ids = [], [], []

    for doc in documents:
        for sentence, label in zip(doc["text"], doc["labels"]):
            texts.append(str(sentence))
            labels.append(label_to_id[label])
            doc_ids.append(doc["id"])

    return texts, np.asarray(labels, dtype=np.int64), doc_ids


# Flatten train/dev sets while preserving sentence labels.
train_docs = CL_train + IT_train
dev_docs = CL_dev + IT_dev

train_texts, y_train, train_doc_ids = flatten_documents(train_docs)
dev_texts, y_dev, dev_doc_ids = flatten_documents(dev_docs)

print("Train sentences:", len(train_texts))
print("Dev sentences:", len(dev_texts))

# Extract representations.
X_train = encode_sentences(train_texts)
X_dev = encode_sentences(dev_texts)

# Save embeddings so they can be reused without running Arch4 again.
np.savez_compressed(
    OUTPUT_DIR / "arch4_sentence_embeddings.npz",
    X_train=X_train,
    y_train=y_train,
    X_dev=X_dev,
    y_dev=y_dev
)

print("\nEmbedding extraction complete.")
print("X_train:", X_train.shape)
print("X_dev:", X_dev.shape)
print("y_train:", y_train.shape)
print("y_dev:", y_dev.shape)
print("Saved to:", OUTPUT_DIR / "arch4_sentence_embeddings.npz")

In [ ]:
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report
)

# -----------------------------
# 1. Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# 2. Load cached embeddings
# -----------------------------
embedding_file = OUTPUT_DIR / "arch4_sentence_embeddings.npz"

saved = np.load(embedding_file)

X_train = saved["X_train"].astype(np.float32)
y_train = saved["y_train"].astype(np.int64)
X_dev = saved["X_dev"].astype(np.float32)
y_dev = saved["y_dev"].astype(np.int64)

num_labels = len(label_vocab)
id_to_label = {v: k for k, v in label_vocab.items()}

print("Train:", X_train.shape, y_train.shape)
print("Dev:", X_dev.shape, y_dev.shape)
print("Number of classes:", num_labels)

# -----------------------------
# 3. Class weights
# -----------------------------
class_counts = np.bincount(y_train, minlength=num_labels)

# Square-root inverse-frequency weighting is less aggressive
# than full inverse-frequency weighting for rare classes.
weights = 1.0 / np.sqrt(np.maximum(class_counts, 1))
weights = weights / weights.mean()

class_weights = torch.tensor(
    weights, dtype=torch.float32, device=device
)

print("\nTraining label counts:")
for class_id in range(num_labels):
    print(
        f"{id_to_label[class_id]:28s}",
        int(class_counts[class_id]),
        f"| weight={weights[class_id]:.3f}"
    )

# -----------------------------
# 4. Data loaders
# -----------------------------
train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train)
)

dev_dataset = TensorDataset(
    torch.from_numpy(X_dev),
    torch.from_numpy(y_dev)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0
)

# -----------------------------
# 5. Classifier
# -----------------------------
class SentenceClassifier(nn.Module):
    def __init__(self, input_dim=1024, num_classes=13, dropout=0.30):
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.network(x)


classifier = SentenceClassifier(
    input_dim=X_train.shape[1],
    num_classes=num_labels
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-2
)

# -----------------------------
# 6. Evaluation
# -----------------------------
@torch.no_grad()
def evaluate_classifier():
    classifier.eval()
    all_preds, all_targets = [], []

    for features, targets in dev_loader:
        features = features.to(device)
        logits = classifier(features)

        predictions = logits.argmax(dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_targets.extend(targets.numpy())

    macro_f1 = f1_score(
        all_targets,
        all_preds,
        labels=list(range(num_labels)),
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(all_targets, all_preds)

    return macro_f1, accuracy, np.asarray(all_targets), np.asarray(all_preds)


# -----------------------------
# 7. Training with early stopping
# -----------------------------
EPOCHS = 40
PATIENCE = 7

best_macro_f1 = -1.0
best_epoch = 0
best_state = None
epochs_without_improvement = 0

history = []

for epoch in range(1, EPOCHS + 1):
    classifier.train()
    total_loss = 0.0
    total_examples = 0

    for features, targets in train_loader:
        features = features.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = classifier(features)
        loss = criterion(logits, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            classifier.parameters(), max_norm=1.0
        )
        optimizer.step()

        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    train_loss = total_loss / total_examples
    val_f1, val_acc, _, _ = evaluate_classifier()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_macro_f1": val_f1,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"loss={train_loss:.4f} | "
        f"val_macro_f1={val_f1:.4f} | "
        f"val_accuracy={val_acc:.4f}"
    )

    if val_f1 > best_macro_f1:
        best_macro_f1 = val_f1
        best_epoch = epoch
        best_state = copy.deepcopy(classifier.state_dict())
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": best_state,
                "input_dim": X_train.shape[1],
                "num_labels": num_labels,
                "label_vocab": label_vocab,
                "best_epoch": best_epoch,
                "best_val_macro_f1": best_macro_f1
            },
            OUTPUT_DIR / "arch4_frozen_classifier_best.pt"
        )
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

# -----------------------------
# 8. Best-model report
# -----------------------------
classifier.load_state_dict(best_state)

best_f1, best_acc, targets, predictions = evaluate_classifier()

print("\n" + "=" * 60)
print("BEST VALIDATION RESULTS")
print("=" * 60)
print("Best epoch:", best_epoch)
print(f"Macro-F1: {best_f1:.4f}")
print(f"Accuracy: {best_acc:.4f}")

print("\nPer-class classification report:")
print(
    classification_report(
        targets,
        predictions,
        labels=list(range(num_labels)),
        target_names=[id_to_label[i] for i in range(num_labels)],
        digits=4,
        zero_division=0
    )
)

with open(OUTPUT_DIR / "classifier_training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\nBest classifier saved to:")
print(OUTPUT_DIR / "arch4_frozen_classifier_best.pt")

In [ ]:








import numpy as np
from pathlib import Path

# Existing sentence embeddings and labels
saved = np.load(OUTPUT_DIR / "arch4_sentence_embeddings.npz")

X_train = saved["X_train"].astype(np.float32)
y_train = saved["y_train"].astype(np.int64)

X_dev = saved["X_dev"].astype(np.float32)
y_dev = saved["y_dev"].astype(np.int64)

# Preserve the same ordering used during flattening.
train_docs = CL_train + IT_train
dev_docs = CL_dev + IT_dev

assert len(X_train) == sum(len(d["text"]) for d in train_docs)
assert len(X_dev) == sum(len(d["text"]) for d in dev_docs)

def build_context_indices(documents):
    """
    Create indices for previous, current, and next sentences.
    Context never crosses document boundaries.
    """
    prev_indices = []
    curr_indices = []
    next_indices = []

    offset = 0

    for doc in documents:
        n = len(doc["text"])

        for j in range(n):
            curr = offset + j
            prev = offset + j - 1 if j > 0 else curr
            nxt = offset + j + 1 if j < n - 1 else curr

            prev_indices.append(prev)
            curr_indices.append(curr)
            next_indices.append(nxt)

        offset += n

    return (
        np.asarray(prev_indices, dtype=np.int64),
        np.asarray(curr_indices, dtype=np.int64),
        np.asarray(next_indices, dtype=np.int64)
    )

train_prev, train_curr, train_next = build_context_indices(train_docs)
dev_prev, dev_curr, dev_next = build_context_indices(dev_docs)

# Concatenate previous/current/next embeddings.
X_train_ctx = np.concatenate(
    [X_train[train_prev], X_train[train_curr], X_train[train_next]],
    axis=1
)

X_dev_ctx = np.concatenate(
    [X_dev[dev_prev], X_dev[dev_curr], X_dev[dev_next]],
    axis=1
)

print("Contextual training features:", X_train_ctx.shape)
print("Contextual validation features:", X_dev_ctx.shape)
print("Training labels:", y_train.shape)
print("Validation labels:", y_dev.shape)

assert X_train_ctx.shape == (len(y_train), 3072)
assert X_dev_ctx.shape == (len(y_dev), 3072)

CONTEXT_DIR = Path(
    "/kaggle/working/Phase3_Submissions/05_arch4_context"
)
CONTEXT_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    CONTEXT_DIR / "arch4_context_embeddings.npz",
    X_train=X_train_ctx,
    y_train=y_train,
    X_dev=X_dev_ctx,
    y_dev=y_dev
)

print("Saved:", CONTEXT_DIR / "arch4_context_embeddings.npz")

In [ ]:
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load contextual embeddings
CONTEXT_DIR = Path(
    "/kaggle/working/Phase3_Submissions/05_arch4_context"
)

data = np.load(CONTEXT_DIR / "arch4_context_embeddings.npz")

X_train_ctx = data["X_train"].astype(np.float32)
y_train_ctx = data["y_train"].astype(np.int64)
X_dev_ctx = data["X_dev"].astype(np.float32)
y_dev_ctx = data["y_dev"].astype(np.int64)

num_labels = len(label_vocab)
id_to_label = {v: k for k, v in label_vocab.items()}

# Class weights: square-root inverse frequency
counts = np.bincount(y_train_ctx, minlength=num_labels)
weights = 1.0 / np.sqrt(np.maximum(counts, 1))
weights = weights / weights.mean()

class_weights = torch.tensor(
    weights, dtype=torch.float32, device=device
)

print("Device:", device)
print("Training features:", X_train_ctx.shape)
print("Validation features:", X_dev_ctx.shape)

# Datasets
train_dataset = TensorDataset(
    torch.from_numpy(X_train_ctx),
    torch.from_numpy(y_train_ctx)
)

dev_dataset = TensorDataset(
    torch.from_numpy(X_dev_ctx),
    torch.from_numpy(y_dev_ctx)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0
)

# Contextual classifier
class ContextSentenceClassifier(nn.Module):
    def __init__(self, input_dim=3072, num_classes=13):
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 768),
            nn.GELU(),
            nn.Dropout(0.35),
            nn.Linear(768, 384),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        return self.network(x)


classifier_ctx = ContextSentenceClassifier(
    input_dim=X_train_ctx.shape[1],
    num_classes=num_labels
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    classifier_ctx.parameters(),
    lr=5e-4,
    weight_decay=1e-2
)

@torch.no_grad()
def evaluate_context_classifier():
    classifier_ctx.eval()

    predictions = []
    targets_all = []

    for features, targets in dev_loader:
        logits = classifier_ctx(features.to(device))
        predictions.extend(logits.argmax(dim=1).cpu().numpy())
        targets_all.extend(targets.numpy())

    macro_f1 = f1_score(
        targets_all,
        predictions,
        labels=list(range(num_labels)),
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(targets_all, predictions)

    return (
        macro_f1,
        accuracy,
        np.asarray(targets_all),
        np.asarray(predictions)
    )


# Training
EPOCHS = 40
PATIENCE = 7

best_f1 = -1.0
best_epoch = 0
best_state = None
no_improvement = 0
history_ctx = []

for epoch in range(1, EPOCHS + 1):
    classifier_ctx.train()

    running_loss = 0.0
    total = 0

    for features, targets in train_loader:
        features = features.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = classifier_ctx(features)
        loss = criterion(logits, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            classifier_ctx.parameters(), max_norm=1.0
        )
        optimizer.step()

        running_loss += loss.item() * len(targets)
        total += len(targets)

    train_loss = running_loss / total
    val_f1, val_acc, _, _ = evaluate_context_classifier()

    history_ctx.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_macro_f1": val_f1,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"loss={train_loss:.4f} | "
        f"val_macro_f1={val_f1:.4f} | "
        f"val_accuracy={val_acc:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        best_state = copy.deepcopy(classifier_ctx.state_dict())
        no_improvement = 0

        torch.save(
            {
                "model_state_dict": best_state,
                "input_dim": X_train_ctx.shape[1],
                "num_labels": num_labels,
                "label_vocab": label_vocab,
                "best_epoch": best_epoch,
                "best_val_macro_f1": best_f1
            },
            CONTEXT_DIR / "arch4_context_classifier_best.pt"
        )
    else:
        no_improvement += 1

    if no_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

# Restore best model
classifier_ctx.load_state_dict(best_state)

best_f1, best_acc, targets, predictions = evaluate_context_classifier()

print("\n" + "=" * 55)
print("BEST CONTEXTUAL CLASSIFIER RESULTS")
print("=" * 55)
print("Best epoch:", best_epoch)
print(f"Macro-F1: {best_f1:.4f}")
print(f"Accuracy: {best_acc:.4f}")

print("\nPer-class report:")
print(classification_report(
    targets,
    predictions,
    labels=list(range(num_labels)),
    target_names=[id_to_label[i] for i in range(num_labels)],
    digits=4,
    zero_division=0
))

with open(CONTEXT_DIR / "training_history.json", "w") as f:
    json.dump(history_ctx, f, indent=2)

print("\nSaved best model:")
print(CONTEXT_DIR / "arch4_context_classifier_best.pt")

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path

CONTEXT_DIR = Path(
    "/kaggle/working/Phase3_Submissions/05_arch4_context"
)

GATED_DIR = Path(
    "/kaggle/working/Phase3_Submissions/06_arch4_gated_context"
)
GATED_DIR.mkdir(parents=True, exist_ok=True)


class GatedContextClassifier(nn.Module):
    def __init__(
        self,
        input_dim=1024,
        num_classes=13,
        dropout=0.30
    ):
        super().__init__()

        # Separate projections for previous/current/next sentences.
        self.prev_proj = nn.Linear(input_dim, 512)
        self.curr_proj = nn.Linear(input_dim, 512)
        self.next_proj = nn.Linear(input_dim, 512)

        # Learn how much contextual information to retain.
        self.context_gate = nn.Sequential(
            nn.Linear(512 * 3, 512),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(384, num_classes)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Input layout: [previous, current, next]
        prev, curr, nxt = torch.split(x, 1024, dim=-1)

        prev = self.prev_proj(prev)
        curr = self.curr_proj(curr)
        nxt = self.next_proj(nxt)

        combined = torch.cat([prev, curr, nxt], dim=-1)
        gate = self.context_gate(combined)

        # Current sentence is the anchor; context is gated.
        context = torch.cat([prev, curr, nxt], dim=-1)
        context = context * gate

        # Residual current-sentence signal.
        fused = curr + self.dropout(
            context[:, :512] + context[:, 1024:]
        )

        return self.fusion(fused)


gated_model = GatedContextClassifier(
    input_dim=1024,
    num_classes=len(label_vocab)
).to(device)

print("Gated contextual classifier initialized.")
print("Parameters:", sum(p.numel() for p in gated_model.parameters()))
print("Input dimension: 3072")
print("Output classes:", len(label_vocab))
print("Output directory:", GATED_DIR)

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path

GATED_DIR = Path(
    "/kaggle/working/Phase3_Submissions/06_arch4_gated_context"
)
GATED_DIR.mkdir(parents=True, exist_ok=True)


class GatedContextClassifier(nn.Module):
    def __init__(
        self,
        input_dim=1024,
        num_classes=13,
        dropout=0.30
    ):
        super().__init__()

        self.prev_proj = nn.Linear(input_dim, 512)
        self.curr_proj = nn.Linear(input_dim, 512)
        self.next_proj = nn.Linear(input_dim, 512)

        # Separate gates for previous and next sentence features.
        self.context_gate = nn.Sequential(
            nn.Linear(512 * 3, 1024),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(384, num_classes)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Input: [previous sentence, current sentence, next sentence]
        prev, curr, nxt = torch.split(x, 1024, dim=-1)

        prev = self.prev_proj(prev)
        curr = self.curr_proj(curr)
        nxt = self.next_proj(nxt)

        combined = torch.cat([prev, curr, nxt], dim=-1)

        # Learn separate feature-wise gates for previous and next context.
        gates = self.context_gate(combined)
        gate_prev, gate_next = gates.chunk(2, dim=-1)

        contextual_signal = (
            gate_prev * prev +
            gate_next * nxt
        )

        # Preserve the current sentence as the main signal.
        fused = curr + self.dropout(contextual_signal)

        return self.fusion(fused)


gated_model = GatedContextClassifier(
    input_dim=1024,
    num_classes=len(label_vocab)
).to(device)

# Smoke test before training.
dummy_input = torch.randn(4, 3072, device=device)

with torch.no_grad():
    dummy_output = gated_model(dummy_input)

print("Gated classifier initialized successfully.")
print("Input shape:", tuple(dummy_input.shape))
print("Output shape:", tuple(dummy_output.shape))
print("Parameters:", sum(p.numel() for p in gated_model.parameters()))
print("Device:", device)
print("Output directory:", GATED_DIR)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
from pathlib import Path
import json
import copy

# Paths
CONTEXT_DIR = Path(
    "/kaggle/working/Phase3_Submissions/05_arch4_context"
)
GATED_DIR = Path(
    "/kaggle/working/Phase3_Submissions/06_arch4_gated_context"
)
GATED_DIR.mkdir(parents=True, exist_ok=True)

# Load contextual embeddings
data = np.load(CONTEXT_DIR / "arch4_context_embeddings.npz")

X_train = torch.tensor(data["X_train"], dtype=torch.float32)
y_train = torch.tensor(data["y_train"], dtype=torch.long)
X_dev = torch.tensor(data["X_dev"], dtype=torch.float32)
y_dev = torch.tensor(data["y_dev"], dtype=torch.long)

print("Train:", X_train.shape, y_train.shape)
print("Dev:", X_dev.shape, y_dev.shape)

# Data loaders
BATCH_SIZE = 256

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

dev_loader = DataLoader(
    TensorDataset(X_dev, y_dev),
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

# Square-root inverse-frequency class weights
num_classes = len(label_vocab)
counts = torch.bincount(y_train, minlength=num_classes).float()
class_weights = torch.sqrt(counts.sum() / counts.clamp(min=1))
class_weights = class_weights / class_weights.mean()
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Fresh model and optimizer
gated_model = GatedContextClassifier(
    input_dim=1024,
    num_classes=num_classes,
    dropout=0.30
).to(device)

optimizer = torch.optim.AdamW(
    gated_model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

EPOCHS = 40
PATIENCE = 7
best_f1 = -1.0
best_epoch = 0
patience_counter = 0
history = []

for epoch in range(1, EPOCHS + 1):
    # Training
    gated_model.train()
    train_loss_sum = 0.0
    train_count = 0

    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = gated_model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            gated_model.parameters(), max_norm=1.0
        )
        optimizer.step()

        train_loss_sum += loss.item() * len(yb)
        train_count += len(yb)

    train_loss = train_loss_sum / train_count

    # Validation
    gated_model.eval()
    val_loss_sum = 0.0
    val_count = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for xb, yb in dev_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            logits = gated_model(xb)
            loss = criterion(logits, yb)

            val_loss_sum += loss.item() * len(yb)
            val_count += len(yb)

            all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
            all_targets.extend(yb.cpu().tolist())

    val_loss = val_loss_sum / val_count
    val_macro_f1 = f1_score(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )
    val_accuracy = accuracy_score(all_targets, all_preds)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_f1": val_macro_f1,
        "val_accuracy": val_accuracy
    })

    improved = val_macro_f1 > best_f1

    if improved:
        best_f1 = val_macro_f1
        best_epoch = epoch
        patience_counter = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": copy.deepcopy(
                gated_model.state_dict()
            ),
            "val_macro_f1": best_f1,
            "val_accuracy": val_accuracy,
            "num_classes": num_classes
        }, GATED_DIR / "gated_context_classifier_best.pt")
    else:
        patience_counter += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"Macro-F1={val_macro_f1:.4f} | "
        f"Accuracy={val_accuracy:.4f}"
        + (" | BEST" if improved else "")
    )

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

# Save history
with open(GATED_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\nTraining complete.")
print("Best epoch:", best_epoch)
print(f"Best validation Macro-F1: {best_f1:.4f}")
print(f"Previous best Macro-F1:   {0.3682:.4f}")
print("Checkpoint:", GATED_DIR / "gated_context_classifier_best.pt")
print("History:", GATED_DIR / "training_history.json")

In [ ]:
import torch
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)

GATED_CHECKPOINT = (
    "/kaggle/working/Phase3_Submissions/"
    "06_arch4_gated_context/"
    "gated_context_classifier_best.pt"
)

# Load the best checkpoint
checkpoint_data = torch.load(
    GATED_CHECKPOINT,
    map_location=device,
    weights_only=False
)

gated_model.load_state_dict(
    checkpoint_data["model_state_dict"]
)
gated_model.eval()

# Use the existing contextual validation embeddings
X_dev_eval = X_dev.to(device)
y_dev_eval = y_dev.cpu().numpy()

with torch.no_grad():
    logits = []
    for start in range(0, len(X_dev_eval), 256):
        xb = X_dev_eval[start:start + 256]
        logits.append(gated_model(xb).cpu())

    logits = torch.cat(logits, dim=0)
    predictions = logits.argmax(dim=-1).numpy()

id_to_label = {
    int(index): label
    for label, index in label_vocab.items()
}

target_names = [
    id_to_label[i]
    for i in range(len(id_to_label))
]

print("Checkpoint epoch:", checkpoint_data["epoch"])
print("Checkpoint Macro-F1:", checkpoint_data["val_macro_f1"])
print("Recomputed Macro-F1:", f1_score(
    y_dev_eval,
    predictions,
    average="macro",
    zero_division=0
))
print("Accuracy:", accuracy_score(y_dev_eval, predictions))

print("\nPER-CLASS REPORT\n")
print(classification_report(
    y_dev_eval,
    predictions,
    labels=list(range(len(target_names))),
    target_names=target_names,
    digits=4,
    zero_division=0
))

print("\nCONFUSION MATRIX\n")
print(confusion_matrix(
    y_dev_eval,
    predictions,
    labels=list(range(len(target_names)))
))

In [ ]:
import numpy as np
import torch
from sklearn.metrics import f1_score, accuracy_score

# Reuse logits and predictions from the previous evaluation.
# logits has shape [2142, 13].
# y_dev_eval contains the validation labels.

logits_np = logits.cpu().numpy()
y_true = y_dev_eval

base_f1 = f1_score(
    y_true,
    logits_np.argmax(axis=1),
    average="macro",
    zero_division=0
)

print(f"Baseline Macro-F1: {base_f1:.4f}")

# Start with no class-specific adjustments.
best_bias = np.zeros(len(label_vocab), dtype=np.float32)
best_f1 = base_f1

# Coordinate search over class-specific logit biases.
# The original model checkpoint remains unchanged.
step_sizes = [0.10, 0.05, 0.02]
max_passes = 3

for step in step_sizes:
    improved_in_pass = True
    passes = 0

    while improved_in_pass and passes < max_passes:
        improved_in_pass = False
        passes += 1

        for class_id in range(len(label_vocab)):
            current_best_bias = best_bias[class_id]
            current_best_f1 = best_f1

            for delta in [-step, step]:
                candidate_bias = best_bias.copy()
                candidate_bias[class_id] += delta

                candidate_preds = (
                    logits_np + candidate_bias[None, :]
                ).argmax(axis=1)

                score = f1_score(
                    y_true,
                    candidate_preds,
                    average="macro",
                    zero_division=0
                )

                if score > current_best_f1 + 1e-6:
                    current_best_f1 = score
                    current_best_bias = candidate_bias[class_id]
                    improved_in_pass = True

            if current_best_f1 > best_f1 + 1e-6:
                best_bias[class_id] = current_best_bias
                best_f1 = current_best_f1

    print(f"Step {step:.2f}: Macro-F1 = {best_f1:.4f}")

optimized_preds = (
    logits_np + best_bias[None, :]
).argmax(axis=1)

id_to_label = {
    int(index): label
    for label, index in label_vocab.items()
}

print("\nBaseline Macro-F1:", round(base_f1, 4))
print("Bias-adjusted Macro-F1:", round(best_f1, 4))
print("Baseline accuracy:", round(
    accuracy_score(y_true, logits_np.argmax(axis=1)), 4
))
print("Adjusted accuracy:", round(
    accuracy_score(y_true, optimized_preds), 4
))

print("\nLearned biases:")
for class_id, bias in enumerate(best_bias):
    print(f"{id_to_label[class_id]:28s}: {bias:+.3f}")

In [ ]:
import json
import torch
from pathlib import Path

GATED_DIR = Path(
    "/kaggle/working/Phase3_Submissions/06_arch4_gated_context"
)
GATED_DIR.mkdir(parents=True, exist_ok=True)

# Save the best neural classifier.
torch.save(
    {
        "model_state_dict": gated_model.state_dict(),
        "input_dim": 3072,
        "num_classes": len(label_vocab),
        "best_epoch": int(checkpoint_data["epoch"]),
        "base_validation_macro_f1": float(base_f1),
        "bias_adjusted_validation_macro_f1": float(best_f1),
        "validation_accuracy_after_bias": float(
            accuracy_score(y_true, optimized_preds)
        ),
    },
    GATED_DIR / "gated_context_classifier_best.pt"
)

# Save prediction biases and label mapping.
bias_config = {
    "class_biases": {
        id_to_label[i]: float(best_bias[i])
        for i in range(len(best_bias))
    },
    "label_to_id": {
        label: int(index)
        for label, index in label_vocab.items()
    },
    "base_validation_macro_f1": float(base_f1),
    "bias_adjusted_validation_macro_f1": float(best_f1),
    "validation_accuracy_after_bias": float(
        accuracy_score(y_true, optimized_preds)
    ),
}

with open(GATED_DIR / "prediction_config.json", "w") as f:
    json.dump(bias_config, f, indent=2)

print("Saved classifier and prediction configuration.")
print("Checkpoint:", GATED_DIR / "gated_context_classifier_best.pt")
print("Bias config:", GATED_DIR / "prediction_config.json")
print(f"Bias-adjusted validation Macro-F1: {best_f1:.4f}")

In [ ]:
import json
from pathlib import Path

BASE_DIR = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")

for filename in ["CL_test.json", "IT_test.json"]:
    path = BASE_DIR / "Test" / filename

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"\n{'=' * 50}")
    print("File:", filename)
    print("Python type:", type(data).__name__)

    if isinstance(data, dict):
        print("Number of documents:", len(data))
        for doc_id, sentences in list(data.items())[:2]:
            print("Document ID:", doc_id)
            print("Value type:", type(sentences).__name__)
            print("First 3 sentences:", sentences[:3])
    elif isinstance(data, list):
        print("Number of documents:", len(data))
        for doc in data[:2]:
            print(json.dumps(doc, ensure_ascii=False)[:1200])
    else:
        print("Unexpected structure:", str(data)[:1000])

In [ ]:
from pathlib import Path

SEARCH_DIRS = [
    Path("/kaggle/working/Phase_3_RR"),
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

keywords = (
    "submission",
    "readme",
    "instruction",
    "sample",
    "baseline",
    "evaluation",
)

seen = set()

for root in SEARCH_DIRS:
    if not root.exists():
        continue

    print(f"\n--- Searching: {root} ---")

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if path in seen:
            continue
        seen.add(path)

        name = path.name.lower()

        if any(keyword in name for keyword in keywords):
            print(path)

In [ ]:
from pathlib import Path

roots = [
    Path("/kaggle/working/Phase_3_RR"),
    Path("/kaggle/input"),
]

for root in roots:
    print(f"\n{'=' * 60}")
    print("ROOT:", root)

    if not root.exists():
        print("Directory does not exist.")
        continue

    for path in sorted(root.iterdir()):
        print(
            "[DIR] " if path.is_dir() else "[FILE]",
            path.name
        )

# Inspect the competition dataset folder recursively,
# but limit output to the first 100 entries.
dataset_root = Path("/kaggle/working/Phase_3_RR")

if dataset_root.exists():
    print("\n--- Dataset file inventory ---")

    files = sorted(
        p for p in dataset_root.rglob("*")
        if p.is_file()
    )

    print("Total files:", len(files))

    for path in files[:100]:
        print(path)

In [ ]:
from pathlib import Path

print("Kaggle working directory exists:",
      Path("/kaggle/working").exists())

print("Phase 3 dataset exists:",
      Path("/kaggle/working/Phase_3_RR/Phase_3_RR").exists())

print("Saved classifier exists:",
      Path(
          "/kaggle/working/Phase3_Submissions/"
          "06_arch4_gated_context/"
          "gated_context_classifier_best.pt"
      ).exists())

print("Saved bias configuration exists:",
      Path(
          "/kaggle/working/Phase3_Submissions/"
          "06_arch4_gated_context/"
          "prediction_config.json"
      ).exists())

In [ ]:
from pathlib import Path

base = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")
gated = Path("/kaggle/working/Phase3_Submissions/06_arch4_gated_context")

print("=== DATA FILES ===")
for name in [
    "Train/label_vocab.json",
    "Test/CL_test.json",
    "Test/IT_test.json",
]:
    p = base / name
    print(f"{name}: {p.exists()}")

print("\n=== SAVED CLASSIFIER ===")
print("Classifier:", (gated / "gated_context_classifier_best.pt").exists())
print("Bias config:", (gated / "prediction_config.json").exists())

print("\n=== NOTEBOOK VARIABLES ===")
for name in [
    "model",
    "tokenizer",
    "label_vocab",
    "CL_test",
    "IT_test",
    "GatedContextClassifier",
]:
    print(f"{name}: {name in globals()}")

print("\n=== GPU ===")
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# PHASE 3 — ARCH4 + GATED CONTEXT PREDICTION PIPELINE
# Corrected checkpoint dimensions and separate context inputs
# ============================================================

import json
import csv
import zipfile
import inspect
import torch
import numpy as np

from pathlib import Path
from collections import OrderedDict

# ============================================================
# 1. PATHS
# ============================================================

DATA_ROOT = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")

GATED_DIR = Path(
    "/kaggle/working/Phase3_Submissions/06_arch4_gated_context"
)

OUT_DIR = Path(
    "/kaggle/working/Phase3_Submissions/07_final_predictions"
)

CLASSIFIER_PATH = GATED_DIR / "gated_context_classifier_best.pt"
BIAS_PATH = GATED_DIR / "prediction_config.json"

OUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [CLASSIFIER_PATH, BIAS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required file missing: {path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Output directory:", OUT_DIR)

# ============================================================
# 2. LOAD LABEL VOCABULARY AND TEST DATA
# ============================================================

with open(DATA_ROOT / "Train/label_vocab.json", "r", encoding="utf-8") as f:
    label_vocab = json.load(f)

label_vocab = {str(k): int(v) for k, v in label_vocab.items()}
id_to_label = {v: k for k, v in label_vocab.items()}

if sorted(id_to_label) != list(range(len(label_vocab))):
    raise ValueError("Label IDs are not contiguous from 0.")

with open(DATA_ROOT / "Test/CL_test.json", "r", encoding="utf-8") as f:
    CL_test = json.load(f)

with open(DATA_ROOT / "Test/IT_test.json", "r", encoding="utf-8") as f:
    IT_test = json.load(f)

print("CL documents:", len(CL_test))
print("IT documents:", len(IT_test))
print("Number of labels:", len(label_vocab))

# ============================================================
# 3. LOAD SAVED CLASSIFIER CHECKPOINT
# ============================================================

checkpoint = torch.load(
    CLASSIFIER_PATH,
    map_location=device,
    weights_only=False
)

if "model_state_dict" not in checkpoint:
    raise KeyError(
        "Classifier checkpoint has no 'model_state_dict'. "
        f"Available keys: {list(checkpoint.keys())}"
    )

state_dict = checkpoint["model_state_dict"]

print("\nCheckpoint tensors:")
for key, value in state_dict.items():
    if key.endswith("weight") and hasattr(value, "shape"):
        print(f"  {key}: {tuple(value.shape)}")

# The checkpoint shows each context projection accepts 1024 features.
INPUT_DIM = state_dict["prev_proj.weight"].shape[1]

if state_dict["curr_proj.weight"].shape[1] != INPUT_DIM:
    raise ValueError("Current-sentence projection dimension mismatch.")

if state_dict["next_proj.weight"].shape[1] != INPUT_DIM:
    raise ValueError("Next-sentence projection dimension mismatch.")

NUM_CLASSES = len(label_vocab)

# Initialize the original classifier architecture.
gated_model = GatedContextClassifier(
    input_dim=INPUT_DIM,
    num_classes=NUM_CLASSES
).to(device)

gated_model.load_state_dict(state_dict, strict=True)
gated_model.eval()

print("\nClassifier loaded successfully.")
print("Per-sentence embedding dimension:", INPUT_DIM)
print("Combined context dimension:", INPUT_DIM * 3)
print("Classes:", NUM_CLASSES)
print("Best epoch:", checkpoint.get("best_epoch", "not recorded"))

# ============================================================
# 4. LOAD CLASS BIASES
# ============================================================

with open(BIAS_PATH, "r", encoding="utf-8") as f:
    bias_config = json.load(f)

print("\nBias configuration keys:", list(bias_config.keys()))

bias_values = bias_config.get(
    "class_biases",
    bias_config.get(
        "biases",
        bias_config.get("class_bias", None)
    )
)

if bias_values is None:
    # Some configurations store the vector under another key.
    # Do not silently assume an incorrect mapping.
    possible = {
        key: value
        for key, value in bias_config.items()
        if isinstance(value, dict)
        and all(label in value for label in label_vocab)
    }

    if len(possible) == 1:
        bias_values = next(iter(possible.values()))
    else:
        raise ValueError(
            "Could not identify class biases automatically. "
            f"Inspect this config: {BIAS_PATH}\n"
            f"Keys: {list(bias_config.keys())}"
        )

bias_vector = np.zeros(NUM_CLASSES, dtype=np.float32)

if isinstance(bias_values, dict):
    for label, value in bias_values.items():
        if label in label_vocab:
            bias_vector[label_vocab[label]] = float(value)

elif isinstance(bias_values, list):
    if len(bias_values) != NUM_CLASSES:
        raise ValueError(
            f"Bias vector has {len(bias_values)} values; "
            f"expected {NUM_CLASSES}."
        )
    bias_vector = np.asarray(bias_values, dtype=np.float32)

else:
    raise TypeError(
        f"Unsupported bias configuration type: {type(bias_values)}"
    )

print("Bias vector loaded.")

# ============================================================
# 5. CHECK MODEL AND TOKENIZER
# ============================================================

if "model" not in globals():
    raise RuntimeError(
        "Arch4 model variable is missing. Reload the Arch4 checkpoint first."
    )

if "tokenizer" not in globals():
    raise RuntimeError(
        "Tokenizer variable is missing. Reload tokenizer.json first."
    )

model = model.to(device)
model.eval()

BOS_ID = 1
EOS_ID = 2
PAD_ID = 0
MAX_SENTENCE_TOKENS = 254
EMBED_BATCH_SIZE = 8

# ============================================================
# 6. SENTENCE ENCODING AND EMBEDDING
# ============================================================

def encode_sentence(text):
    token_ids = tokenizer.encode(
        str(text),
        add_special_tokens=False
    ).ids[:MAX_SENTENCE_TOKENS]

    return [BOS_ID] + token_ids + [EOS_ID]


@torch.no_grad()
def embed_sentences(sentences, batch_size=EMBED_BATCH_SIZE):
    all_embeddings = []

    for start in range(0, len(sentences), batch_size):
        batch_sentences = sentences[start:start + batch_size]

        encoded = [encode_sentence(s) for s in batch_sentences]
        max_len = max(len(ids) for ids in encoded)

        input_ids = torch.full(
            (len(encoded), max_len),
            PAD_ID,
            dtype=torch.long,
            device=device
        )

        eos_positions = []

        for i, ids in enumerate(encoded):
            input_ids[i, :len(ids)] = torch.tensor(
                ids,
                dtype=torch.long,
                device=device
            )
            eos_positions.append(len(ids) - 1)

        output = model(input_ids, return_hidden=True)

        if isinstance(output, dict):
            hidden = output.get(
                "hidden_states",
                output.get("last_hidden_state")
            )
        elif isinstance(output, (tuple, list)):
            hidden = output[0]
        else:
            hidden = output

        if hidden is None or hidden.ndim != 3:
            raise RuntimeError(
                "Unexpected Arch4 output. "
                f"Output type: {type(output)}, "
                f"hidden shape: {getattr(hidden, 'shape', None)}"
            )

        rows = torch.arange(len(encoded), device=device)
        positions = torch.tensor(eos_positions, device=device)

        embeddings = hidden[rows, positions].float().cpu().numpy()
        all_embeddings.append(embeddings)

    if not all_embeddings:
        return np.empty((0, INPUT_DIM), dtype=np.float32)

    result = np.concatenate(all_embeddings, axis=0)

    if result.shape[1] != INPUT_DIM:
        raise RuntimeError(
            f"Arch4 produced embedding dimension {result.shape[1]}, "
            f"but the classifier expects {INPUT_DIM}."
        )

    return result


# ============================================================
# 7. BUILD CONTEXT EMBEDDINGS
# ============================================================

def build_context_features(test_documents):
    document_ids = list(test_documents.keys())

    flat_sentences = []
    doc_ranges = {}
    records = []

    for doc_id in document_ids:
        sentences = test_documents[doc_id]

        if not isinstance(sentences, list):
            raise TypeError(
                f"Document {doc_id}: expected a list of sentences."
            )

        start = len(flat_sentences)

        for sentence_index, sentence in enumerate(sentences):
            flat_sentences.append(str(sentence))

            records.append({
                "document_id": str(doc_id),
                "sentence_index": sentence_index,
                "sentence": str(sentence)
            })

        end = len(flat_sentences)
        doc_ranges[doc_id] = (start, end)

    embeddings = embed_sentences(flat_sentences)

    if len(embeddings) != len(flat_sentences):
        raise RuntimeError("Sentence embedding count mismatch.")

    # Keep context as three separate arrays:
    # previous: (N, 1024)
    # current:  (N, 1024)
    # next:     (N, 1024)
    previous = np.zeros_like(embeddings)
    current = embeddings.copy()
    following = np.zeros_like(embeddings)

    for doc_id, (start, end) in doc_ranges.items():
        for idx in range(start, end):
            previous[idx] = embeddings[max(start, idx - 1)]
            following[idx] = embeddings[min(end - 1, idx + 1)]

    return document_ids, records, doc_ranges, previous, current, following


# ============================================================
# 8. INSPECT CLASSIFIER FORWARD INTERFACE
# ============================================================

print("\nClassifier forward signature:")
print(inspect.signature(gated_model.forward))

print("\nClassifier module:")
print(gated_model)

# ============================================================
# 9. RUN A SMALL INTERFACE TEST BEFORE FULL INFERENCE
# ============================================================

with torch.no_grad():
    dummy = torch.zeros((2, INPUT_DIM), device=device)

    try:
        test_logits = gated_model(dummy, dummy, dummy)
        FORWARD_MODE = "three_arguments"
    except TypeError:
        try:
            test_logits = gated_model(
                torch.cat([dummy, dummy, dummy], dim=-1)
            )
            FORWARD_MODE = "concatenated"
        except Exception as exc:
            raise RuntimeError(
                "Could not determine the classifier's forward interface. "
                "Inspect the printed classifier architecture."
            ) from exc

    if test_logits.ndim != 2 or test_logits.shape[-1] != NUM_CLASSES:
        raise RuntimeError(
            f"Unexpected classifier output shape: {tuple(test_logits.shape)}"
        )

print("Classifier interface:", FORWARD_MODE)
print("Dummy output shape:", tuple(test_logits.shape))

# ============================================================
# 10. PREDICTION FUNCTION
# ============================================================

@torch.no_grad()
def predict_context(previous, current, following, batch_size=256):
    predictions = []

    for start in range(0, len(current), batch_size):
        end = min(start + batch_size, len(current))

        prev_batch = torch.from_numpy(previous[start:end]).float().to(device)
        curr_batch = torch.from_numpy(current[start:end]).float().to(device)
        next_batch = torch.from_numpy(following[start:end]).float().to(device)

        if FORWARD_MODE == "three_arguments":
            logits = gated_model(
                prev_batch,
                curr_batch,
                next_batch
            )
        else:
            combined = torch.cat(
                [prev_batch, curr_batch, next_batch],
                dim=-1
            )
            logits = gated_model(combined)

        logits = logits + torch.tensor(
            bias_vector,
            dtype=logits.dtype,
            device=device
        ).unsqueeze(0)

        predictions.extend(logits.argmax(dim=-1).cpu().tolist())

    return predictions


# ============================================================
# 11. PREDICT CL AND IT
# ============================================================

all_predictions = {}
all_records = []

for corpus_name, documents in [
    ("CL", CL_test),
    ("IT", IT_test)
]:
    print(f"\nProcessing {corpus_name}...")

    doc_ids, records, ranges, prev, curr, nxt = build_context_features(
        documents
    )

    pred_ids = predict_context(prev, curr, nxt)

    if len(pred_ids) != len(records):
        raise RuntimeError(
            f"{corpus_name}: prediction count mismatch."
        )

    corpus_predictions = OrderedDict()

    for doc_id in doc_ids:
        start, end = ranges[doc_id]

        corpus_predictions[str(doc_id)] = [
            id_to_label[pred_ids[i]]
            for i in range(start, end)
        ]

    all_predictions[corpus_name] = corpus_predictions

    for record, pred_id in zip(records, pred_ids):
        all_records.append({
            "corpus": corpus_name,
            "document_id": record["document_id"],
            "sentence_index": record["sentence_index"],
            "sentence": record["sentence"],
            "predicted_label": id_to_label[pred_id],
            "predicted_label_id": pred_id
        })

    print(
        f"{corpus_name}: {len(doc_ids)} documents, "
        f"{len(records)} sentences predicted."
    )

# ============================================================
# 12. SAVE OUTPUT FILES
# ============================================================

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

cl_path = OUT_DIR / "CL_predictions.json"
it_path = OUT_DIR / "IT_predictions.json"
combined_path = OUT_DIR / "predictions_combined.json"
csv_path = OUT_DIR / "predictions.csv"
metadata_path = OUT_DIR / "metadata.json"

save_json(cl_path, all_predictions["CL"])
save_json(it_path, all_predictions["IT"])
save_json(combined_path, all_predictions)

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "corpus",
            "document_id",
            "sentence_index",
            "sentence",
            "predicted_label",
            "predicted_label_id"
        ]
    )
    writer.writeheader()
    writer.writerows(all_records)

metadata = {
    "model": "Arch4 + Gated Context Classifier",
    "classifier_checkpoint": str(CLASSIFIER_PATH),
    "bias_config": str(BIAS_PATH),
    "corpora": {
        corpus: {
            "documents": len(all_predictions[corpus]),
            "sentences": sum(
                len(sentences)
                for sentences in all_predictions[corpus].values()
            )
        }
        for corpus in ["CL", "IT"]
    },
    "total_documents": sum(
        len(all_predictions[c]) for c in ["CL", "IT"]
    ),
    "total_sentences": len(all_records),
    "label_vocab": label_vocab,
    "official_submission_schema_verified": False
}

save_json(metadata_path, metadata)

zip_path = OUT_DIR / "Phase3_predictions_bundle.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in [
        cl_path,
        it_path,
        combined_path,
        csv_path,
        metadata_path
    ]:
        zf.write(path, arcname=path.name)

# ============================================================
# 13. VALIDATE AND REPORT
# ============================================================

print("\n" + "=" * 60)
print("PREDICTION GENERATION COMPLETE")
print("=" * 60)

print("CL documents:", len(all_predictions["CL"]))
print("IT documents:", len(all_predictions["IT"]))
print("Total documents:", metadata["total_documents"])
print("Total sentence predictions:", metadata["total_sentences"])

for path in [
    cl_path,
    it_path,
    combined_path,
    csv_path,
    metadata_path,
    zip_path
]:
    print(f"{path.name}: {path.stat().st_size:,} bytes")

assert metadata["total_documents"] == 10
assert metadata["total_sentences"] == 2197

print("\nAll count validations passed.")
print("Bundle:", zip_path)

In [ ]:
import torch

checkpoint_path = (
    "/kaggle/working/Phase3_Submissions/"
    "06_arch4_gated_context/gated_context_classifier_best.pt"
)

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("=== CHECKPOINT KEYS ===")
print(list(ckpt.keys()))

print("\n=== CLASSIFIER MODULES ===")
for name, module in gated_model.named_modules():
    if name:
        print(name, ":", module)

print("\n=== CHECKPOINT TENSOR SHAPES ===")
for name, tensor in ckpt["model_state_dict"].items():
    print(name, tuple(tensor.shape))

print("\n=== FORWARD BYTECODE ===")
import dis
dis.dis(GatedContextClassifier.forward)

In [ ]:
import os
import json
import zipfile
import shutil

# Architecture-specific output folder
BASE_DIR = "/kaggle/working/Phase3_Submissions"
ARCH = "Arch4"
ARCH_DIR = os.path.join(BASE_DIR, ARCH)

os.makedirs(ARCH_DIR, exist_ok=True)

# Predictions generated by the previous pipeline
SOURCE_JSON = os.path.join(
    BASE_DIR, "07_final_predictions", "predictions_combined.json"
)

PREDICTION_JSON = os.path.join(ARCH_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(ARCH_DIR, "submission.zip")

# Check source file
if not os.path.isfile(SOURCE_JSON):
    raise FileNotFoundError(f"Predictions not found: {SOURCE_JSON}")

# Copy combined predictions into the architecture folder
shutil.copy2(SOURCE_JSON, PREDICTION_JSON)

# Create Codabench-compatible ZIP:
# prediction.json must be at the ZIP root.
with zipfile.ZipFile(
    SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    zf.write(PREDICTION_JSON, arcname="prediction.json")

# Validate the package
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    assert zf.namelist() == ["prediction.json"], zf.namelist()

    with zf.open("prediction.json") as f:
        data = json.load(f)

print("Architecture:", ARCH)
print("Folder:", ARCH_DIR)
print("Files:", os.listdir(ARCH_DIR))
print("ZIP contents:", zipfile.ZipFile(SUBMISSION_ZIP).namelist())
print("Submission ZIP:", SUBMISSION_ZIP)

In [ ]:
import json
import os

BASE = "/kaggle/working/Phase_3_RR/Phase_3_RR"
OUT = "/kaggle/working/Phase3_Submissions/07_final_predictions"

files = {
    "CL test": os.path.join(BASE, "Test", "CL_test.json"),
    "IT test": os.path.join(BASE, "Test", "IT_test.json"),
    "Combined predictions": os.path.join(OUT, "predictions_combined.json"),
}

for name, path in files.items():
    print(f"\n{'=' * 65}\n{name}\nPath: {path}")

    if not os.path.exists(path):
        print("FILE NOT FOUND")
        continue

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Top-level type:", type(data).__name__)

    if isinstance(data, dict):
        print("Top-level keys:", list(data.keys())[:15])

        for key, value in list(data.items())[:3]:
            print(f"\n  Key: {key!r}")
            print("  Value type:", type(value).__name__)

            if isinstance(value, list):
                print("  List length:", len(value))
                print("  First 3 items:", repr(value[:3]))

            elif isinstance(value, dict):
                print("  Nested keys:", list(value.keys())[:10])

    elif isinstance(data, list):
        print("Number of items:", len(data))
        print("First item:", repr(data[0])[:1200] if data else "EMPTY")

In [ ]:
import os
import json
import zipfile

BASE = "/kaggle/working/Phase_3_RR/Phase_3_RR"
SOURCE = "/kaggle/working/Phase3_Submissions/07_final_predictions/predictions_combined.json"
ARCH_DIR = "/kaggle/working/Phase3_Submissions/Arch4"

os.makedirs(ARCH_DIR, exist_ok=True)

# Load official test inputs and existing predictions
with open(os.path.join(BASE, "Test", "CL_test.json"), encoding="utf-8") as f:
    cl_test = json.load(f)

with open(os.path.join(BASE, "Test", "IT_test.json"), encoding="utf-8") as f:
    it_test = json.load(f)

with open(SOURCE, encoding="utf-8") as f:
    combined = json.load(f)

# Flatten the phase grouping: document_id -> list of labels
predictions = {}
predictions.update(combined.get("CL", {}))
predictions.update(combined.get("IT", {}))

expected = {**cl_test, **it_test}

print("Expected documents:", len(expected))
print("Predicted documents:", len(predictions))

# Validate document IDs and sentence counts
missing = sorted(set(expected) - set(predictions))
extra = sorted(set(predictions) - set(expected))
bad = []

for doc_id, sentences in expected.items():
    if doc_id not in predictions:
        continue

    labels = predictions[doc_id]

    if not isinstance(labels, list):
        bad.append((doc_id, "prediction is not a list", type(labels).__name__))
    elif len(labels) != len(sentences):
        bad.append((doc_id, "sentence-count mismatch", len(sentences), len(labels)))
    elif not all(isinstance(label, (str, int)) for label in labels):
        bad.append((doc_id, "invalid label type"))

print("Missing documents:", missing)
print("Unexpected documents:", extra)
print("Sentence-count/type errors:", bad[:10])

if missing or extra or bad:
    raise ValueError(
        "Validation failed. Do not submit yet; inspect the reported documents."
    )

# Save prediction.json directly inside the architecture folder
prediction_path = os.path.join(ARCH_DIR, "prediction.json")
zip_path = os.path.join(ARCH_DIR, "submission.zip")

with open(prediction_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False)

# Codabench requires prediction.json at the ZIP root
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(prediction_path, arcname="prediction.json")

with zipfile.ZipFile(zip_path, "r") as zf:
    assert zf.namelist() == ["prediction.json"]
    with zf.open("prediction.json") as f:
        packaged = json.load(f)

assert len(packaged) == 10
print("\nSUCCESS")
print("Documents:", len(packaged))
print("Total expected sentences:", sum(len(x) for x in expected.values()))
print("Total predicted labels:", sum(len(x) for x in packaged.values()))
print("Submission ZIP:", zip_path)

In [ ]:
import json
import zipfile
from pathlib import Path

BASE = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")
OUT = Path("/kaggle/working/Phase3_Submissions/Arch4")
OUT.mkdir(parents=True, exist_ok=True)

# Load official test documents
with open(BASE / "Test" / "CL_test.json", encoding="utf-8") as f:
    cl_test = json.load(f)

with open(BASE / "Test" / "IT_test.json", encoding="utf-8") as f:
    it_test = json.load(f)

# Load existing predictions
pred_path = Path(
    "/kaggle/working/Phase3_Submissions/"
    "07_final_predictions/predictions_combined.json"
)

with open(pred_path, encoding="utf-8") as f:
    combined = json.load(f)

# Load official label vocabulary
with open(BASE / "Train" / "label_vocab.json", encoding="utf-8") as f:
    label_vocab = json.load(f)

# Flatten CL and IT predictions
predictions = {}
predictions.update(combined.get("CL", {}))
predictions.update(combined.get("IT", {}))

# Merge expected test documents
expected = {}
expected.update(cl_test)
expected.update(it_test)

print("Expected documents:", len(expected))
print("Predicted documents:", len(predictions))

# Validate document IDs
missing = sorted(set(expected) - set(predictions))
extra = sorted(set(predictions) - set(expected))

# Validate sentence counts and label values
bad_counts = []
bad_types = []
bad_labels = []

official_labels = set(label_vocab.keys())

for doc_id, doc in expected.items():
    pred = predictions.get(doc_id)

    # Test documents contain ordered sentence lists
    if isinstance(doc, dict):
        sentences = doc.get("text")
    else:
        sentences = doc

    if not isinstance(sentences, list):
        bad_types.append(
            (doc_id, "test document does not contain a sentence list")
        )
        continue

    if not isinstance(pred, list):
        bad_types.append(
            (doc_id, f"prediction type is {type(pred).__name__}, expected list")
        )
        continue

    if len(pred) != len(sentences):
        bad_counts.append(
            (doc_id, len(sentences), len(pred))
        )

    invalid = sorted({
        label for label in pred
        if not isinstance(label, str) or label not in official_labels
    })

    if invalid:
        bad_labels.append((doc_id, invalid[:5]))

print("\nMissing documents:", missing[:10])
print("Extra documents:", extra[:10])
print("Sentence-count mismatches:", bad_counts[:10])
print("Type errors:", bad_types[:10])
print("Invalid labels:", bad_labels[:10])

if missing or extra or bad_counts or bad_types or bad_labels:
    raise ValueError(
        "Validation failed. Do not submit yet; inspect the errors above."
    )

# Save prediction.json
prediction_path = OUT / "prediction.json"

with open(prediction_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

# ZIP must contain prediction.json at its root
zip_path = OUT / "submission.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(prediction_path, arcname="prediction.json")

# Verify ZIP contents
with zipfile.ZipFile(zip_path, "r") as zf:
    print("\nZIP contents:", zf.namelist())
    assert zf.namelist() == ["prediction.json"]

print("\nSUCCESS")
print("Prediction JSON:", prediction_path)
print("Submission ZIP:", zip_path)
print("Total predicted sentences:", sum(len(v) for v in predictions.values()))

Experiment 1: fine-tune your existing Arch4 classifier with a lower learning rate (1e-4).

In [ ]:
# EXPERIMENT 1: Arch4 classifier fine-tuning
# Lower LR + early stopping using validation Macro-F1

import os
import glob
import json
import torch
import numpy as np

from pathlib import Path
from sklearn.metrics import f1_score
from torch.utils.data import TensorDataset, DataLoader

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------
BASE_DIR = Path("/kaggle/working/Phase3_Submissions")
SOURCE_DIR = BASE_DIR / "06_arch4_gated_context"
OUTPUT_DIR = BASE_DIR / "07_arch4_finetune_lr1e4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT = SOURCE_DIR / "gated_context_classifier_best.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LR = 1e-4
BATCH_SIZE = 128
MAX_EPOCHS = 60
PATIENCE = 10
NUM_LABELS = 13

torch.manual_seed(42)
np.random.seed(42)

print("Device:", DEVICE)
print("Source checkpoint exists:", CHECKPOINT.exists())

if not CHECKPOINT.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT}")

# --------------------------------------------------
# LOCATE EXISTING TRAINING ARRAYS
# --------------------------------------------------
npz_files = list(SOURCE_DIR.glob("*.npz"))

if not npz_files:
    npz_files = list(BASE_DIR.rglob("*.npz"))

print("\nAvailable NPZ files:")
for p in npz_files:
    print(" ", p)

if not npz_files:
    raise FileNotFoundError(
        "No .npz files found. This cell needs your existing "
        "training/validation embeddings and labels."
    )

# Inspect array names rather than guessing their meaning
for p in npz_files:
    with np.load(p, allow_pickle=False) as data:
        print(f"\n{p.name}")
        for key in data.files:
            arr = data[key]
            print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")

print(
    "\nDiagnostic complete. Check the printed array names and shapes "
    "before training; the existing classifier's exact forward method "
    "and checkpoint format must be respected."
)

In [ ]:
# EXPERIMENT 1 — Arch4 classifier fine-tuning
# First inspect checkpoint structure safely before changing model weights.

import torch
from pathlib import Path

checkpoint_path = Path(
    "/kaggle/working/Phase3_Submissions/"
    "06_arch4_gated_context/gated_context_classifier_best.pt"
)

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(ckpt).__name__)

if isinstance(ckpt, dict):
    print("\nTop-level keys:")
    for key, value in ckpt.items():
        if isinstance(value, dict):
            print(f"  {key}: dict with {len(value)} keys")
            print("    Sample keys:", list(value.keys())[:12])
        elif isinstance(value, torch.Tensor):
            print(f"  {key}: Tensor {tuple(value.shape)}")
        else:
            print(f"  {key}: {str(value)[:200]}")

    # Find likely model state dictionary
    for key, value in ckpt.items():
        if isinstance(value, dict) and value:
            tensor_items = {
                k: v for k, v in value.items()
                if isinstance(v, torch.Tensor)
            }
            if tensor_items:
                print(f"\nPossible state dict: '{key}'")
                for name, tensor in list(tensor_items.items())[:20]:
                    print(f"  {name}: {tuple(tensor.shape)}")
else:
    print("Checkpoint is not a dictionary.")

print("\nClassifier source files:")
for p in Path("/kaggle/working/Phase3_Submissions").rglob("*.py"):
    if "arch4" in str(p).lower() or "classifier" in p.name.lower():
        print(p)

In [ ]:
# EXPERIMENT 1: Fine-tune Arch4 Gated Context Classifier
# Uses existing 3072-dim context embeddings and checkpoint.
# Saves the best model separately; original checkpoint remains untouched.

import copy
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score

# ---------------- CONFIG ----------------
SEED = 42
LR = 1e-4
BATCH_SIZE = 128
MAX_EPOCHS = 30
PATIENCE = 7
WEIGHT_DECAY = 1e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

root = Path("/kaggle/working/Phase3_Submissions")
data_path = root / "05_arch4_context" / "arch4_context_embeddings.npz"
checkpoint_path = (
    root / "06_arch4_gated_context" / "gated_context_classifier_best.pt"
)
out_dir = root / "07_arch4_finetune_lr1e4"
out_dir.mkdir(parents=True, exist_ok=True)

assert data_path.exists(), f"Missing data: {data_path}"
assert checkpoint_path.exists(), f"Missing checkpoint: {checkpoint_path}"

# ---------------- DATA ----------------
with np.load(data_path) as data:
    X_train = torch.tensor(data["X_train"], dtype=torch.float32)
    y_train = torch.tensor(data["y_train"], dtype=torch.long)
    X_dev = torch.tensor(data["X_dev"], dtype=torch.float32)
    y_dev = torch.tensor(data["y_dev"], dtype=torch.long)

assert X_train.shape[1] == 3072
assert X_dev.shape[1] == 3072
assert len(X_train) == len(y_train)
assert len(X_dev) == len(y_dev)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=torch.cuda.is_available()
)
dev_loader = DataLoader(
    TensorDataset(X_dev, y_dev),
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)

# ---------------- MODEL ----------------
class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)

        self.context_gate = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)
        p = self.prev_proj(prev)
        c = self.curr_proj(curr)
        n = self.next_proj(nxt)

        gate = self.context_gate(torch.cat([p, c, n], dim=1))
        context = gate[:, :512] * p + gate[:, 512:] * n
        fused = c + context
        return self.fusion(fused)

# ---------------- LOAD CHECKPOINT ----------------
ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

assert ckpt.get("input_dim") == 3072, ckpt.get("input_dim")
assert ckpt.get("num_classes") == 13, ckpt.get("num_classes")

model = GatedContextClassifier(num_classes=13)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.to(device)

print("Device:", device)
print("Train/dev:", len(X_train), "/", len(X_dev))
print("Previous best epoch:", ckpt.get("best_epoch"))
print("Previous base validation Macro-F1:",
      ckpt.get("base_validation_macro_f1"))
print("Previous bias-adjusted validation Macro-F1:",
      ckpt.get("bias_adjusted_validation_macro_f1"))

# Preserve the original experiment's loss weighting if present.
# If not stored in the checkpoint, use ordinary cross-entropy.
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-6
)

# ---------------- TRAIN ----------------
best_f1 = -1.0
best_state = None
best_epoch = 0
bad_epochs = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_items = 0

    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(yb)
        total_items += len(yb)

    # Validation
    model.eval()
    targets, preds = [], []
    dev_loss, dev_items = 0.0, 0

    with torch.no_grad():
        for xb, yb in dev_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            dev_loss += loss.item() * len(yb)
            dev_items += len(yb)

            preds.extend(logits.argmax(dim=1).cpu().tolist())
            targets.extend(yb.cpu().tolist())

    macro_f1 = f1_score(
        targets, preds, average="macro", zero_division=0
    )
    train_loss = total_loss / max(total_items, 1)
    val_loss = dev_loss / max(dev_items, 1)

    scheduler.step(macro_f1)

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": val_loss,
        "validation_macro_f1": macro_f1,
        "learning_rate": optimizer.param_groups[0]["lr"]
    }
    history.append(record)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_macro_f1={macro_f1:.6f} | "
        f"lr={optimizer.param_groups[0]['lr']:.2e}"
    )

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        bad_epochs = 0

        torch.save({
            "model_state_dict": best_state,
            "input_dim": 3072,
            "num_classes": 13,
            "best_epoch": best_epoch,
            "validation_macro_f1": best_f1,
            "experiment": "arch4_classifier_finetune_lr1e4"
        }, out_dir / "classifier_best.pt")
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

# ---------------- SAVE RESULTS ----------------
with open(out_dir / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\nExperiment complete.")
print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", round(best_f1, 6))
print("Previous bias-adjusted validation Macro-F1:",
      ckpt.get("bias_adjusted_validation_macro_f1"))
print("Best checkpoint:", out_dir / "classifier_best.pt")
print("History:", out_dir / "training_history.json")

In [ ]:
# EXPERIMENT 2: Validation-only class-bias optimization
# Compares original and fine-tuned classifiers fairly.
# Does NOT use private test labels.

import json
import copy
import numpy as np
import torch
import torch.nn as nn

from pathlib import Path
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, TensorDataset

# ---------------- PATHS ----------------
ROOT = Path("/kaggle/working/Phase3_Submissions")

DATA_PATH = ROOT / "05_arch4_context" / "arch4_context_embeddings.npz"

CHECKPOINTS = {
    "original": ROOT / "06_arch4_gated_context" /
                "gated_context_classifier_best.pt",
    "finetuned": ROOT / "07_arch4_finetune_lr1e4" /
                 "classifier_best.pt"
}

OUT_DIR = ROOT / "08_arch4_bias_optimization"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
STEPS = 3000
LR = 0.03
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

# ---------------- MODEL ----------------
class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)

        self.context_gate = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)
        p = self.prev_proj(prev)
        c = self.curr_proj(curr)
        n = self.next_proj(nxt)

        gate = self.context_gate(torch.cat([p, c, n], dim=1))
        context = gate[:, :512] * p + gate[:, 512:] * n
        return self.fusion(c + context)

# ---------------- DATA ----------------
with np.load(DATA_PATH) as d:
    X_dev = torch.tensor(d["X_dev"], dtype=torch.float32)
    y_dev = torch.tensor(d["y_dev"], dtype=torch.long)

loader = DataLoader(
    TensorDataset(X_dev, y_dev),
    batch_size=256,
    shuffle=False
)

y_true = y_dev.numpy()
results = {}

def get_logits(checkpoint_path):
    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    model = GatedContextClassifier(13)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.to(DEVICE).eval()

    all_logits = []
    with torch.no_grad():
        for xb, _ in loader:
            logits = model(xb.to(DEVICE))
            all_logits.append(logits.cpu())

    return torch.cat(all_logits).numpy()

def optimize_bias(logits, y):
    # Optimize additive class biases directly for validation Macro-F1.
    # Coordinate search avoids differentiating through the discrete F1 metric.
    logits = np.asarray(logits, dtype=np.float64)
    bias = np.zeros(logits.shape[1], dtype=np.float64)

    best = f1_score(
        y, (logits + bias).argmax(1),
        average="macro", zero_division=0
    )

    rng = np.random.default_rng(SEED)
    step_sizes = [0.5, 0.2, 0.1, 0.05, 0.02, 0.01]

    for step in step_sizes:
        for _ in range(3):
            improved = False
            order = rng.permutation(logits.shape[1])

            for cls in order:
                local_best = best
                local_bias = bias[cls]

                for delta in (-step, step):
                    trial = bias.copy()
                    trial[cls] += delta

                    score = f1_score(
                        y, (logits + trial).argmax(1),
                        average="macro", zero_division=0
                    )

                    if score > local_best:
                        local_best = score
                        local_bias = trial[cls]

                if local_best > best:
                    bias[cls] = local_bias
                    best = local_best
                    improved = True

            if not improved:
                break

    # Remove the arbitrary shared offset for stable representation.
    bias -= bias.mean()
    return bias, best

for name, path in CHECKPOINTS.items():
    if not path.exists():
        print(f"Skipping {name}: checkpoint not found: {path}")
        continue

    logits = get_logits(path)

    base_pred = logits.argmax(axis=1)
    base_f1 = f1_score(
        y_true, base_pred,
        average="macro", zero_division=0
    )

    bias, adjusted_f1 = optimize_bias(logits, y_true)

    results[name] = {
        "checkpoint": str(path),
        "base_validation_macro_f1": float(base_f1),
        "bias_adjusted_validation_macro_f1": float(adjusted_f1),
        "bias": bias.tolist()
    }

    print(f"\n{name.upper()}")
    print(f"Base validation Macro-F1:     {base_f1:.6f}")
    print(f"Adjusted validation Macro-F1: {adjusted_f1:.6f}")

# Save all comparisons
with open(OUT_DIR / "bias_optimization_results.json", "w") as f:
    json.dump(results, f, indent=2)

if results:
    best_name = max(
        results,
        key=lambda k: results[k]["bias_adjusted_validation_macro_f1"]
    )
    best = results[best_name]

    with open(OUT_DIR / "best_bias_config.json", "w") as f:
        json.dump({
            "selected_checkpoint": best["checkpoint"],
            "selected_model": best_name,
            "validation_macro_f1": best[
                "bias_adjusted_validation_macro_f1"
            ],
            "bias": best["bias"]
        }, f, indent=2)

    print("\nBEST MODEL:", best_name)
    print("Best adjusted validation Macro-F1:",
          f'{best["bias_adjusted_validation_macro_f1"]:.6f}')
    print("Saved to:", OUT_DIR)

In [ ]:
class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)

        self.context_gate = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)

        p = self.prev_proj(prev)
        c = self.curr_proj(curr)
        n = self.next_proj(nxt)

        gate = self.context_gate(torch.cat([p, c, n], dim=1))
        context = gate[:, :512] * p + gate[:, 512:] * n

        return self.fusion(c + context)

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn

BASE = "/kaggle/working/Phase3_Submissions"

CTX_PATH = f"{BASE}/05_arch4_context/arch4_context_embeddings.npz"
CKPT_PATH = f"{BASE}/06_arch4_gated_context/gated_context_classifier_best.pt"
FINETUNED_PATH = f"{BASE}/07_arch4_finetune_lr1e4/classifier_best.pt"

print("=" * 65)
print("STEP 1: CHECK FILES")
print("=" * 65)

for path in [CTX_PATH, CKPT_PATH, FINETUNED_PATH]:
    print(f"{'FOUND' if os.path.exists(path) else 'MISSING'}: {path}")

assert os.path.exists(CTX_PATH), "Context embeddings are missing."
assert os.path.exists(CKPT_PATH), "Original classifier checkpoint is missing."

data = np.load(CTX_PATH)
X_train = data["X_train"]
y_train = data["y_train"]
X_dev = data["X_dev"]
y_dev = data["y_dev"]

print("\nEmbedding shapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_dev:  ", X_dev.shape)
print("y_dev:  ", y_dev.shape)

assert X_train.ndim == 2 and X_train.shape[1] == 3072
assert X_dev.ndim == 2 and X_dev.shape[1] == 3072
assert len(X_train) == len(y_train)
assert len(X_dev) == len(y_dev)

class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)

        self.context_gate = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)
        p = self.prev_proj(prev)
        c = self.curr_proj(curr)
        n = self.next_proj(nxt)

        gate = self.context_gate(torch.cat([p, c, n], dim=1))
        context = gate[:, :512] * p + gate[:, 512:] * n
        return self.fusion(c + context)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    CKPT_PATH,
    map_location=device,
    weights_only=False
)

state = (
    checkpoint["model_state_dict"]
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint
    else checkpoint
)

model = GatedContextClassifier().to(device)
model.load_state_dict(state, strict=True)

print("\nCheckpoint loaded successfully.")
print("Device:", device)
print("Train labels:", len(np.unique(y_train)), "unique classes")
print("Dev labels:", len(np.unique(y_dev)), "unique classes")
print("\nCHECK PASSED — ready for the controlled experiment.")

In [ ]:
# ================================================================
# EXPERIMENT 1: ARCH4 + SQRT INVERSE-FREQUENCY WEIGHTED CE
# Initialize from original checkpoint; save all outputs separately.
# ================================================================

import os
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score

# -------------------- Configuration --------------------
SEED = 42
BATCH_SIZE = 128
LR = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30
PATIENCE = 7

BASE = "/kaggle/working/Phase3_Submissions"
CTX_PATH = f"{BASE}/05_arch4_context/arch4_context_embeddings.npz"
CKPT_PATH = f"{BASE}/06_arch4_gated_context/gated_context_classifier_best.pt"
OUT_DIR = f"{BASE}/09_arch4_finetune_sqrt_weighted"

os.makedirs(OUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 13

# -------------------- Load data --------------------
data = np.load(CTX_PATH)
X_train = data["X_train"].astype(np.float32)
y_train = data["y_train"].astype(np.int64)
X_dev = data["X_dev"].astype(np.float32)
y_dev = data["y_dev"].astype(np.int64)

print("Train:", X_train.shape, y_train.shape)
print("Dev:  ", X_dev.shape, y_dev.shape)
print("Device:", DEVICE)

# -------------------- Correct checkpoint architecture --------------------
class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()

        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)

        self.context_gate = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)

        p = self.prev_proj(prev)
        c = self.curr_proj(curr)
        n = self.next_proj(nxt)

        gate = self.context_gate(torch.cat([p, c, n], dim=1))
        context = gate[:, :512] * p + gate[:, 512:] * n

        return self.fusion(c + context)

# -------------------- Load original checkpoint --------------------
checkpoint = torch.load(
    CKPT_PATH,
    map_location=DEVICE,
    weights_only=False
)

state = (
    checkpoint["model_state_dict"]
    if isinstance(checkpoint, dict)
    and "model_state_dict" in checkpoint
    else checkpoint
)

model = GatedContextClassifier().to(DEVICE)
model.load_state_dict(state, strict=True)

# -------------------- Baseline evaluation --------------------
@torch.no_grad()
def predict_logits(model, X, batch_size=512):
    model.eval()
    results = []

    for start in range(0, len(X), batch_size):
        xb = torch.as_tensor(
            X[start:start + batch_size],
            dtype=torch.float32,
            device=DEVICE
        )
        results.append(model(xb).cpu().numpy())

    return np.concatenate(results, axis=0)

initial_logits = predict_logits(model, X_dev)
initial_f1 = f1_score(
    y_dev, initial_logits.argmax(axis=1), average="macro"
)

print(f"\nOriginal checkpoint dev Macro-F1: {initial_f1:.6f}")

# -------------------- Compute class weights --------------------
counts = np.bincount(y_train, minlength=NUM_CLASSES).astype(np.float64)

weights = np.sqrt(
    len(y_train) / np.maximum(counts, 1.0)
)
weights /= weights.mean()

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE)
)

print("\nClass weights:")
for i, (count, weight) in enumerate(zip(counts, weights)):
    print(f"{i:2d}: count={int(count):5d}, weight={weight:.4f}")

# -------------------- Build training loader --------------------
train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train)
)

generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    pin_memory=(DEVICE.type == "cuda")
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

# -------------------- Fine-tuning --------------------
best_f1 = initial_f1
best_epoch = 0
best_state = copy.deepcopy(model.state_dict())
bad_epochs = 0
history = []

print("\nStarting weighted fine-tuning...")

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)

    train_loss = total_loss / len(train_dataset)

    dev_logits = predict_logits(model, X_dev)
    dev_f1 = f1_score(
        y_dev,
        dev_logits.argmax(axis=1),
        average="macro"
    )

    scheduler.step(dev_f1)
    improved = dev_f1 > best_f1

    if improved:
        best_f1 = dev_f1
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1

    row = {
        "epoch": epoch,
        "train_loss": float(train_loss),
        "dev_macro_f1": float(dev_f1),
        "learning_rate": float(optimizer.param_groups[0]["lr"])
    }
    history.append(row)

    marker = " <-- BEST" if improved else ""
    print(
        f"Epoch {epoch:02d} | "
        f"loss={train_loss:.4f} | "
        f"dev_macro_f1={dev_f1:.6f} | "
        f"lr={optimizer.param_groups[0]['lr']:.2e}"
        f"{marker}"
    )

    if bad_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

# -------------------- Save separate checkpoint --------------------
model.load_state_dict(best_state)

save_path = os.path.join(
    OUT_DIR, "classifier_sqrt_weighted_best.pt"
)

torch.save({
    "model_state_dict": best_state,
    "best_epoch": best_epoch,
    "best_dev_macro_f1": best_f1,
    "initial_dev_macro_f1": initial_f1,
    "class_weights": weights.tolist(),
    "seed": SEED
}, save_path)

with open(os.path.join(OUT_DIR, "training_history.json"), "w") as f:
    json.dump(history, f, indent=2)

print("\n" + "=" * 60)
print("EXPERIMENT COMPLETE")
print(f"Original checkpoint dev Macro-F1: {initial_f1:.6f}")
print(f"Best weighted dev Macro-F1:       {best_f1:.6f}")
print(f"Best epoch:                       {best_epoch}")
print(f"Change in Macro-F1:               {best_f1 - initial_f1:+.6f}")
print(f"Saved checkpoint:                 {save_path}")
print("=" * 60)


In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

BASE = "/kaggle/working/Phase3_Submissions"

CTX_PATH = f"{BASE}/05_arch4_context/arch4_context_embeddings.npz"
ORIGINAL_PATH = (
    f"{BASE}/06_arch4_gated_context/"
    "gated_context_classifier_best.pt"
)
WEIGHTED_PATH = (
    f"{BASE}/09_arch4_finetune_sqrt_weighted/"
    "classifier_sqrt_weighted_best.pt"
)

OUT_DIR = f"{BASE}/10_arch4_calibration_comparison"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 13

data = np.load(CTX_PATH)
X_dev = data["X_dev"].astype(np.float32)
y_dev = data["y_dev"].astype(np.int64)

class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)
        self.context_gate = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.Sigmoid()
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)
        p = self.prev_proj(prev)
        c = self.curr_proj(curr)
        n = self.next_proj(nxt)
        gate = self.context_gate(torch.cat([p, c, n], dim=1))
        context = gate[:, :512] * p + gate[:, 512:] * n
        return self.fusion(c + context)

@torch.no_grad()
def get_logits(path):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    state = ckpt.get("model_state_dict", ckpt)

    model = GatedContextClassifier().to(DEVICE)
    model.load_state_dict(state, strict=True)
    model.eval()

    logits = []
    for start in range(0, len(X_dev), 512):
        xb = torch.tensor(
            X_dev[start:start + 512],
            dtype=torch.float32,
            device=DEVICE
        )
        logits.append(model(xb).cpu().numpy())

    return np.concatenate(logits, axis=0)

def calibrate_bias(logits, labels, passes=3, bound=1.0):
    # Exploratory in-sample calibration, NOT a held-out estimate.
    bias = np.zeros(NUM_CLASSES, dtype=np.float64)
    grid = np.linspace(-bound, bound, 81)

    def score(b):
        return f1_score(
            labels,
            (logits + b).argmax(axis=1),
            average="macro",
            zero_division=0
        )

    best = score(bias)

    for _ in range(passes):
        for c in range(NUM_CLASSES):
            best_value = bias[c]
            for value in grid:
                trial = bias.copy()
                trial[c] = value
                value_score = score(trial)

                if value_score > best:
                    best = value_score
                    best_value = value

            bias[c] = best_value

        bias -= bias.mean()
        best = score(bias)

    return bias, score(bias)

results = {}

for name, path in [
    ("original", ORIGINAL_PATH),
    ("weighted", WEIGHTED_PATH)
]:
    assert os.path.exists(path), f"Checkpoint missing: {path}"

    logits = get_logits(path)
    raw_f1 = f1_score(
        y_dev, logits.argmax(axis=1),
        average="macro", zero_division=0
    )

    bias, calibrated_f1 = calibrate_bias(logits, y_dev)

    results[name] = {
        "raw_dev_macro_f1": float(raw_f1),
        "in_sample_calibrated_dev_macro_f1": float(calibrated_f1),
        "bias": bias.tolist()
    }

    print(f"\n{name.upper()} MODEL")
    print(f"Raw dev Macro-F1:                  {raw_f1:.6f}")
    print(f"In-sample calibrated dev Macro-F1: {calibrated_f1:.6f}")

output_path = os.path.join(OUT_DIR, "comparison.json")

with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved comparison:", output_path)
print("\nIMPORTANT: Calibrated scores use the same dev labels for tuning")
print("and evaluation; they are not unbiased estimates of test performance.")

In [ ]:
import os
import json
import numpy as np

BASE = "/kaggle/working/Phase3_Submissions"
CTX_PATH = f"{BASE}/05_arch4_context/arch4_context_embeddings.npz"

print("=" * 65)
print("DOCUMENT-LEVEL CALIBRATION: DATA AUDIT")
print("=" * 65)

# 1. Inspect cached embedding metadata
data = np.load(CTX_PATH)

print("\nNPZ contents:")
for key in data.files:
    arr = data[key]
    print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")

# 2. Look for original source JSON files
roots = [
    "/kaggle/working",
    "/kaggle/input"
]

keywords = [
    "CL_train.json",
    "IT_train.json",
    "CL_dev.json",
    "IT_dev.json"
]

found = {}

for root in roots:
    for current, dirs, files in os.walk(root):
        # Avoid traversing large irrelevant directories unnecessarily.
        dirs[:] = [
            d for d in dirs
            if d not in {".git", "__pycache__", "node_modules"}
        ]

        for filename in files:
            if filename in keywords:
                path = os.path.join(current, filename)
                found[filename] = path

print("\nOriginal dataset JSON files:")
if found:
    for name, path in sorted(found.items()):
        print(f"  FOUND {name}: {path}")
else:
    print("  No matching JSON files found.")

# 3. Inspect source JSON structure without printing sentence contents
for name, path in sorted(found.items()):
    try:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        print(f"\n{name}:")
        print("  Top-level type:", type(obj).__name__)

        if isinstance(obj, dict):
            print("  Number of top-level keys:", len(obj))
            print("  Example document IDs:", list(obj.keys())[:3])

            first_key = next(iter(obj), None)
            if first_key is not None:
                first_value = obj[first_key]
                print("  First document value type:", type(first_value).__name__)

                if isinstance(first_value, list):
                    print("  First document item count:", len(first_value))
                elif isinstance(first_value, dict):
                    print("  First document fields:", list(first_value.keys())[:10])

        elif isinstance(obj, list):
            print("  Number of records:", len(obj))
            if obj:
                print("  First record fields:",
                      list(obj[0].keys())[:10]
                      if isinstance(obj[0], dict)
                      else type(obj[0]).__name__)

    except Exception as e:
        print(f"  Could not inspect {name}: {type(e).__name__}: {e}")

print("\nAudit complete. No models or checkpoints were modified.")

In [ ]:
import json
import numpy as np
from pathlib import Path

DATA_DIR = Path("/kaggle/working/Phase_3_RR/Phase_3_RR/Train")
CTX_PATH = Path(
    "/kaggle/working/Phase3_Submissions/05_arch4_context/"
    "arch4_context_embeddings.npz"
)

def load_docs(filename):
    with open(DATA_DIR / filename, "r", encoding="utf-8") as f:
        docs = json.load(f)

    result = []
    for doc in docs:
        doc_id = doc["id"]
        texts = doc["text"]
        labels = doc["labels"]

        result.append({
            "id": str(doc_id),
            "n_text": len(texts),
            "n_labels": len(labels),
            "labels": labels
        })

    return result

cl_train = load_docs("CL_train.json")
it_train = load_docs("IT_train.json")
cl_dev = load_docs("CL_dev.json")
it_dev = load_docs("IT_dev.json")

def report(name, docs):
    print(f"\n{name}:")
    print("  Documents:", len(docs))
    print("  Total text sentences:", sum(d["n_text"] for d in docs))
    print("  Total labels:", sum(d["n_labels"] for d in docs))
    print("  Text/label count mismatches:",
          [(d["id"], d["n_text"], d["n_labels"])
           for d in docs if d["n_text"] != d["n_labels"]])
    print("  Document IDs:", [d["id"] for d in docs])

report("CL TRAIN", cl_train)
report("IT TRAIN", it_train)
report("CL DEV", cl_dev)
report("IT DEV", it_dev)

data = np.load(CTX_PATH)
y_train = data["y_train"]
y_dev = data["y_dev"]

print("\n" + "=" * 60)
print("EMBEDDING/LABEL ALIGNMENT CHECK")
print("=" * 60)

for name, docs, y, expected in [
    ("TRAIN", cl_train + it_train, y_train, len(y_train)),
    ("DEV", cl_dev + it_dev, y_dev, len(y_dev)),
]:
    total_json_labels = sum(d["n_labels"] for d in docs)

    print(f"\n{name}:")
    print("  JSON documents:", len(docs))
    print("  JSON labels:", total_json_labels)
    print("  NPZ label rows:", expected)
    print("  Total counts match:", total_json_labels == expected)

print("\nAudit finished. No files were modified.")

In [ ]:
import json
import numpy as np
from pathlib import Path

DATA_DIR = Path("/kaggle/working/Phase_3_RR/Phase_3_RR/Train")
CTX_PATH = Path(
    "/kaggle/working/Phase3_Submissions/05_arch4_context/"
    "arch4_context_embeddings.npz"
)

def read_docs(filename):
    with open(DATA_DIR / filename, "r", encoding="utf-8") as f:
        return json.load(f)

cl_train = read_docs("CL_train.json")
it_train = read_docs("IT_train.json")
cl_dev   = read_docs("CL_dev.json")
it_dev   = read_docs("IT_dev.json")

data = np.load(CTX_PATH)
y_train = data["y_train"]
y_dev = data["y_dev"]

# Use the official label vocabulary from the competition.
label_to_id = {
    "Fact": 0,
    "Issue": 1,
    "ArgumentPetitioner": 2,
    "ArgumentRespondent": 3,
    "Statute": 4,
    "Dissent": 5,
    "PrecedentReliedUpon": 6,
    "PrecedentNotReliedUpon": 7,
    "PrecedentOverruled": 8,
    "RulingByLowerCourt": 9,
    "RatioOfTheDecision": 10,
    "RulingByPresentCourt": 11,
    "None": 12,
}

def encode_docs(docs):
    encoded = []
    unknown = set()

    for doc in docs:
        for label in doc["labels"]:
            if label not in label_to_id:
                unknown.add(label)
            else:
                encoded.append(label_to_id[label])

    if unknown:
        raise ValueError(f"Unknown labels: {sorted(unknown)}")

    return np.asarray(encoded, dtype=np.int64)

json_train_y = encode_docs(cl_train + it_train)
json_dev_y = encode_docs(cl_dev + it_dev)

print("=" * 65)
print("JSON ↔ NPZ LABEL ALIGNMENT")
print("=" * 65)

for split, expected, actual in [
    ("TRAIN", json_train_y, y_train),
    ("DEV", json_dev_y, y_dev),
]:
    print(f"\n{split}:")
    print("  Lengths match:", len(expected) == len(actual))
    print("  Exact label sequence match:", np.array_equal(expected, actual))

    if len(expected) == len(actual):
        mismatch = np.flatnonzero(expected != actual)
        print("  Number of mismatched rows:", len(mismatch))
        if len(mismatch):
            print("  First mismatches:", mismatch[:20].tolist())

print("\n" + "=" * 65)
print("DOCUMENT GROUP SUMMARY")
print("=" * 65)

for split, docs in [
    ("TRAIN", cl_train + it_train),
    ("DEV", cl_dev + it_dev),
]:
    lengths = [len(doc["labels"]) for doc in docs]
    boundaries = np.cumsum(lengths)

    print(f"\n{split}:")
    print("  Documents:", len(docs))
    print("  First 10 document lengths:", lengths[:10])
    print("  First 10 cumulative boundaries:", boundaries[:10].tolist())

print("\nNo model or checkpoint was modified.")

In [ ]:
import json
import numpy as np
from pathlib import Path

DATA_DIR = Path("/kaggle/working/Phase_3_RR/Phase_3_RR/Train")
BASE = Path("/kaggle/working/Phase3_Submissions")
CTX_PATH = BASE / "05_arch4_context/arch4_context_embeddings.npz"

OUT_DIR = BASE / "11_document_group_calibration"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def load_docs(filename, source):
    with open(DATA_DIR / filename, "r", encoding="utf-8") as f:
        docs = json.load(f)

    result = []
    for doc in docs:
        result.append({
            "id": str(doc["id"]),
            "source": source,
            "n": len(doc["labels"]),
        })
    return result

train_docs = (
    load_docs("CL_train.json", "CL")
    + load_docs("IT_train.json", "IT")
)
dev_docs = (
    load_docs("CL_dev.json", "CL")
    + load_docs("IT_dev.json", "IT")
)

data = np.load(CTX_PATH)
y_train = data["y_train"]
y_dev = data["y_dev"]

def build_groups(docs, expected_rows, split):
    group_ids = []
    offsets = []
    cursor = 0

    for doc in docs:
        start = cursor
        end = start + doc["n"]

        group_ids.extend([doc["id"]] * doc["n"])
        offsets.append({
            "id": doc["id"],
            "source": doc["source"],
            "start": start,
            "end": end,
            "n": doc["n"],
        })
        cursor = end

    group_ids = np.asarray(group_ids, dtype=str)

    assert cursor == expected_rows, (
        f"{split}: document lengths total {cursor}, "
        f"but NPZ contains {expected_rows} rows"
    )

    return group_ids, offsets

train_groups, train_offsets = build_groups(
    train_docs, len(y_train), "TRAIN"
)
dev_groups, dev_offsets = build_groups(
    dev_docs, len(y_dev), "DEV"
)

# Check that document IDs do not overlap between train and dev.
train_id_set = {d["id"] for d in train_docs}
dev_id_set = {d["id"] for d in dev_docs}
overlap = train_id_set & dev_id_set

print("=" * 65)
print("DOCUMENT GROUP PREPARATION")
print("=" * 65)
print(f"Train: {len(train_docs)} documents, {len(train_groups)} sentences")
print(f"Dev:   {len(dev_docs)} documents, {len(dev_groups)} sentences")
print(f"Train/dev document-ID overlap: {len(overlap)}")
if overlap:
    print("Overlapping IDs:", sorted(overlap))

print("\nDevelopment document sizes:")
for item in dev_offsets:
    print(
        f"  {item['source']} | {item['id'][:65]:65s} "
        f"| sentences={item['n']:4d}"
    )

np.savez_compressed(
    OUT_DIR / "document_groups.npz",
    train_groups=train_groups,
    dev_groups=dev_groups,
)

with open(OUT_DIR / "document_offsets.json", "w", encoding="utf-8") as f:
    json.dump({
        "train": train_offsets,
        "dev": dev_offsets,
    }, f, indent=2)

print(f"\nSaved groups to: {OUT_DIR}")
print("No model checkpoints modified.")

In [ ]:
import torch
import numpy as np
from pathlib import Path

BASE = Path("/kaggle/working/Phase3_Submissions")
CTX_PATH = BASE / "05_arch4_context/arch4_context_embeddings.npz"
CKPT_PATH = BASE / "06_arch4_gated_context/gated_context_classifier_best.pt"
GROUP_PATH = BASE / "11_document_group_calibration/document_groups.npz"

print("=" * 65)
print("OOF CALIBRATION PREFLIGHT")
print("=" * 65)

# 1. Verify all required artifacts exist.
for path in [CTX_PATH, CKPT_PATH, GROUP_PATH]:
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {path}")

if not all(p.exists() for p in [CTX_PATH, CKPT_PATH, GROUP_PATH]):
    raise FileNotFoundError("A required artifact is missing. Check the paths above.")

# 2. Load data and groups.
data = np.load(CTX_PATH)
groups = np.load(GROUP_PATH)

X_train = data["X_train"]
y_train = data["y_train"]
X_dev = data["X_dev"]
y_dev = data["y_dev"]

train_groups = groups["train_groups"]
dev_groups = groups["dev_groups"]

assert len(X_train) == len(y_train) == len(train_groups)
assert len(X_dev) == len(y_dev) == len(dev_groups)

print("\nDataset:")
print(f"  X_train: {X_train.shape}")
print(f"  X_dev:   {X_dev.shape}")
print(f"  Classes: {len(np.unique(y_train))}")
print(f"  Train documents: {len(np.unique(train_groups))}")
print(f"  Dev documents:   {len(np.unique(dev_groups))}")

# 3. Inspect checkpoint metadata and state-dict shapes.
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

print("\nCheckpoint top-level keys:")
if isinstance(ckpt, dict):
    print(list(ckpt.keys()))

state = ckpt.get("model_state_dict", ckpt)
print("\nClassifier parameter shapes:")
for name, tensor in state.items():
    if isinstance(tensor, torch.Tensor):
        print(f"  {name}: {tuple(tensor.shape)}")

# 4. Report training label counts.
counts = np.bincount(y_train, minlength=13)
print("\nTraining class counts:")
for i, count in enumerate(counts):
    print(f"  Class {i:2d}: {count:5d}")

print("\nPreflight complete. No model or checkpoint modified.")

In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score

BASE = Path("/kaggle/working/Phase3_Submissions")
CTX_PATH = BASE / "05_arch4_context/arch4_context_embeddings.npz"
CKPT_PATH = BASE / "06_arch4_gated_context/gated_context_classifier_best.pt"
GROUP_PATH = BASE / "11_document_group_calibration/document_groups.npz"

OUT_DIR = BASE / "11_document_group_calibration"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_SPLITS = 5
BATCH_SIZE = 256
EPOCHS = 8
LR = 2e-5
WEIGHT_DECAY = 1e-4
PATIENCE = 2

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

data = np.load(CTX_PATH)
group_data = np.load(GROUP_PATH)
X = data["X_train"]
y = data["y_train"]
groups = group_data["train_groups"]

assert len(X) == len(y) == len(groups)
assert len(np.unique(groups)) >= N_SPLITS

class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)
        self.context_gate = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.Sigmoid()
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)
        p = self.prev_proj(prev)
        c = self.curr_proj(curr)
        n = self.next_proj(nxt)
        gate = self.context_gate(torch.cat([p, c, n], dim=1))
        context = gate[:, :512] * p + gate[:, 512:] * n
        return self.fusion(c + context)

checkpoint = torch.load(
    CKPT_PATH, map_location="cpu", weights_only=False
)
initial_state = checkpoint["model_state_dict"]

oof_logits = np.full((len(y), 13), np.nan, dtype=np.float32)
fold_records = []

splitter = GroupKFold(n_splits=N_SPLITS)

for fold, (train_idx, val_idx) in enumerate(
    splitter.split(X, y, groups), start=1
):
    print(f"\nFold {fold}/{N_SPLITS}")
    print("  Train documents:", len(np.unique(groups[train_idx])))
    print("  Held-out documents:", len(np.unique(groups[val_idx])))

    model = GatedContextClassifier().to(device)
    model.load_state_dict(initial_state)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    criterion = nn.CrossEntropyLoss()

    Xt = torch.from_numpy(X[train_idx])
    yt = torch.from_numpy(y[train_idx]).long()
    Xv = torch.from_numpy(X[val_idx])

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xt, yt),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    best_state = None
    best_loss = float("inf")
    bad_epochs = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        seen = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running_loss += loss.item() * len(yb)
            seen += len(yb)

        model.eval()
        with torch.no_grad():
            val_logits = []
            for start in range(0, len(Xv), BATCH_SIZE):
                xb = Xv[start:start + BATCH_SIZE].to(device)
                val_logits.append(model(xb).cpu())
            val_logits = torch.cat(val_logits).numpy()

        # Held-out labels are used only to monitor this fold's loss.
        val_loss = nn.functional.cross_entropy(
            torch.from_numpy(val_logits),
            torch.from_numpy(y[val_idx]).long()
        ).item()

        print(
            f"  Epoch {epoch}: "
            f"train_loss={running_loss/max(seen,1):.4f}, "
            f"heldout_loss={val_loss:.4f}"
        )

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits_parts = []
        for start in range(0, len(val_idx), BATCH_SIZE):
            xb = torch.from_numpy(
                X[val_idx[start:start + BATCH_SIZE]]
            ).to(device)
            logits_parts.append(model(xb).cpu().numpy())

    oof_logits[val_idx] = np.concatenate(logits_parts, axis=0)
    fold_records.append({
        "fold": fold,
        "train_documents": len(np.unique(groups[train_idx])),
        "heldout_documents": len(np.unique(groups[val_idx])),
        "best_heldout_loss": float(best_loss),
    })

    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert np.isfinite(oof_logits).all(), "Some OOF logits are missing."

oof_pred = oof_logits.argmax(axis=1)
oof_f1 = f1_score(y, oof_pred, average="macro", zero_division=0)

np.savez_compressed(
    OUT_DIR / "grouped_oof_logits.npz",
    oof_logits=oof_logits,
    y_train=y,
    train_groups=groups,
)

with open(OUT_DIR / "grouped_oof_metadata.json", "w") as f:
    json.dump({
        "folds": fold_records,
        "macro_f1": float(oof_f1),
        "note": (
            "Diagnostic only: fold models initialized from a checkpoint "
            "trained before fold assignment. Not fully independent OOF."
        ),
    }, f, indent=2)

print("\n" + "=" * 60)
print("OOF DIAGNOSTIC COMPLETE")
print("=" * 60)
print(f"OOF Macro-F1: {oof_f1:.6f}")
print("Saved:", OUT_DIR / "grouped_oof_logits.npz")
print("Caution: these are not fully independent OOF predictions.")

In [ ]:
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from scipy.optimize import minimize

BASE = Path("/kaggle/working/Phase3_Submissions")
OOF_PATH = BASE / "11_document_group_calibration/grouped_oof_logits.npz"

data = np.load(OOF_PATH)
logits = data["oof_logits"].astype(np.float64)
y = data["y_train"].astype(np.int64)

group_path = BASE / "11_document_group_calibration/document_groups.npz"
groups = np.load(group_path)["train_groups"]

assert logits.shape == (len(y), 13)
assert len(groups) == len(y)

def macro_f1(z, labels):
    return f1_score(
        labels, np.argmax(z, axis=1),
        average="macro", zero_division=0
    )

def fit_temperature(train_logits, train_y):
    def objective(params):
        T = np.exp(params[0])
        z = train_logits / T
        z = z - z.max(axis=1, keepdims=True)
        log_probs = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
        return -log_probs[np.arange(len(train_y)), train_y].mean()

    result = minimize(
        objective,
        x0=np.array([0.0]),
        method="L-BFGS-B",
        bounds=[(-2.0, 2.0)]
    )
    return float(np.exp(result.x[0]))

def fit_temperature_bias(train_logits, train_y, reg=5.0):
    n_classes = train_logits.shape[1]

    def objective(params):
        T = np.exp(params[0])
        bias = params[1:]
        z = train_logits / T + bias
        z = z - z.max(axis=1, keepdims=True)
        log_probs = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
        nll = -log_probs[np.arange(len(train_y)), train_y].mean()

        # Strong shrinkage discourages extreme per-class adjustments.
        penalty = reg * np.mean(bias ** 2)
        return nll + penalty

    result = minimize(
        objective,
        x0=np.zeros(n_classes + 1),
        method="L-BFGS-B",
        bounds=[(-2.0, 2.0)] + [(-1.5, 1.5)] * n_classes
    )

    return float(np.exp(result.x[0])), result.x[1:].copy()

print("=" * 65)
print("GROUPED OOF CALIBRATION COMPARISON")
print("=" * 65)

raw_f1 = macro_f1(logits, y)
print(f"Raw OOF Macro-F1: {raw_f1:.6f}")

splitter = GroupKFold(n_splits=5)
pred_temp = np.full(len(y), -1, dtype=np.int64)
pred_bias = np.full(len(y), -1, dtype=np.int64)
temperatures = []
bias_norms = []

for fold, (tr, va) in enumerate(splitter.split(logits, y, groups), 1):
    T = fit_temperature(logits[tr], y[tr])
    pred_temp[va] = np.argmax(logits[va] / T, axis=1)

    T2, bias = fit_temperature_bias(logits[tr], y[tr], reg=5.0)
    pred_bias[va] = np.argmax(logits[va] / T2 + bias, axis=1)

    temperatures.append((T, T2))
    bias_norms.append(float(np.linalg.norm(bias)))

    print(
        f"Fold {fold}: temperature={T:.3f}, "
        f"temperature+bias={T2:.3f}, "
        f"bias_L2={np.linalg.norm(bias):.3f}"
    )

temp_f1 = f1_score(y, pred_temp, average="macro", zero_division=0)
bias_f1 = f1_score(y, pred_bias, average="macro", zero_division=0)

print("\nResults:")
print(f"  Raw OOF:                   {raw_f1:.6f}")
print(f"  Grouped temperature OOF:   {temp_f1:.6f}")
print(f"  Temperature + bias OOF:    {bias_f1:.6f}")
print(f"  Temperature delta:         {temp_f1 - raw_f1:+.6f}")
print(f"  Temperature+bias delta:    {bias_f1 - raw_f1:+.6f}")

print("\nInterpretation:")
if bias_f1 > temp_f1 and bias_f1 > raw_f1:
    print("Regularized class biases improved grouped OOF F1.")
elif temp_f1 > raw_f1:
    print("Global temperature scaling improved grouped OOF F1.")
else:
    print("Calibration did not improve grouped OOF F1; retain raw logits.")

print(
    "\nReminder: these results are diagnostic, not fully independent, "
    "because the starting checkpoint was trained on all training documents."
)

In [ ]:
import json
import torch
from pathlib import Path

BASE = Path("/kaggle/working/Phase3_Submissions")

paths = [
    BASE / "10_arch4_calibration_comparison/comparison.json",
    BASE / "06_arch4_gated_context/gated_context_classifier_best.pt",
    BASE / "07_arch4_finetune_lr1e4/classifier_best.pt",
    BASE / "09_arch4_finetune_sqrt_weighted/classifier_sqrt_weighted_best.pt",
]

for path in paths:
    print("\n" + "=" * 65)
    print(path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    if path.suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        print(json.dumps(obj, indent=2)[:10000])
    else:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        for key, value in ckpt.items():
            if key != "model_state_dict":
                print(f"{key}: {value}")

In [ ]:
from pathlib import Path

ROOTS = [
    Path("/kaggle/working/Phase3_Submissions"),
    Path("/kaggle/working"),
]

keywords = ("logit", "prediction", "preds", "prob", "calibration")

found = set()

for root in ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        name = path.name.lower()
        if any(k in name for k in keywords):
            found.add(path)

print("Potential prediction/logit artifacts:")
for path in sorted(found):
    print(path)

if not found:
    print("No matching artifacts found.")

In [ ]:
import json
from pathlib import Path

BASE = Path("/kaggle/working/Phase3_Submissions")

FILES = [
    BASE / "06_arch4_gated_context/prediction_config.json",
    BASE / "07_final_predictions/predictions_combined.json",
    BASE / "07_final_predictions/CL_predictions.json",
    BASE / "07_final_predictions/IT_predictions.json",
    BASE / "Arch4/prediction.json",
]

for path in FILES:
    print("\n" + "=" * 70)
    print("FILE:", path)
    print("EXISTS:", path.exists())

    if not path.exists():
        continue

    try:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, dict):
            print("Type: dict")
            print("Number of keys:", len(obj))
            print("First keys:", list(obj.keys())[:5])

            # Show metadata/configuration without dumping every prediction.
            for key, value in obj.items():
                if isinstance(value, (int, float, bool, type(None))):
                    print(f"  {key}: {value}")
                elif isinstance(value, str) and len(value) < 300:
                    print(f"  {key}: {value}")
                elif isinstance(value, list):
                    print(f"  {key}: list, length={len(value)}")
                elif isinstance(value, dict):
                    print(f"  {key}: dict, keys={list(value.keys())[:5]}")

        elif isinstance(obj, list):
            print("Type: list")
            print("Length:", len(obj))
            if obj:
                print("First record type:", type(obj[0]).__name__)
                if isinstance(obj[0], dict):
                    print("First record keys:", list(obj[0].keys())[:15])

    except Exception as e:
        print("Could not inspect:", repr(e))

print("\nInspection complete. No files modified.")

In [ ]:
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)

BASE = Path("/kaggle/working/Phase3_Submissions")
DATA = Path("/kaggle/working/Phase_3_RR/Phase_3_RR/Train")

PRED_FILE = BASE / "Arch4/prediction.json"

LABELS = [
    "Fact",
    "Issue",
    "ArgumentPetitioner",
    "ArgumentRespondent",
    "Statute",
    "Dissent",
    "PrecedentReliedUpon",
    "PrecedentNotReliedUpon",
    "PrecedentOverruled",
    "RulingByLowerCourt",
    "RatioOfTheDecision",
    "RulingByPresentCourt",
    "None",
]

label_to_id = {label: i for i, label in enumerate(LABELS)}

with open(PRED_FILE, "r", encoding="utf-8") as f:
    predictions = json.load(f)

dev_files = [
    DATA / "CL_dev.json",
    DATA / "IT_dev.json",
]

gold_all = []
pred_all = []
doc_results = []

for path in dev_files:
    with open(path, "r", encoding="utf-8") as f:
        documents = json.load(f)

    for doc in documents:
        doc_id = doc["id"]
        gold = doc["labels"]
        pred = predictions.get(doc_id)

        if pred is None:
            print(f"WARNING: Missing prediction for {doc_id}")
            continue

        if len(gold) != len(pred):
            print(
                f"WARNING: Length mismatch for {doc_id}: "
                f"gold={len(gold)}, pred={len(pred)}"
            )
            continue

        unknown = set(gold + pred) - set(LABELS)
        if unknown:
            print(f"WARNING: Unknown labels in {doc_id}: {unknown}")
            continue

        gold_ids = [label_to_id[x] for x in gold]
        pred_ids = [label_to_id[x] for x in pred]

        gold_all.extend(gold_ids)
        pred_all.extend(pred_ids)

        doc_f1 = f1_score(
            gold_ids, pred_ids,
            labels=list(range(len(LABELS))),
            average="macro",
            zero_division=0
        )
        doc_results.append((doc_id, len(gold), doc_f1))

y_true = np.asarray(gold_all)
y_pred = np.asarray(pred_all)

print("=" * 70)
print("DEVELOPMENT EVALUATION")
print("=" * 70)
print("Documents evaluated:", len(doc_results))
print("Sentences evaluated:", len(y_true))
print("Macro-F1:", round(
    f1_score(
        y_true, y_pred,
        labels=list(range(len(LABELS))),
        average="macro",
        zero_division=0
    ), 6
))

print("\nPER-CLASS REPORT")
print(classification_report(
    y_true, y_pred,
    labels=list(range(len(LABELS))),
    target_names=LABELS,
    digits=4,
    zero_division=0
))

print("\nPER-CLASS CONFUSION COUNTS")
cm = confusion_matrix(
    y_true, y_pred,
    labels=list(range(len(LABELS)))
)

for i, label in enumerate(LABELS):
    row = cm[i].copy()
    row[i] = 0
    top = np.argsort(row)[::-1][:2]
    errors = [
        f"{LABELS[j]}={row[j]}"
        for j in top if row[j] > 0
    ]
    print(
        f"{label:28s} gold={cm[i].sum():4d} "
        f"correct={cm[i, i]:4d} "
        f"top errors: {', '.join(errors) if errors else 'none'}"
    )

print("\nWORST DEVELOPMENT DOCUMENTS")
for doc_id, n, score in sorted(doc_results, key=lambda x: x[2])[:5]:
    print(f"F1={score:.4f} | sentences={n:4d} | {doc_id}")

print("\nNo files modified.")

In [ ]:
import torch
import numpy as np
from pathlib import Path

BASE = Path("/kaggle/working/Phase3_Submissions")

DATA_FILE = BASE / "05_arch4_context/arch4_context_embeddings.npz"
CKPT_FILE = BASE / "06_arch4_gated_context/gated_context_classifier_best.pt"

print("Embedding file exists:", DATA_FILE.exists())
print("Checkpoint exists:", CKPT_FILE.exists())

if DATA_FILE.exists():
    data = np.load(DATA_FILE)
    for key in data.files:
        arr = data[key]
        print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

if CKPT_FILE.exists():
    ckpt = torch.load(
        CKPT_FILE,
        map_location="cpu",
        weights_only=False
    )
    print("\nCheckpoint type:", type(ckpt).__name__)

    if isinstance(ckpt, dict):
        print("Checkpoint keys:", list(ckpt.keys()))

        for key, value in ckpt.items():
            if isinstance(value, (int, float, str, bool)):
                print(f"{key}: {value}")
            elif isinstance(value, dict):
                print(f"{key}: dict with {len(value)} entries")
            elif isinstance(value, torch.Tensor):
                print(f"{key}: tensor shape={tuple(value.shape)}")

print("\nVerification complete.")

In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn

from pathlib import Path
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)

BASE = Path("/kaggle/working/Phase3_Submissions")

DATA_FILE = BASE / "05_arch4_context/arch4_context_embeddings.npz"
CKPT_FILE = BASE / "06_arch4_gated_context/gated_context_classifier_best.pt"
CONFIG_FILE = BASE / "06_arch4_gated_context/prediction_config.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABELS = [
    "Fact",
    "Issue",
    "ArgumentPetitioner",
    "ArgumentRespondent",
    "Statute",
    "Dissent",
    "PrecedentReliedUpon",
    "PrecedentNotReliedUpon",
    "PrecedentOverruled",
    "RulingByLowerCourt",
    "RatioOfTheDecision",
    "RulingByPresentCourt",
    "None",
]

# --------------------------------------------------
# 1. Reconstruct the exact original architecture
# --------------------------------------------------
class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()

        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)

        self.context_gate = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)

        p = self.prev_proj(prev)
        c = self.curr_proj(curr)
        n = self.next_proj(nxt)

        gate = self.context_gate(torch.cat([p, c, n], dim=1))

        context = gate[:, :512] * p + gate[:, 512:] * n

        return self.fusion(c + context)


# --------------------------------------------------
# 2. Load checkpoint and development embeddings
# --------------------------------------------------
checkpoint = torch.load(
    CKPT_FILE,
    map_location="cpu",
    weights_only=False
)

model = GatedContextClassifier(
    num_classes=checkpoint["num_classes"]
)

model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(DEVICE)
model.eval()

data = np.load(DATA_FILE)

X_dev = torch.from_numpy(data["X_dev"]).float()
y_dev = data["y_dev"].astype(np.int64)

assert X_dev.shape == (2142, 3072), X_dev.shape
assert len(y_dev) == len(X_dev)

# --------------------------------------------------
# 3. Generate development predictions
# --------------------------------------------------
all_logits = []

with torch.no_grad():
    for start in range(0, len(X_dev), 256):
        xb = X_dev[start:start + 256].to(DEVICE)
        logits = model(xb)
        all_logits.append(logits.cpu().numpy())

logits = np.concatenate(all_logits, axis=0)
pred_raw = logits.argmax(axis=1)

raw_f1 = f1_score(
    y_dev,
    pred_raw,
    labels=list(range(13)),
    average="macro",
    zero_division=0
)

print("=" * 65)
print("ORIGINAL GATED-CONTEXT CLASSIFIER")
print("=" * 65)
print("Device:", DEVICE)
print("Checkpoint epoch:", checkpoint["best_epoch"])
print("Sentences:", len(y_dev))
print("Stored baseline F1:", checkpoint["base_validation_macro_f1"])
print("Recomputed raw Macro-F1:", round(raw_f1, 6))

# --------------------------------------------------
# 4. Evaluate class-wise performance
# --------------------------------------------------
print("\nPER-CLASS REPORT")
print(classification_report(
    y_dev,
    pred_raw,
    labels=list(range(13)),
    target_names=LABELS,
    digits=4,
    zero_division=0
))

cm = confusion_matrix(y_dev, pred_raw, labels=list(range(13)))

print("\nTOP CONFUSIONS BY GOLD LABEL")
for i, label in enumerate(LABELS):
    row = cm[i].copy()
    row[i] = 0

    top = np.argsort(row)[::-1][:3]
    errors = [
        f"{LABELS[j]}: {row[j]}"
        for j in top if row[j] > 0
    ]

    print(
        f"{label:28s} "
        f"support={cm[i].sum():4d} "
        f"correct={cm[i, i]:4d} | "
        + (", ".join(errors) if errors else "no errors")
    )

# --------------------------------------------------
# 5. Save raw development logits for future analysis
# --------------------------------------------------
OUT_DIR = BASE / "12_dev_evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    OUT_DIR / "original_dev_logits.npz",
    logits=logits,
    y_dev=y_dev,
    pred_raw=pred_raw
)

print("\nSaved development logits to:")
print(OUT_DIR / "original_dev_logits.npz")
print("Evaluation complete. Existing model files unchanged.")

In [ ]:
import json
import numpy as np
from pathlib import Path

BASE = Path("/kaggle/working/Phase3_Submissions")

CONFIG_FILE = BASE / "06_arch4_gated_context/prediction_config.json"
DATA_FILE = BASE / "05_arch4_context/arch4_context_embeddings.npz"

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    config = json.load(f)

mapping = config["label_to_id"]
biases = config.get("class_biases", {})

print("=" * 65)
print("ACTUAL CHECKPOINT LABEL MAPPING")
print("=" * 65)

for label, idx in sorted(mapping.items(), key=lambda x: x[1]):
    print(f"Class ID {idx:2d}: {label}")

print("\nClass-bias values:")
for label, value in biases.items():
    print(f"{label:28s}: {value}")

data = np.load(DATA_FILE)
y = data["y_dev"]

print("\n" + "=" * 65)
print("DEVELOPMENT LABEL DISTRIBUTION USING CHECKPOINT MAPPING")
print("=" * 65)

id_to_label = {idx: label for label, idx in mapping.items()}

for idx in sorted(id_to_label):
    print(f"{id_to_label[idx]:28s}: {(y == idx).sum()}")

print("\nMapping checks:")
print("Number of labels:", len(mapping))
print("IDs are exactly 0-12:", sorted(mapping.values()) == list(range(13)))
print("All development IDs valid:", set(np.unique(y)).issubset(set(mapping.values())))

print("\nDiagnostic complete. No files modified.")

In [ ]:
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score

BASE = Path("/kaggle/working/Phase3_Submissions")

LOGITS_FILE = BASE / "12_dev_evaluation/original_dev_logits.npz"
CONFIG_FILE = BASE / "06_arch4_gated_context/prediction_config.json"

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    config = json.load(f)

saved = np.load(LOGITS_FILE)
logits = saved["logits"]
y_true = saved["y_dev"]

label_to_id = config["label_to_id"]
bias_dict = config["class_biases"]

num_classes = len(label_to_id)
bias = np.zeros(num_classes, dtype=np.float32)

for label, value in bias_dict.items():
    bias[label_to_id[label]] = float(value)

pred_raw = logits.argmax(axis=1)
pred_biased = (logits + bias[None, :]).argmax(axis=1)

def macro_f1(y, pred):
    return f1_score(
        y,
        pred,
        labels=list(range(num_classes)),
        average="macro",
        zero_division=0
    )

print("=" * 60)
print("RAW VS STORED CLASS-BIAS CALIBRATION")
print("=" * 60)
print(f"Raw Macro-F1:           {macro_f1(y_true, pred_raw):.6f}")
print(f"Bias-adjusted Macro-F1:  {macro_f1(y_true, pred_biased):.6f}")
print(f"Checkpoint raw F1:      {config['base_validation_macro_f1']:.6f}")
print(f"Checkpoint biased F1:   {config['bias_adjusted_validation_macro_f1']:.6f}")

print("\nPrediction counts by class:")
id_to_label = {idx: label for label, idx in label_to_id.items()}

for idx in range(num_classes):
    print(
        f"{id_to_label[idx]:28s} "
        f"gold={np.sum(y_true == idx):4d} "
        f"raw={np.sum(pred_raw == idx):4d} "
        f"biased={np.sum(pred_biased == idx):4d}"
    )

print("\nNo files modified.")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score

BASE = Path("/kaggle/working/Phase3_Submissions")
DATA_FILE = BASE / "05_arch4_context/arch4_context_embeddings.npz"

CHECKPOINTS = {
    "original": BASE / "06_arch4_gated_context/gated_context_classifier_best.pt",
    "unweighted": BASE / "07_arch4_finetune_lr1e4/classifier_best.pt",
    "sqrt_weighted": BASE / "09_arch4_finetune_sqrt_weighted/classifier_sqrt_weighted_best.pt",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)
        self.context_gate = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.Sigmoid()
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)
        p = self.prev_proj(prev)
        c = self.curr_proj(curr)
        n = self.next_proj(nxt)
        gate = self.context_gate(torch.cat([p, c, n], dim=1))
        context = gate[:, :512] * p + gate[:, 512:] * n
        return self.fusion(c + context)


data = np.load(DATA_FILE)
X_dev = torch.from_numpy(data["X_dev"]).float()
y_dev = data["y_dev"]

print("Device:", DEVICE)
print("Development examples:", len(y_dev))
print("\nCONSISTENT CHECKPOINT EVALUATION")
print("-" * 65)

results = {}

for name, path in CHECKPOINTS.items():
    if not path.exists():
        print(f"{name}: checkpoint missing")
        continue

    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    state = ckpt["model_state_dict"]

    model = GatedContextClassifier(
        num_classes=ckpt.get("num_classes", 13)
    )
    model.load_state_dict(state, strict=True)
    model.to(DEVICE)
    model.eval()

    logits_parts = []

    with torch.no_grad():
        for start in range(0, len(X_dev), 256):
            xb = X_dev[start:start + 256].to(DEVICE)
            logits_parts.append(model(xb).cpu().numpy())

    logits = np.concatenate(logits_parts)
    pred = logits.argmax(axis=1)

    score = f1_score(
        y_dev, pred,
        labels=list(range(13)),
        average="macro",
        zero_division=0
    )

    results[name] = score

    print(f"\n{name}")
    print(f"  Recomputed Macro-F1: {score:.6f}")

    for key in [
        "base_validation_macro_f1",
        "validation_macro_f1",
        "best_dev_macro_f1",
        "initial_dev_macro_f1",
    ]:
        if key in ckpt:
            print(f"  Stored {key}: {ckpt[key]:.6f}")

    del model

print("\n" + "-" * 65)
if results:
    best = max(results, key=results.get)
    print(f"Best checkpoint under this evaluation: {best}")
    print(f"Macro-F1: {results[best]:.6f}")

print("\nNo checkpoints or embeddings modified.")

In [ ]:
import torch
import numpy as np
from pathlib import Path

BASE = Path("/kaggle/working/Phase3_Submissions")

CHECKPOINTS = {
    "original": BASE / "06_arch4_gated_context/gated_context_classifier_best.pt",
    "unweighted": BASE / "07_arch4_finetune_lr1e4/classifier_best.pt",
    "sqrt_weighted": BASE / "09_arch4_finetune_sqrt_weighted/classifier_sqrt_weighted_best.pt",
}

loaded = {}

for name, path in CHECKPOINTS.items():
    print("\n" + "=" * 65)
    print(name.upper())
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    state = ckpt.get("model_state_dict", ckpt)

    loaded[name] = state

    print("Checkpoint keys:", list(ckpt.keys()))

    for key in [
        "best_epoch",
        "base_validation_macro_f1",
        "bias_adjusted_validation_macro_f1",
        "validation_macro_f1",
        "best_dev_macro_f1",
    ]:
        if key in ckpt:
            print(f"{key}: {ckpt[key]}")

    print("Number of parameter tensors:", len(state))
    print("Total parameters:", sum(v.numel() for v in state.values()
                                  if isinstance(v, torch.Tensor)))

if "original" in loaded:
    original = loaded["original"]

    for other_name in ["unweighted", "sqrt_weighted"]:
        if other_name not in loaded:
            continue

        other = loaded[other_name]

        common = set(original) & set(other)
        if not common:
            print(f"\nNo matching state-dict keys: {other_name}")
            continue

        differences = []
        for key in sorted(common):
            a, b = original[key], other[key]

            if not isinstance(a, torch.Tensor) or not isinstance(b, torch.Tensor):
                continue
            if a.shape != b.shape:
                continue

            differences.append(
                (key, float((a.float() - b.float()).abs().max()))
            )

        changed = [(k, d) for k, d in differences if d > 0]

        print(f"\n{other_name.upper()} VS ORIGINAL")
        print("Matching tensors:", len(differences))
        print("Changed tensors:", len(changed))
        print("Largest absolute parameter difference:",
              max((d for _, d in differences), default=0.0))
        print("First changed tensors:", changed[:8])

print("\nCheckpoint comparison complete. No files modified.")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score

BASE = Path("/kaggle/working/Phase3_Submissions")
DATA_FILE = BASE / "05_arch4_context/arch4_context_embeddings.npz"

CHECKPOINTS = {
    "original": BASE / "06_arch4_gated_context/gated_context_classifier_best.pt",
    "unweighted": BASE / "07_arch4_finetune_lr1e4/classifier_best.pt",
    "sqrt_weighted": BASE / "09_arch4_finetune_sqrt_weighted/classifier_sqrt_weighted_best.pt",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)
        self.context_gate = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.Sigmoid()
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)
        p = self.prev_proj(prev)
        c = self.curr_proj(curr)
        n = self.next_proj(nxt)
        gate = self.context_gate(torch.cat([p, c, n], dim=1))
        context = gate[:, :512] * p + gate[:, 512:] * n
        return self.fusion(c + context)


data = np.load(DATA_FILE)
X_dev = torch.from_numpy(data["X_dev"]).float()
y_dev = data["y_dev"]

print("Device:", DEVICE)
print("Development examples:", len(y_dev))
print("\nCONSISTENT CHECKPOINT EVALUATION")
print("-" * 65)

results = {}

for name, path in CHECKPOINTS.items():
    if not path.exists():
        print(f"{name}: checkpoint missing")
        continue

    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    state = ckpt["model_state_dict"]

    model = GatedContextClassifier(
        num_classes=ckpt.get("num_classes", 13)
    )
    model.load_state_dict(state, strict=True)
    model.to(DEVICE)
    model.eval()

    logits_parts = []

    with torch.no_grad():
        for start in range(0, len(X_dev), 256):
            xb = X_dev[start:start + 256].to(DEVICE)
            logits_parts.append(model(xb).cpu().numpy())

    logits = np.concatenate(logits_parts)
    pred = logits.argmax(axis=1)

    score = f1_score(
        y_dev, pred,
        labels=list(range(13)),
        average="macro",
        zero_division=0
    )

    results[name] = score

    print(f"\n{name}")
    print(f"  Recomputed Macro-F1: {score:.6f}")

    for key in [
        "base_validation_macro_f1",
        "validation_macro_f1",
        "best_dev_macro_f1",
        "initial_dev_macro_f1",
    ]:
        if key in ckpt:
            print(f"  Stored {key}: {ckpt[key]:.6f}")

    del model

print("\n" + "-" * 65)
if results:
    best = max(results, key=results.get)
    print(f"Best checkpoint under this evaluation: {best}")
    print(f"Macro-F1: {results[best]:.6f}")

print("\nNo checkpoints or embeddings modified.")

In [ ]:
from pathlib import Path

ROOTS = [
    Path("/kaggle/working/Phase3_Submissions"),
    Path("/kaggle/working/Phase_3_RR"),
]

KEYWORDS = [
    "GatedContextClassifier",
    "base_validation_macro_f1",
    "bias_adjusted_validation_macro_f1",
    "best_dev_macro_f1",
    "validation_macro_f1",
]

print("SEARCHING ORIGINAL TRAINING CODE")
print("=" * 70)

matches = []

for root in ROOTS:
    if not root.exists():
        print(f"Directory not found: {root}")
        continue

    for path in root.rglob("*.py"):
        try:
            if path.stat().st_size > 5_000_000:
                continue

            content = path.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        except Exception:
            continue

        found = [kw for kw in KEYWORDS if kw in content]

        if found:
            matches.append((path, content, found))

for path, content, found in matches:
    print(f"\n{'=' * 70}")
    print("FILE:", path)
    print("MATCHES:", ", ".join(found))

    lines = content.splitlines()
    selected = set()

    for i, line in enumerate(lines):
        if any(kw in line for kw in found):
            selected.update(range(max(0, i - 5), min(len(lines), i + 9)))

    previous = -2
    for i in sorted(selected):
        if i > previous + 1:
            print("...")
        print(f"{i + 1:4d}: {lines[i]}")
        previous = i

if not matches:
    print("\nNo matching Python scripts found in the searched directories.")

print("\nSearch complete. No files modified.")

In [ ]:
import numpy as np
from pathlib import Path

BASE = Path("/kaggle/working/Phase3_Submissions")

FILES = [
    BASE / "05_arch4_context/arch4_context_embeddings.npz",
    BASE / "11_document_group_calibration/grouped_oof_logits.npz",
    BASE / "12_dev_evaluation/original_dev_logits.npz",
]

for path in FILES:
    print("\n" + "=" * 70)
    print("FILE:", path)
    print("EXISTS:", path.exists())

    if not path.exists():
        continue

    with np.load(path, allow_pickle=False) as data:
        for key in data.files:
            arr = data[key]

            print(
                f"\n{key}: shape={arr.shape}, dtype={arr.dtype}"
            )

            if arr.size == 0:
                print("  Empty array")
                continue

            if arr.ndim == 1:
                print("  Unique values:", len(np.unique(arr)))

                if np.issubdtype(arr.dtype, np.number):
                    print("  Min:", arr.min())
                    print("  Max:", arr.max())

                    if key.startswith("y_"):
                        values, counts = np.unique(
                            arr, return_counts=True
                        )
                        print(
                            "  Label counts:",
                            dict(zip(values.tolist(), counts.tolist()))
                        )

                elif arr.dtype.kind in ("U", "S"):
                    unique_ids = np.unique(arr)
                    print("  First 10 IDs:", unique_ids[:10].tolist())
                    print("  Last 10 IDs:", unique_ids[-10:].tolist())

            elif arr.ndim == 2:
                print("  All finite:", np.isfinite(arr).all())
                print("  Mean:", float(arr.mean()))
                print("  Std:", float(arr.std()))
                print("  Min:", float(arr.min()))
                print("  Max:", float(arr.max()))

                if arr.shape[1] == 13:
                    print(
                        "  Predicted class counts:",
                        dict(
                            zip(
                                *np.unique(
                                    arr.argmax(axis=1),
                                    return_counts=True
                                )
                            )
                        )
                    )

print("\nDiagnostic complete. No files modified.")

In [ ]:
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score

BASE = Path("/kaggle/working/Phase3_Submissions")

LOGITS_PATH = BASE / "12_dev_evaluation/original_dev_logits.npz"
CONFIG_PATH = BASE / "06_arch4_gated_context/prediction_config.json"

# Load saved development logits
with np.load(LOGITS_PATH, allow_pickle=False) as data:
    logits = data["logits"]
    y_dev = data["y_dev"]
    pred_saved = data["pred_raw"]

# Load calibration configuration
with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

# 1. Evaluate raw logits
pred_argmax = logits.argmax(axis=1)

print("=" * 60)
print("1. RAW LOGITS")
print("=" * 60)
print("Logits argmax equals saved pred_raw:",
      np.array_equal(pred_argmax, pred_saved))
print("Macro-F1 from logits:",
      round(f1_score(y_dev, pred_argmax, average="macro",
                     zero_division=0), 6))
print("Macro-F1 from pred_raw:",
      round(f1_score(y_dev, pred_saved, average="macro",
                     zero_division=0), 6))
print("Accuracy:",
      round(accuracy_score(y_dev, pred_argmax), 6))

# 2. Reconstruct class-bias vector
biases = config.get("class_biases", {})
label_to_id = config["label_to_id"]

bias_vector = np.zeros(logits.shape[1], dtype=np.float32)

for label, class_id in label_to_id.items():
    bias_vector[int(class_id)] = float(biases.get(label, 0.0))

# 3. Evaluate calibrated logits
pred_calibrated = (logits + bias_vector).argmax(axis=1)

print("\n" + "=" * 60)
print("2. CALIBRATED LOGITS")
print("=" * 60)
print("Class biases:", bias_vector.tolist())
print("Macro-F1:",
      round(f1_score(y_dev, pred_calibrated, average="macro",
                     zero_division=0), 6))
print("Accuracy:",
      round(accuracy_score(y_dev, pred_calibrated), 6))

# 4. Show stored metadata
print("\n" + "=" * 60)
print("3. STORED CHECKPOINT METADATA")
print("=" * 60)

for key in [
    "base_validation_macro_f1",
    "bias_adjusted_validation_macro_f1",
    "validation_accuracy_after_bias",
]:
    print(f"{key}: {config.get(key, 'Not present in config')}")

print("\nDiagnostic complete. No files modified.")

In [ ]:
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn

BASE = Path("/kaggle/working/Phase3_Submissions")

EMBED_PATH = BASE / "05_arch4_context/arch4_context_embeddings.npz"
LOGITS_PATH = BASE / "12_dev_evaluation/original_dev_logits.npz"
CKPT_PATH = BASE / "06_arch4_gated_context/gated_context_classifier_best.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load development embeddings and saved logits
with np.load(EMBED_PATH, allow_pickle=False) as d:
    X_dev = d["X_dev"]

with np.load(LOGITS_PATH, allow_pickle=False) as d:
    saved_logits = d["logits"]

# Reconstruct the classifier
class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)

        self.context_gate = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)

        p = self.prev_proj(prev)
        c = self.curr_proj(curr)
        n = self.next_proj(nxt)

        gate = self.context_gate(torch.cat([p, c, n], dim=1))
        context = gate[:, :512] * p + gate[:, 512:] * n

        return self.fusion(c + context)

# Load checkpoint
checkpoint = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False
)

model = GatedContextClassifier(num_classes=13)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(device)
model.eval()

# Recompute logits
with torch.no_grad():
    chunks = []
    batch_size = 256

    for start in range(0, len(X_dev), batch_size):
        batch = torch.from_numpy(
            X_dev[start:start + batch_size]
        ).float().to(device)

        chunks.append(model(batch).cpu().numpy())

recomputed_logits = np.concatenate(chunks, axis=0)

# Compare
difference = recomputed_logits - saved_logits

print("Saved logits shape:", saved_logits.shape)
print("Recomputed logits shape:", recomputed_logits.shape)
print("Maximum absolute difference:",
      float(np.abs(difference).max()))
print("Mean absolute difference:",
      float(np.abs(difference).mean()))
print("Correlation:",
      float(np.corrcoef(
          saved_logits.ravel(),
          recomputed_logits.ravel()
      )[0, 1]))

print("\nRecomputed logits match saved logits:",
      np.allclose(
          recomputed_logits,
          saved_logits,
          rtol=1e-4,
          atol=1e-4
      ))

In [ ]:
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, classification_report

BASE = Path("/kaggle/working/Phase3_Submissions")

LOGITS_PATH = BASE / "12_dev_evaluation/original_dev_logits.npz"
CONFIG_PATH = BASE / "06_arch4_gated_context/prediction_config.json"
OUT_DIR = BASE / "13_verified_submission"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load development logits and labels
with np.load(LOGITS_PATH, allow_pickle=False) as data:
    logits = data["logits"]
    y_dev = data["y_dev"]

# Load class mapping and calibrated biases
with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

label_to_id = config["label_to_id"]
class_biases = config.get("class_biases", {})

id_to_label = {int(idx): label for label, idx in label_to_id.items()}

bias_vector = np.zeros(logits.shape[1], dtype=np.float32)
for label, class_id in label_to_id.items():
    bias_vector[int(class_id)] = float(class_biases.get(label, 0.0))

# Apply verified calibration
calibrated_logits = logits + bias_vector
predictions = calibrated_logits.argmax(axis=1)

macro_f1 = f1_score(
    y_dev, predictions, average="macro", zero_division=0
)
accuracy = accuracy_score(y_dev, predictions)

print(f"Development Macro-F1: {macro_f1:.6f}")
print(f"Development Accuracy: {accuracy:.6f}")

print("\nClassification report:")
print(classification_report(
    y_dev,
    predictions,
    labels=list(range(len(id_to_label))),
    target_names=[id_to_label[i] for i in range(len(id_to_label))],
    zero_division=0,
    digits=4
))

# Save development predictions for verification only
result = {
    "development_macro_f1": float(macro_f1),
    "development_accuracy": float(accuracy),
    "num_examples": int(len(predictions)),
    "predictions": [id_to_label[int(p)] for p in predictions],
}

out_path = OUT_DIR / "calibrated_dev_predictions.json"
with open(out_path, "w") as f:
    json.dump(result, f, indent=2)

print(f"\nSaved development predictions to:\n{out_path}")
print("\nNote: This is NOT the hidden-test submission.")

In [ ]:
from pathlib import Path

BASE = Path("/kaggle/working/Phase3_Submissions")

print("=== Existing prediction artifacts ===")
for p in sorted(BASE.rglob("*")):
    if p.is_file() and (
        "prediction" in p.name.lower()
        or "submission" in p.name.lower()
    ):
        print(f"{p}  ({p.stat().st_size:,} bytes)")

print("\n=== Relevant Python scripts ===")
SEARCH_ROOTS = [
    Path("/kaggle/working/Phase3_Submissions"),
    Path("/kaggle/working/Phase_3_RR"),
]

keywords = [
    "CL_predictions.json",
    "IT_predictions.json",
    "prediction.json",
    "predictions_combined.json",
    "GatedContextClassifier",
]

seen = set()

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for p in root.rglob("*.py"):
        if p in seen:
            continue
        seen.add(p)

        try:
            text = p.read_text(errors="ignore")
        except Exception:
            continue

        matches = [k for k in keywords if k in text]
        if matches:
            print(f"\nFILE: {p}")
            print("Matches:", matches)

print("\nInspection complete. No files modified.")

In [ ]:
import json
import zipfile
from pathlib import Path
from collections import Counter

BASE = Path("/kaggle/working/Phase3_Submissions")

FILES = [
    BASE / "07_final_predictions/CL_predictions.json",
    BASE / "07_final_predictions/IT_predictions.json",
    BASE / "07_final_predictions/predictions_combined.json",
    BASE / "Arch4/prediction.json",
]

for path in FILES:
    print("\n" + "=" * 65)
    print("FILE:", path)
    print("EXISTS:", path.exists())

    if not path.exists():
        continue

    try:
        with open(path, "r") as f:
            obj = json.load(f)

        print("Top-level type:", type(obj).__name__)

        if isinstance(obj, dict):
            print("Number of keys:", len(obj))
            print("First 3 entries:")

            for key in list(obj.keys())[:3]:
                value = obj[key]
                preview = (
                    value[:5]
                    if isinstance(value, list)
                    else str(value)[:200]
                )
                print(f"  {key!r}: {preview!r}")

            values = list(obj.values())
            if values and all(isinstance(v, list) for v in values):
                labels = [
                    label
                    for value in values
                    for label in value
                    if isinstance(label, str)
                ]
                print("Total labels:", len(labels))
                print("Label counts:", Counter(labels))

        elif isinstance(obj, list):
            print("Length:", len(obj))
            print("First 3 records:", obj[:3])

    except Exception as e:
        print("Could not inspect JSON:", repr(e))

# Inspect existing submission ZIP without modifying it
zip_path = BASE / "Arch4/submission.zip"

print("\n" + "=" * 65)
print("SUBMISSION ZIP:", zip_path)

if zip_path.exists():
    with zipfile.ZipFile(zip_path, "r") as z:
        for info in z.infolist():
            print(f"\nEntry: {info.filename} ({info.file_size:,} bytes)")
            if info.file_size < 2_000_000:
                try:
                    content = z.read(info.filename).decode("utf-8")
                    print("Preview:", content[:500])
                except Exception:
                    print("Not readable as UTF-8 text")

# Find saved inference artifacts
print("\n" + "=" * 65)
print("POSSIBLE HIDDEN-TEST LOGITS / EMBEDDINGS")

for path in BASE.rglob("*"):
    if not path.is_file():
        continue

    name = path.name.lower()
    if any(term in name for term in [
        "test_logits", "hidden_logits", "test_embeddings",
        "hidden_embeddings", "test_logits.npz"
    ]):
        print(path)

print("\nInspection complete. No files modified.")

In [ ]:
from pathlib import Path
import torch
import json
import zipfile

BASE = Path("/kaggle/working/Phase3_Submissions")

print("=" * 65)
print("1. MODEL CHECKPOINTS")
print("=" * 65)

for p in sorted(BASE.rglob("*.pt")):
    print(f"{p} ({p.stat().st_size / 1024**2:.2f} MB)")

print("\n" + "=" * 65)
print("2. SUBMISSION DIRECTORY")
print("=" * 65)

for folder in [
    BASE / "06_arch4_gated_context",
    BASE / "07_final_predictions",
    BASE / "Arch4",
]:
    print(f"\n{folder}")
    if folder.exists():
        for p in sorted(folder.iterdir()):
            if p.is_file():
                print(f"  {p.name} ({p.stat().st_size:,} bytes)")

print("\n" + "=" * 65)
print("3. ARCH4 SUBMISSION DOCUMENT IDS")
print("=" * 65)

with open(BASE / "Arch4/prediction.json", "r") as f:
    predictions = json.load(f)

for doc_id, labels in predictions.items():
    print(f"{doc_id}: {len(labels)} sentences")

print("\nTotal documents:", len(predictions))
print("Total predictions:", sum(len(v) for v in predictions.values()))

print("\n" + "=" * 65)
print("4. CLASSIFIER CHECKPOINT METADATA")
print("=" * 65)

ckpt_path = (
    BASE / "06_arch4_gated_context/"
    "gated_context_classifier_best.pt"
)

ckpt = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=False
)

for key, value in ckpt.items():
    if key == "model_state_dict":
        print("model_state_dict tensors:", len(value))
        print("First 10 keys:", list(value.keys())[:10])
    elif isinstance(value, (int, float, str, bool, type(None))):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {type(value).__name__}")

print("\nInspection complete. No files modified.")

In [ ]:
from pathlib import Path
import os
import numpy as np
import torch

ROOT = Path("/kaggle/working/Phase3_Submissions")

print("=" * 65)
print("1. AVAILABLE TEST EMBEDDING FILES")
print("=" * 65)

for base in [Path("/kaggle/working"), Path("/kaggle/input")]:
    for p in base.rglob("*.npz"):
        try:
            data = np.load(p, allow_pickle=False)
            keys = list(data.files)
            relevant = [
                k for k in keys
                if any(s in k.lower() for s in
                       ["test", "hidden", "embed", "logit", "pred"])
            ]
            if relevant:
                print(f"\n{p}")
                for k in relevant:
                    print(f"  {k}: {data[k].shape} {data[k].dtype}")
        except Exception:
            pass

print("\n" + "=" * 65)
print("2. ARCH4 CHECKPOINTS AND PREDICTIONS")
print("=" * 65)

for base in [Path("/kaggle/working"), Path("/kaggle/input")]:
    for p in base.rglob("*"):
        if not p.is_file():
            continue

        name = p.name.lower()
        if (
            ("arch4" in name and
             (name.endswith(".pt") or name.endswith(".pth")))
            or name in {
                "prediction.json",
                "cl_predictions.json",
                "it_predictions.json",
            }
        ):
            print(p)

print("\n" + "=" * 65)
print("3. VARIABLES ALREADY LOADED IN MEMORY")
print("=" * 65)

for name, value in list(globals().items()):
    if name.startswith("_"):
        continue

    try:
        if isinstance(value, np.ndarray) and value.ndim >= 2:
            print(f"{name}: NumPy {value.shape} {value.dtype}")
        elif torch.is_tensor(value) and value.ndim >= 2:
            print(f"{name}: Tensor {tuple(value.shape)} {value.dtype}")
    except Exception:
        pass

print("\nDiagnostic complete.")

In [ ]:
from pathlib import Path
import json
import numpy as np
import torch

ROOT = Path("/kaggle/working/Phase3_Submissions")

# 1. Inspect the existing 858-sentence arrays
print("=== Existing embedding arrays ===")

for name in ["prev", "curr", "nxt"]:
    value = globals().get(name)

    if value is not None:
        arr = np.asarray(value)
        print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")

# 2. Inspect existing hidden-test predictions
pred_path = ROOT / "Arch4" / "prediction.json"

with open(pred_path, "r", encoding="utf-8") as f:
    predictions = json.load(f)

print("\n=== Existing hidden-test predictions ===")
print("Documents:", len(predictions))
print("Total sentences:", sum(len(v) for v in predictions.values()))

for doc_id, labels in predictions.items():
    print(f"{doc_id}: {len(labels)} sentences")

# 3. Inspect available source data
print("\n=== Available dataset JSON files ===")

for base in [Path("/kaggle/working"), Path("/kaggle/input")]:
    for path in base.rglob("*.json"):
        if any(term in path.name.lower()
               for term in ["test", "hidden", "cl_", "it_"]):
            print(path)

# 4. Inspect the original Arch4 model files
print("\n=== Available model files ===")

for path in ROOT.rglob("*.pt"):
    print(f"{path} ({path.stat().st_size / 1024**2:.2f} MB)")

# 5. Inspect the embedding-generation variables already in memory
print("\n=== Candidate model/tokenizer variables ===")

for name in [
    "model", "backbone", "arch4_model", "tokenizer",
    "input_ids", "attention_mask", "features"
]:
    if name in globals():
        obj = globals()[name]
        print(name, type(obj).__name__)
        if torch.is_tensor(obj):
            print("  shape:", tuple(obj.shape))

In [ ]:
from pathlib import Path
import json
import torch

ROOT = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")

# 1. Inspect hidden-test JSON structure
for filename in ["CL_test.json", "IT_test.json"]:
    path = ROOT / "Test" / filename

    print(f"\n{'=' * 60}")
    print(filename)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Container type:", type(data).__name__)

    if isinstance(data, dict):
        print("Document count:", len(data))
        for doc_id, doc in list(data.items())[:2]:
            print("\nDocument ID:", doc_id)
            print("Record type:", type(doc).__name__)

            if isinstance(doc, dict):
                print("Keys:", list(doc.keys()))
                for k, v in doc.items():
                    if isinstance(v, list):
                        print(f"{k}: list, length={len(v)}")
                        if v:
                            print("First item:", repr(v[0])[:250])
                    elif isinstance(v, str):
                        print(f"{k}: {v[:250]!r}")
            elif isinstance(doc, list):
                print("Length:", len(doc))
                print("First item:", repr(doc[0])[:250] if doc else None)

    elif isinstance(data, list):
        print("Record count:", len(data))
        print("First record:", repr(data[0])[:1000] if data else None)


# 2. Search likely Kaggle model mount locations
print(f"\n{'=' * 60}")
print("ARCH4 BACKBONE CHECKPOINTS")

search_roots = [
    Path("/kaggle/input/models"),
    Path("/kaggle/input/datasets"),
    Path("/kaggle/working"),
]

seen = set()

for base in search_roots:
    if not base.exists():
        continue

    for pattern in ["*.pt", "*.pth", "*.ckpt"]:
        for p in base.rglob(pattern):
            key = str(p)
            if key in seen:
                continue
            seen.add(key)

            name = p.name.lower()
            # Print likely pretrained language-model checkpoints
            if (
                "arch4" in str(p).lower()
                or "latest" in name
                or "best" in name
                or "model" in name
            ):
                print(p, f"({p.stat().st_size / 1024**2:.1f} MB)")


# 3. Inspect potentially useful functions already defined
print(f"\n{'=' * 60}")
print("EXISTING INFERENCE FUNCTIONS")

for name, obj in list(globals().items()):
    if callable(obj) and not name.startswith("_"):
        if any(term in name.lower() for term in [
            "embed", "backbone", "arch4", "encode",
            "predict", "load_model", "build_model"
        ]):
            print(name, getattr(obj, "__module__", ""))

print("\nDiagnostic complete. No files were modified.")

In [ ]:
from pathlib import Path
import torch
import inspect

ROOTS = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

print("=" * 65)
print("1. ARCH4 CHECKPOINTS")
print("=" * 65)

for root in ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        name = path.name.lower()
        if any(k in name for k in [
            "arch4", "latest.pt", "best.pt", "model.pt"
        ]):
            if path.suffix in [".pt", ".pth", ".py", ".json"]:
                print(path)

print("\n" + "=" * 65)
print("2. MODEL VARIABLES")
print("=" * 65)

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if isinstance(obj, torch.nn.Module):
        print(f"\n{name}: {type(obj).__name__}")
        print("Parameters:", sum(p.numel() for p in obj.parameters()))

        for attr in ["config", "args", "model", "transformer", "blocks"]:
            if hasattr(obj, attr):
                value = getattr(obj, attr)
                print(f"  {attr}: {type(value).__name__}")

print("\n" + "=" * 65)
print("3. MODEL-RELATED FUNCTIONS")
print("=" * 65)

for name, obj in list(globals().items()):
    if callable(obj) and not name.startswith("_"):
        if any(k in name.lower() for k in [
            "build_arch", "build_backbone", "load_arch",
            "load_backbone", "get_hidden", "extract_hidden",
            "encode_sentence", "embed_sentence"
        ]):
            try:
                print(f"\n{name}{inspect.signature(obj)}")
                print(inspect.getsource(obj)[:1800])
            except Exception:
                print(name, "(source unavailable)")

print("\nDiagnostic complete.")

In [ ]:
# ==============================================================================
# SELF-CONTAINED FIX & UPGRADE CELL:
# 1. Separates `arch4_backbone` from `clf_model` (fixes `return_hidden` TypeError)
# 2. Caches CL_test & IT_test embeddings (2,197 sentences) to disk for instant reuse
# 3. Runs Document-Disjoint OOF Calibration + Markov Viterbi Decoding on 06 & 09
# 4. Exports verified Codabench `/kaggle/working/Phase3_Submissions/Arch4/submission.zip`
# ==============================================================================
import os
import json
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import Counter, OrderedDict
from tokenizers import Tokenizer
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import GroupKFold
from scipy.special import log_softmax
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_DIR = Path("/kaggle/working/Phase3_Submissions")
DATA_ROOT = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")
ARCH4_CKPT_PATH = Path(
    "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/"
    "transformers/default/1/arch4_hybrid_gqa_201m/best.pt"
)
TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
)

# ------------------------------------------------------------------------------
# 1. Load Official Label Vocab & Train/Dev/Test JSONs
# ------------------------------------------------------------------------------
with open(DATA_ROOT / "Train/label_vocab.json", "r", encoding="utf-8") as f:
    label_vocab = {str(k): int(v) for k, v in json.load(f).items()}
id_to_label = {v: k for k, v in label_vocab.items()}
NUM_CLASSES = len(label_vocab)

with open(DATA_ROOT / "Train/CL_train.json", "r", encoding="utf-8") as f:
    CL_train = json.load(f)
with open(DATA_ROOT / "Train/IT_train.json", "r", encoding="utf-8") as f:
    IT_train = json.load(f)
with open(DATA_ROOT / "Train/CL_dev.json", "r", encoding="utf-8") as f:
    CL_dev = json.load(f)
with open(DATA_ROOT / "Train/IT_dev.json", "r", encoding="utf-8") as f:
    IT_dev = json.load(f)
with open(DATA_ROOT / "Test/CL_test.json", "r", encoding="utf-8") as f:
    CL_test = json.load(f)
with open(DATA_ROOT / "Test/IT_test.json", "r", encoding="utf-8") as f:
    IT_test = json.load(f)

# ------------------------------------------------------------------------------
# 2. Extract or Load Cached Test Context Embeddings (Fixes `model` Collision)
# ------------------------------------------------------------------------------
TEST_CACHE_PATH = BASE_DIR / "05_arch4_context/arch4_test_context_embeddings.npz"

def get_or_create_test_embeddings():
    test_docs_ordered = OrderedDict()
    for doc_id, sents in CL_test.items():
        test_docs_ordered[str(doc_id)] = ("CL", sents)
    for doc_id, sents in IT_test.items():
        test_docs_ordered[str(doc_id)] = ("IT", sents)

    flat_sentences = []
    doc_ranges = OrderedDict()
    test_rel_pos = []
    for doc_id, (dom, sents) in test_docs_ordered.items():
        start = len(flat_sentences)
        L = len(sents)
        for idx, s in enumerate(sents):
            flat_sentences.append(str(s))
            test_rel_pos.append(idx / max(L - 1, 1))
        doc_ranges[doc_id] = (start, len(flat_sentences))

    if TEST_CACHE_PATH.exists():
        print(f"Loading cached test context embeddings from: {TEST_CACHE_PATH}")
        cached = np.load(TEST_CACHE_PATH)
        return cached["X_test_ctx"], doc_ranges, np.array(test_rel_pos, dtype=np.float32)

    print("Encoding 2,197 test sentences with Arch4 backbone (will cache to disk)...")
    tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
    ckpt_arch4 = torch.load(ARCH4_CKPT_PATH, map_location="cpu", weights_only=False)
    arch4_backbone = Arch4(ckpt_arch4["cfg"])
    arch4_backbone.load_state_dict(ckpt_arch4["model"], strict=True)
    arch4_backbone.to(DEVICE).eval()

    all_embs = []
    batch_size = 16
    with torch.no_grad():
        for start in tqdm(range(0, len(flat_sentences), batch_size), desc="Arch4 Test Encoding"):
            batch_sents = flat_sentences[start:start + batch_size]
            encoded = [[1] + tokenizer.encode(s).ids[:254] + [2] for s in batch_sents]
            max_len = max(len(ids) for ids in encoded)
            input_ids = torch.zeros((len(encoded), max_len), dtype=torch.long, device=DEVICE)
            eos_pos = []
            for i, ids in enumerate(encoded):
                input_ids[i, :len(ids)] = torch.tensor(ids, dtype=torch.long, device=DEVICE)
                eos_pos.append(len(ids) - 1)
            hidden = arch4_backbone(input_ids, return_hidden=True)
            rows = torch.arange(len(encoded), device=DEVICE)
            pos = torch.tensor(eos_pos, device=DEVICE)
            all_embs.append(hidden[rows, pos].float().cpu().numpy())

    del arch4_backbone, ckpt_arch4
    torch.cuda.empty_cache()

    sent_embs = np.concatenate(all_embs, axis=0)
    prev_embs = np.zeros_like(sent_embs)
    next_embs = np.zeros_like(sent_embs)
    for doc_id, (start, end) in doc_ranges.items():
        for idx in range(start, end):
            prev_embs[idx] = sent_embs[max(start, idx - 1)]
            next_embs[idx] = sent_embs[min(end - 1, idx + 1)]

    X_test_ctx = np.concatenate([prev_embs, sent_embs, next_embs], axis=1).astype(np.float32)
    np.savez_compressed(TEST_CACHE_PATH, X_test_ctx=X_test_ctx)
    print(f"Cached test context embeddings: {X_test_ctx.shape} -> {TEST_CACHE_PATH}")
    return X_test_ctx, doc_ranges, np.array(test_rel_pos, dtype=np.float32)

X_test_ctx, test_doc_ranges, test_rel_pos = get_or_create_test_embeddings()

# ------------------------------------------------------------------------------
# 3. Load Train/Dev Context Embeddings & Exact Document Metadata
# ------------------------------------------------------------------------------
ctx_data = np.load(BASE_DIR / "05_arch4_context/arch4_context_embeddings.npz")
X_train_ctx = ctx_data["X_train"].astype(np.float32)
y_train = ctx_data["y_train"].astype(np.int64)
X_dev_ctx = ctx_data["X_dev"].astype(np.float32)
y_dev = ctx_data["y_dev"].astype(np.int64)

def build_doc_arrays(docs):
    doc_ids, rel_pos = [], []
    for doc in docs:
        L = len(doc["text"])
        doc_ids.extend([str(doc["id"])] * L)
        rel_pos.extend([i / max(L - 1, 1) for i in range(L)])
    return np.array(doc_ids), np.array(rel_pos, dtype=np.float32)

train_doc_ids, train_rel_pos = build_doc_arrays(CL_train + IT_train)
dev_doc_ids, dev_rel_pos = build_doc_arrays(CL_dev + IT_dev)

# ------------------------------------------------------------------------------
# 4. Build Legal Discourse Transition & Positional Priors from 80 Train Docs
# ------------------------------------------------------------------------------
def build_transition_and_position_priors(y_seq, doc_ids, rel_pos, num_bins=10):
    trans_counts = np.ones((NUM_CLASSES, NUM_CLASSES), dtype=np.float64) * 0.5
    pos_counts = np.ones((num_bins, NUM_CLASSES), dtype=np.float64) * 0.5
    for i in range(len(y_seq)):
        b = min(int(rel_pos[i] * num_bins), num_bins - 1)
        pos_counts[b, y_seq[i]] += 1.0
        if i > 0 and doc_ids[i] == doc_ids[i - 1]:
            trans_counts[y_seq[i - 1], y_seq[i]] += 1.0
    log_trans = np.log(trans_counts / trans_counts.sum(axis=1, keepdims=True))
    log_pos = np.log(pos_counts / pos_counts.sum(axis=1, keepdims=True))
    return log_trans, log_pos

log_trans_mat, log_pos_mat = build_transition_and_position_priors(y_train, train_doc_ids, train_rel_pos)

def viterbi_decode_spans(logits, doc_spans, rel_pos, eta_trans=0.12, eta_pos=0.06):
    log_probs = log_softmax(logits, axis=-1)
    preds = np.zeros(len(logits), dtype=np.int64)
    num_bins = log_pos_mat.shape[0]
    for start, end in doc_spans:
        L = end - start
        dp = np.zeros((L, NUM_CLASSES), dtype=np.float64)
        backptr = np.zeros((L, NUM_CLASSES), dtype=np.int64)
        b0 = min(int(rel_pos[start] * num_bins), num_bins - 1)
        dp[0] = log_probs[start] + eta_pos * log_pos_mat[b0]
        for t in range(1, L):
            idx = start + t
            bt = min(int(rel_pos[idx] * num_bins), num_bins - 1)
            emission = log_probs[idx] + eta_pos * log_pos_mat[bt]
            scores = dp[t - 1][:, None] + eta_trans * log_trans_mat + emission[None, :]
            backptr[t] = np.argmax(scores, axis=0)
            dp[t] = np.max(scores, axis=0)
        path = np.zeros(L, dtype=np.int64)
        path[-1] = np.argmax(dp[-1])
        for t in range(L - 2, -1, -1):
            path[t] = backptr[t + 1, path[t + 1]]
        preds[start:end] = path
    return preds

# ------------------------------------------------------------------------------
# 5. Load Classifier Checkpoints & Compare Under Both 11-Class & 13-Class F1
# ------------------------------------------------------------------------------
class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)
        self.context_gate = nn.Sequential(nn.Linear(1536, 1024), nn.Sigmoid())
        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)
        p, c, n = self.prev_proj(prev), self.curr_proj(curr), self.next_proj(nxt)
        gate = self.context_gate(torch.cat([p, c, n], dim=1))
        return self.fusion(c + gate[:, :512] * p + gate[:, 512:] * n)

@torch.no_grad()
def get_ckpt_logits(ckpt_path, X_np):
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    clf = GatedContextClassifier(num_classes=13).to(DEVICE)
    clf.load_state_dict(ckpt.get("model_state_dict", ckpt), strict=True)
    clf.eval()
    parts = []
    for i in range(0, len(X_np), 512):
        xb = torch.from_numpy(X_np[i:i+512]).float().to(DEVICE)
        parts.append(clf(xb).cpu().numpy())
    return np.concatenate(parts, axis=0)

CKPT_06 = BASE_DIR / "06_arch4_gated_context/gated_context_classifier_best.pt"
CKPT_07 = BASE_DIR / "07_arch4_finetune_lr1e4/classifier_best.pt"
CKPT_09 = BASE_DIR / "09_arch4_finetune_sqrt_weighted/classifier_sqrt_weighted_best.pt"

logits_06_dev = get_ckpt_logits(CKPT_06, X_dev_ctx)
logits_07_dev = get_ckpt_logits(CKPT_07, X_dev_ctx)
logits_09_dev = get_ckpt_logits(CKPT_09, X_dev_ctx)
logits_ens_dev = 0.50 * logits_06_dev + 0.50 * logits_09_dev

# Reconstruct ordered dev document spans
dev_spans = []
cursor = 0
for doc in CL_dev + IT_dev:
    L = len(doc["text"])
    dev_spans.append((cursor, cursor + L))
    cursor += L

def fit_regularized_bias(logits_cal, y_cal, l2_reg=0.04, bound=0.40, passes=3):
    bias = np.zeros(NUM_CLASSES, dtype=np.float64)
    grid = np.linspace(-bound, bound, 41)
    present_classes = np.unique(y_cal)
    
    def score_fn(b):
        preds = (logits_cal + b).argmax(axis=1)
        # Evaluate only on classes present in calibration split so zero-support classes aren't distorted
        f1 = f1_score(y_cal, preds, labels=present_classes, average="macro", zero_division=0)
        return f1 - l2_reg * np.sum(b ** 2)

    best = score_fn(bias)
    for _ in range(passes):
        for c in present_classes:
            best_c = bias[c]
            for val in grid:
                trial = bias.copy()
                trial[c] = val
                s = score_fn(trial)
                if s > best + 1e-6:
                    best = s
                    best_c = val
            bias[c] = best_c
    return bias

print("\n" + "=" * 78)
print("STRICT DOCUMENT-DISJOINT 5-FOLD OOF EVALUATION ON DEV (10 DOCS, 2,142 SENTS)")
print("=" * 78)

gkf = GroupKFold(n_splits=5)
candidates = {
    "06_original": logits_06_dev,
    "07_unweighted": logits_07_dev,
    "09_sqrt_weighted": logits_09_dev,
    "Ensemble_06_09": logits_ens_dev,
}

oof_summary = {}
for name, L_dev in candidates.items():
    raw_11 = f1_score(y_dev, L_dev.argmax(1), average="macro", zero_division=0)
    raw_13 = f1_score(y_dev, L_dev.argmax(1), labels=list(range(13)), average="macro", zero_division=0)
    
    oof_cal_preds = np.zeros_like(y_dev)
    oof_vit_preds = np.zeros_like(y_dev)
    fold_biases = []
    
    for cal_idx, val_idx in gkf.split(L_dev, y_dev, groups=dev_doc_ids):
        b_fold = fit_regularized_bias(L_dev[cal_idx], y_dev[cal_idx], l2_reg=0.04, bound=0.40)
        fold_biases.append(b_fold)
        shifted_val = L_dev[val_idx] + b_fold
        oof_cal_preds[val_idx] = shifted_val.argmax(axis=1)
        
        # Build spans for held-out validation documents
        val_spans = []
        v_docs = dev_doc_ids[val_idx]
        start_i = 0
        for i in range(1, len(v_docs) + 1):
            if i == len(v_docs) or v_docs[i] != v_docs[start_i]:
                val_spans.append((start_i, i))
                start_i = i
        oof_vit_preds[val_idx] = viterbi_decode_spans(
            shifted_val, val_spans, dev_rel_pos[val_idx], eta_trans=0.10, eta_pos=0.05
        )
        
    oof_cal_11 = f1_score(y_dev, oof_cal_preds, average="macro", zero_division=0)
    oof_cal_13 = f1_score(y_dev, oof_cal_preds, labels=list(range(13)), average="macro", zero_division=0)
    oof_vit_11 = f1_score(y_dev, oof_vit_preds, average="macro", zero_division=0)
    oof_vit_13 = f1_score(y_dev, oof_vit_preds, labels=list(range(13)), average="macro", zero_division=0)
    
    mean_bias = np.mean(fold_biases, axis=0)
    oof_summary[name] = {
        "mean_bias": mean_bias,
        "oof_vit_13": oof_vit_13,
        "oof_cal_13": oof_cal_13,
    }
    print(f"\n[{name}]")
    print(f"  Raw Logits            -> 11-Class F1: {raw_11:.6f} | 13-Class F1: {raw_13:.6f}")
    print(f"  Disjoint OOF Reg-Bias -> 11-Class F1: {oof_cal_11:.6f} | 13-Class F1: {oof_cal_13:.6f}")
    print(f"  Disjoint OOF + Viterbi-> 11-Class F1: {oof_vit_11:.6f} | 13-Class F1: {oof_vit_13:.6f}")

# ------------------------------------------------------------------------------
# 6. Generate Test Predictions Using the Best Generalizing Pipeline (Ensemble 06 + 09)
# ------------------------------------------------------------------------------
logits_06_test = get_ckpt_logits(CKPT_06, X_test_ctx)
logits_09_test = get_ckpt_logits(CKPT_09, X_test_ctx)
logits_ens_test = 0.50 * logits_06_test + 0.50 * logits_09_test

best_oof_bias = oof_summary["Ensemble_06_09"]["mean_bias"]
use_viterbi = oof_summary["Ensemble_06_09"]["oof_vit_13"] >= oof_summary["Ensemble_06_09"]["oof_cal_13"]

shifted_test_logits = logits_ens_test + best_oof_bias[None, :]
test_spans = list(test_doc_ranges.values())

if use_viterbi:
    print("\nApplying OOF-Calibrated Bias + Document Viterbi Decoding to Test Set...")
    test_pred_ids = viterbi_decode_spans(
        shifted_test_logits, test_spans, test_rel_pos, eta_trans=0.10, eta_pos=0.05
    )
else:
    print("\nApplying OOF-Calibrated Bias (Argmax) to Test Set...")
    test_pred_ids = shifted_test_logits.argmax(axis=1)

# Package predictions into exact Codabench format
predictions = OrderedDict()
for doc_id, (start, end) in test_doc_ranges.items():
    predictions[doc_id] = [id_to_label[int(i)] for i in test_pred_ids[start:end]]

# Strict Validation Against CL_test and IT_test
expected_test = {**CL_test, **IT_test}
assert len(predictions) == 10, f"Expected 10 docs, got {len(predictions)}"
for doc_id, sents in expected_test.items():
    assert doc_id in predictions, f"Missing {doc_id}"
    assert len(predictions[doc_id]) == len(sents), f"Length mismatch on {doc_id}"
    assert all(lbl in label_vocab for lbl in predictions[doc_id]), f"Invalid label in {doc_id}"

ARCH_OUT_DIR = BASE_DIR / "Arch4"
ARCH_OUT_DIR.mkdir(parents=True, exist_ok=True)
pred_json_path = ARCH_OUT_DIR / "prediction.json"
sub_zip_path = ARCH_OUT_DIR / "submission.zip"

with open(pred_json_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_json_path, arcname="prediction.json")

print("\n" + "=" * 78)
print("NEW VERIFIED SUBMISSION READY")
print("=" * 78)
print("Total test sentences predicted:", len(test_pred_ids))
print("Predicted test label distribution:", Counter([id_to_label[int(i)] for i in test_pred_ids]))
print("Saved prediction.json:", pred_json_path)
print("Saved submission.zip :", sub_zip_path)

In [ ]:
# ============================================================
# PHASE 3 RR — RAW ENSEMBLE SUBMISSION
# Checkpoint 06 + Checkpoint 09
# No class-bias calibration; no Viterbi decoding
# Uses cached hidden-test embeddings
# ============================================================

import json
import zipfile
import numpy as np
import torch
import torch.nn as nn

from pathlib import Path
from collections import OrderedDict, Counter

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

BASE_DIR = Path("/kaggle/working/Phase3_Submissions")
DATA_ROOT = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")

CACHE_PATH = (
    BASE_DIR / "05_arch4_context"
    / "arch4_test_context_embeddings.npz"
)

CKPT_06 = (
    BASE_DIR / "06_arch4_gated_context"
    / "gated_context_classifier_best.pt"
)

CKPT_09 = (
    BASE_DIR / "09_arch4_finetune_sqrt_weighted"
    / "classifier_sqrt_weighted_best.pt"
)

LABEL_PATH = DATA_ROOT / "Train" / "label_vocab.json"

OUTPUT_DIR = BASE_DIR / "Arch4_RawEnsemble"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_PATH = OUTPUT_DIR / "prediction.json"
SUBMISSION_PATH = OUTPUT_DIR / "submission.zip"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

for path in [CACHE_PATH, CKPT_06, CKPT_09, LABEL_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

# ------------------------------------------------------------
# 2. LOAD TEST DOCUMENTS IN THE ORIGINAL ORDER
# ------------------------------------------------------------

with open(DATA_ROOT / "Test" / "CL_test.json", "r",
          encoding="utf-8") as f:
    CL_test = json.load(f)

with open(DATA_ROOT / "Test" / "IT_test.json", "r",
          encoding="utf-8") as f:
    IT_test = json.load(f)

test_docs = OrderedDict()

for doc_id, sentences in CL_test.items():
    test_docs[str(doc_id)] = sentences

for doc_id, sentences in IT_test.items():
    test_docs[str(doc_id)] = sentences

# Build sentence boundaries matching the cached embedding order.
doc_ranges = OrderedDict()
cursor = 0

for doc_id, sentences in test_docs.items():
    start = cursor
    cursor += len(sentences)
    doc_ranges[doc_id] = (start, cursor)

# ------------------------------------------------------------
# 3. LOAD CACHED CONTEXT EMBEDDINGS
# ------------------------------------------------------------

with np.load(CACHE_PATH) as cached:
    if "X_test_ctx" not in cached.files:
        raise KeyError(
            f"X_test_ctx not found. Available arrays: {cached.files}"
        )

    X_test = cached["X_test_ctx"].astype(np.float32)

expected_sentences = sum(
    len(sentences) for sentences in test_docs.values()
)

assert X_test.shape == (expected_sentences, 3072), (
    f"Expected ({expected_sentences}, 3072), got {X_test.shape}"
)

assert expected_sentences == 2197
assert len(test_docs) == 10

print("Loaded cached test embeddings:", X_test.shape)
print("Documents:", len(test_docs))
print("Device:", DEVICE)

# ------------------------------------------------------------
# 4. DEFINE CLASSIFIER ARCHITECTURE
# ------------------------------------------------------------

class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()

        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)

        self.context_gate = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)

        p = self.prev_proj(prev)
        c = self.curr_proj(curr)
        n = self.next_proj(nxt)

        gate = self.context_gate(
            torch.cat([p, c, n], dim=1)
        )

        context = (
            gate[:, :512] * p
            + gate[:, 512:] * n
        )

        return self.fusion(c + context)

# ------------------------------------------------------------
# 5. LOAD CHECKPOINT AND GENERATE LOGITS
# ------------------------------------------------------------

@torch.inference_mode()
def predict_logits(checkpoint_path, X, batch_size=512):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    clf = GatedContextClassifier(num_classes=13).to(DEVICE)

    state = checkpoint.get(
        "model_state_dict",
        checkpoint.get("state_dict", checkpoint)
    )

    clf.load_state_dict(state, strict=True)
    clf.eval()

    parts = []

    for start in range(0, len(X), batch_size):
        xb = torch.from_numpy(
            X[start:start + batch_size]
        ).float().to(DEVICE)

        parts.append(clf(xb).float().cpu().numpy())

    return np.concatenate(parts, axis=0)


print("\nGenerating checkpoint 06 logits...")
logits_06 = predict_logits(CKPT_06, X_test)

print("Generating checkpoint 09 logits...")
logits_09 = predict_logits(CKPT_09, X_test)

assert logits_06.shape == (2197, 13)
assert logits_09.shape == (2197, 13)

# Equal-weight raw-logit ensemble.
logits_ensemble = 0.50 * logits_06 + 0.50 * logits_09

# IMPORTANT: Do not add the class-bias vector here.
# IMPORTANT: Do not apply Viterbi decoding here.
predicted_ids = logits_ensemble.argmax(axis=1)

# ------------------------------------------------------------
# 6. LOAD OFFICIAL LABEL VOCABULARY
# ------------------------------------------------------------

with open(LABEL_PATH, "r", encoding="utf-8") as f:
    label_vocab = json.load(f)

label_vocab = {
    str(label): int(idx)
    for label, idx in label_vocab.items()
}

id_to_label = {
    idx: label
    for label, idx in label_vocab.items()
}

assert len(id_to_label) == 13

# ------------------------------------------------------------
# 7. RESTORE DOCUMENT STRUCTURE
# ------------------------------------------------------------

predictions = OrderedDict()

for doc_id, (start, end) in doc_ranges.items():
    predictions[doc_id] = [
        id_to_label[int(i)]
        for i in predicted_ids[start:end]
    ]

# ------------------------------------------------------------
# 8. STRICT VALIDATION
# ------------------------------------------------------------

allowed_labels = set(label_vocab.keys())

assert len(predictions) == 10
assert sum(len(v) for v in predictions.values()) == 2197

for doc_id, sentences in test_docs.items():
    assert doc_id in predictions, f"Missing document: {doc_id}"

    assert len(predictions[doc_id]) == len(sentences), (
        f"Sentence-count mismatch: {doc_id}"
    )

    invalid = set(predictions[doc_id]) - allowed_labels

    assert not invalid, (
        f"Invalid labels in {doc_id}: {invalid}"
    )

# ------------------------------------------------------------
# 9. SAVE A NEW SUBMISSION WITHOUT OVERWRITING THE OLD ONE
# ------------------------------------------------------------

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

with zipfile.ZipFile(
    SUBMISSION_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:
    zf.write(
        PREDICTION_PATH,
        arcname="prediction.json"
    )

# Verify the actual archive.
with zipfile.ZipFile(SUBMISSION_PATH, "r") as zf:
    assert zf.namelist() == ["prediction.json"]

    with zf.open("prediction.json") as f:
        verified = json.load(f)

assert verified == predictions

# ------------------------------------------------------------
# 10. REPORT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RAW ENSEMBLE SUBMISSION CREATED")
print("=" * 60)

print("Documents:", len(predictions))
print("Sentence predictions:", sum(len(v) for v in predictions.values()))

print(
    "Label distribution:",
    dict(Counter(
        label
        for labels in predictions.values()
        for label in labels
    ))
)

print("\nPrediction JSON:", PREDICTION_PATH)
print("Submission ZIP:", SUBMISSION_PATH)
print("Validation: PASS")

In [ ]:
# ============================================================
# PHASE 3 RR — DEVELOPMENT-SELECTED ENSEMBLE WEIGHT
# Tests checkpoints 06 and 09 using held-out dev labels.
# Generates a separate submission.zip.
# ============================================================

import json
import zipfile
import numpy as np
import torch
import torch.nn as nn

from pathlib import Path
from collections import OrderedDict, Counter
from sklearn.metrics import f1_score

BASE = Path("/kaggle/working/Phase3_Submissions")
DATA = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DEV_CACHE = BASE / "05_arch4_context/arch4_context_embeddings.npz"
TEST_CACHE = BASE / "05_arch4_context/arch4_test_context_embeddings.npz"

CKPT_06 = BASE / "06_arch4_gated_context/gated_context_classifier_best.pt"
CKPT_09 = BASE / "09_arch4_finetune_sqrt_weighted/classifier_sqrt_weighted_best.pt"

LABEL_PATH = DATA / "Train/label_vocab.json"

for p in [DEV_CACHE, TEST_CACHE, CKPT_06, CKPT_09, LABEL_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required file not found: {p}")

# ------------------------------------------------------------
# 1. LOAD DEVELOPMENT AND TEST EMBEDDINGS
# ------------------------------------------------------------

with np.load(DEV_CACHE) as data:
    X_dev = data["X_dev"].astype(np.float32)
    y_dev = data["y_dev"].astype(np.int64)

with np.load(TEST_CACHE) as data:
    X_test = data["X_test_ctx"].astype(np.float32)

with open(LABEL_PATH, "r", encoding="utf-8") as f:
    label_vocab = {str(k): int(v) for k, v in json.load(f).items()}

id_to_label = {v: k for k, v in label_vocab.items()}
NUM_CLASSES = len(label_vocab)

assert X_dev.shape == (len(y_dev), 3072)
assert X_test.shape == (2197, 3072)
assert NUM_CLASSES == 13

print("Dev embeddings:", X_dev.shape)
print("Dev labels:", y_dev.shape)
print("Test embeddings:", X_test.shape)

# ------------------------------------------------------------
# 2. CLASSIFIER ARCHITECTURE
# ------------------------------------------------------------

class GatedContextClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.prev_proj = nn.Linear(1024, 512)
        self.curr_proj = nn.Linear(1024, 512)
        self.next_proj = nn.Linear(1024, 512)
        self.context_gate = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.Sigmoid()
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        prev, curr, nxt = x.split(1024, dim=1)
        p = self.prev_proj(prev)
        c = self.curr_proj(curr)
        n = self.next_proj(nxt)
        gate = self.context_gate(torch.cat([p, c, n], dim=1))
        context = gate[:, :512] * p + gate[:, 512:] * n
        return self.fusion(c + context)


@torch.inference_mode()
def get_logits(checkpoint_path, X, batch_size=512):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    clf = GatedContextClassifier(NUM_CLASSES).to(DEVICE)
    state = checkpoint.get(
        "model_state_dict",
        checkpoint.get("state_dict", checkpoint)
    )
    clf.load_state_dict(state, strict=True)
    clf.eval()

    parts = []
    for start in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[start:start + batch_size]).to(DEVICE)
        parts.append(clf(xb).float().cpu().numpy())

    return np.concatenate(parts, axis=0)


# ------------------------------------------------------------
# 3. COMPUTE DEVELOPMENT LOGITS
# ------------------------------------------------------------

print("\nCalculating checkpoint 06 development logits...")
dev_06 = get_logits(CKPT_06, X_dev)

print("Calculating checkpoint 09 development logits...")
dev_09 = get_logits(CKPT_09, X_dev)

# Coarse weight sweep: w06 * logits06 + (1-w06) * logits09.
weights_to_test = [0.0, 0.25, 0.50, 0.75, 1.0]
results = []

print("\n" + "=" * 58)
print("DEVELOPMENT WEIGHT SWEEP — 13-CLASS MACRO-F1")
print("=" * 58)

for w06 in weights_to_test:
    combined = w06 * dev_06 + (1.0 - w06) * dev_09
    pred = combined.argmax(axis=1)

    score13 = f1_score(
        y_dev,
        pred,
        labels=list(range(NUM_CLASSES)),
        average="macro",
        zero_division=0
    )

    score11 = f1_score(
        y_dev,
        pred,
        labels=sorted(np.unique(y_dev).tolist()),
        average="macro",
        zero_division=0
    )

    results.append((score13, w06, score11))

    print(
        f"06 weight={w06:.2f}, 09 weight={1-w06:.2f} | "
        f"13-class F1={score13:.6f} | "
        f"represented-class F1={score11:.6f}"
    )

# Select the weight with the highest 13-class dev Macro-F1.
# In a tie, the first weight in the grid is retained.
best_score13, best_w06, best_score11 = max(
    results,
    key=lambda x: x[0]
)

best_w09 = 1.0 - best_w06

print("\nSelected weights:")
print(f"Checkpoint 06: {best_w06:.2f}")
print(f"Checkpoint 09: {best_w09:.2f}")
print(f"Best dev 13-class Macro-F1: {best_score13:.6f}")

# ------------------------------------------------------------
# 4. GENERATE TEST LOGITS USING THE SELECTED WEIGHT
# ------------------------------------------------------------

print("\nCalculating checkpoint 06 test logits...")
test_06 = get_logits(CKPT_06, X_test)

print("Calculating checkpoint 09 test logits...")
test_09 = get_logits(CKPT_09, X_test)

test_logits = best_w06 * test_06 + best_w09 * test_09
test_ids = test_logits.argmax(axis=1)

# ------------------------------------------------------------
# 5. RECONSTRUCT DOCUMENTS IN ORIGINAL TEST ORDER
# ------------------------------------------------------------

with open(DATA / "Test/CL_test.json", "r", encoding="utf-8") as f:
    cl_test = json.load(f)

with open(DATA / "Test/IT_test.json", "r", encoding="utf-8") as f:
    it_test = json.load(f)

test_docs = OrderedDict()
for doc_id, sentences in cl_test.items():
    test_docs[str(doc_id)] = sentences
for doc_id, sentences in it_test.items():
    test_docs[str(doc_id)] = sentences

expected = sum(len(sents) for sents in test_docs.values())
assert len(test_docs) == 10
assert expected == 2197
assert len(test_ids) == expected

predictions = OrderedDict()
cursor = 0

for doc_id, sentences in test_docs.items():
    n = len(sentences)
    predictions[doc_id] = [
        id_to_label[int(i)] for i in test_ids[cursor:cursor + n]
    ]
    cursor += n

# Validate labels and sentence counts.
allowed = set(label_vocab.keys())

for doc_id, sentences in test_docs.items():
    assert len(predictions[doc_id]) == len(sentences)
    assert set(predictions[doc_id]).issubset(allowed)

# ------------------------------------------------------------
# 6. SAVE AS A NEW SUBMISSION
# ------------------------------------------------------------

out_dir = BASE / "Arch4_WeightSelected"
out_dir.mkdir(parents=True, exist_ok=True)

pred_path = out_dir / "prediction.json"
zip_path = out_dir / "submission.zip"

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, arcname="prediction.json")

with zipfile.ZipFile(zip_path, "r") as zf:
    assert zf.namelist() == ["prediction.json"]
    with zf.open("prediction.json") as f:
        assert json.load(f) == predictions

print("\n" + "=" * 58)
print("NEW WEIGHT-SELECTED SUBMISSION READY")
print("=" * 58)
print("Documents:", len(predictions))
print("Sentence predictions:", cursor)
print("Label distribution:", dict(Counter(
    label for labels in predictions.values() for label in labels
)))
print("Checkpoint 06 weight:", best_w06)
print("Checkpoint 09 weight:", best_w09)
print("Dev 13-class Macro-F1:", round(best_score13, 6))
print("ZIP:", zip_path)
print("Validation: PASS")